In [19]:
import pandas as pd
import numpy as np

bars_df = pd.read_parquet("../data/bars_seen_train.parquet")
unseen_bars_df = pd.read_parquet("../data/bars_unseen_train.parquet")
sentiment_df = pd.read_csv("../sentiment_overview.csv")
headlines_df_seen = pd.read_parquet("../data/headlines_seen_train.parquet")
headlines_df_unseen = pd.read_parquet("../data/headlines_unseen_train.parquet")

# headlines_df_public = pd.read_parquet("../data/headlines_seen_public_test.parquet")
# headlines_df_private = pd.read_parquet("../data/headlines_seen_private_test.parquet")

In [20]:
import re
session_of_interest = -1
# company = "Brevon Microchips"
keyword = ""

keywords = [
    "completes planned facility upgrade",
    "warns of supply chain disruptions affecting",
    "wins industry award",
    "announces significant capital expenditure plan for",
    "withdraws from",
    "recalls products in",
    "reports rising costs pressuring margins",
    "completes strategic acquisition",
    "increase in customer acquisition",
    "loses key contract",
    "reports record quarterly",
    "names new",
    "decline in operating income",
    "secures",
    "expands operations into",
    "launches next-generation",
    "begins scheduled maintenance",
    "margin improvement",
    "drop in new customer orders this quarter",
    "reports strong demand in",
    "files routine",
    "files for regulatory",
    "explores strategic alternatives",
    "confirms participation",
    "delays product launch",
    "opens new office",
    "enters joint venture",
    "faces class action",
    "sees mixed results in",
    "board meeting",
    "misses quarterly revenue estimates",
    "faces regulatory review",
    "to host investor day focused on",
    "announces breakthrough in",
    "revises long-term strategy with focus on",
    "share buyback program",
    "signs multi-year partnership with a",
    "to present at",
    "announces restructuring plan",
    "reports unexpected decline in",
    "announces major organizational restructuring",
    "publishes annual sustainability report",
    "lowers full-year guidance amid softening demand",
    "addresses investor concerns in open letter",
    "steps down unexpectedly",
    "raises full-year guidance citing robust demand",
    "in talks for potential merger, details undisclosed",
    "achieves key regulatory milestone ahead of schedule",
    "beats analyst expectations with strong earnings growth",
    "schedules annual shareholder meeting for next month"
]
# all_day_company_session_keywords = []

# for session_of_interest in range(1000):
#     headlines_seen = headlines_df_seen[headlines_df_seen["session"] == session_of_interest]
#     headlines_seen = headlines_seen[headlines_seen["headline"].str.contains("secures", na=False)]
#     headlines_unseen = headlines_df_unseen[headlines_df_unseen["session"] == session_of_interest]
#     headlines_unseen = headlines_unseen[headlines_unseen["headline"].str.contains("secures", na=False)]
#     df = pd.concat([headlines_seen, headlines_unseen])
#     if df.empty:
#         continue
#     for idx, row in df.iterrows():
#         print(row.headline)


In [21]:
# For each keyword, what is the average (close_end/close_mid - 1)?
from collections import defaultdict
import numpy as np

keyword_returns = defaultdict(list)

for session_id, headline_group in headlines_df_seen.groupby('session'):
    session_bars = bars_df[bars_df["session"] == session_id]  # your return label
    unseen_session_bars = unseen_bars_df[unseen_bars_df["session"] == session_id]  # your return label
    close_mid = session_bars.loc[session_bars['bar_ix'] == 49, 'close'].iloc[0]
    close_end = unseen_session_bars.loc[unseen_session_bars['bar_ix'] == 99, 'close'].iloc[0]
    print(close_mid, close_end)
    outcome = close_end/close_mid - 1
    for _, row in headline_group.iterrows():
        print(row)
        for kw in keywords:
            if kw in row['headline']:
                keyword_returns[kw].append(outcome)
                break

# Print empirical signal strength
for kw, returns in sorted(keyword_returns.items(), 
                           key=lambda x: np.mean(x[1]), reverse=True):
    print(f"{np.mean(returns):+.4f}  (n={len(returns):4d})  {kw}")


1.0316 1.0528
session                                                     0
headline    Relvos Biosciences opens new office in Southea...
bar_ix                                                      6
Name: 94361545557888, dtype: object
session                                                     0
headline    Orevex Renewables secures $500M contract with ...
bar_ix                                                     12
Name: 94361495510352, dtype: object
session                                                     0
headline    Relvos Biosciences names new head of precision...
bar_ix                                                     14
Name: 0, dtype: object
session                                                     0
headline    Calvis Sciences secures $650M contract with a ...
bar_ix                                                     20
Name: 0, dtype: object
session                                                     0
headline    Yorvov Pharmaceuticals secures $180M contract ...


In [22]:
bars_df = pd.read_parquet("../data/bars_seen_train.parquet")
unseen_bars_df = pd.read_parquet("../data/bars_unseen_train.parquet")

mid_price = bars_df.groupby('session')['close'].last()
end_price = unseen_bars_df.groupby('session')['close'].last()

returns = end_price / mid_price - 1

print(f"Mean return:     {returns.mean():.6f}")
print(f"Std return:      {returns.std():.6f}")
print(f"Sharpe (16x):    {returns.mean()/returns.std()*16:.4f}")
print(f"% positive:      {(returns > 0).mean():.3f}")
print(f"Skewness:        {returns.skew():.3f}")
print(f"Kurtosis:        {returns.kurt():.3f}")

# Baseline: always long 1 unit
baseline_sharpe = returns.mean() / returns.std() * 16
print(f"\nAlways-long Sharpe: {baseline_sharpe:.4f}")


Mean return:     0.003531
Std return:      0.020434
Sharpe (16x):    2.7647
% positive:      0.570
Skewness:        0.029
Kurtosis:        0.488

Always-long Sharpe: 2.7647


In [23]:
import pandas as pd
import numpy as np
from scipy import stats

bars_df = pd.read_parquet("../data/bars_seen_train.parquet")
unseen_bars_df = pd.read_parquet("../data/bars_unseen_train.parquet")
mid_price = bars_df.groupby('session')['close'].last()
end_price = unseen_bars_df.groupby('session')['close'].last()
returns = (end_price / mid_price - 1).rename('return')

# ── Build session-level features from seen bars ──────────────────────────
def session_features(grp):
    closes = grp.sort_values('bar_ix')['close'].values
    opens  = grp.sort_values('bar_ix')['open'].values
    highs  = grp.sort_values('bar_ix')['high'].values
    lows   = grp.sort_values('bar_ix')['low'].values
    n      = len(closes)

    bar_returns = np.diff(closes) / closes[:-1]

    return pd.Series({
        # Momentum
        'cum_return':      closes[-1] / closes[0] - 1,
        'last3_return':    closes[-1] / closes[max(0, n-4)] - 1,
        'first3_return':   closes[min(3, n-1)] / closes[0] - 1,
        'slope':           np.polyfit(np.arange(n), closes, 1)[0] / closes.mean(),

        # Volatility
        'realized_vol':    np.std(bar_returns),
        'range_mean':      np.mean((highs - lows) / closes),

        # Microstructure
        'up_bar_frac':     np.mean(closes > opens),
        'last_bar_up':     float(closes[-1] > opens[-1]),
        'wick_ratio':      np.mean((highs - closes) / (highs - lows + 1e-8)),

        # Regime
        'n_bars':          n,
        'accel':           bar_returns[-1] - bar_returns[0] if n > 1 else 0,
    })

features = bars_df.groupby('session').apply(session_features)
df = features.join(returns)

# ── Correlation with forward return ──────────────────────────────────────
print("=== Pearson correlation with forward return ===")
for col in features.columns:
    r, p = stats.pearsonr(df[col], df['return'])
    print(f"  {col:25s}  r={r:+.4f}  p={p:.4f}")

# ── Quintile analysis on strongest feature ────────────────────────────────
print("\n=== Quintile Sharpe by cum_return ===")
df['quintile'] = pd.qcut(df['cum_return'], 5, labels=False)
for q, grp in df.groupby('quintile'):
    sh = grp['return'].mean() / grp['return'].std() * 16
    print(f"  Q{q+1}  mean_ret={grp['return'].mean():+.5f}  sharpe={sh:.3f}  n={len(grp)}")

=== Pearson correlation with forward return ===
  cum_return                 r=-0.0694  p=0.0281
  last3_return               r=+0.0441  p=0.1631
  first3_return              r=+0.0229  p=0.4688
  slope                      r=-0.0670  p=0.0341
  realized_vol               r=+0.0716  p=0.0235
  range_mean                 r=+0.0787  p=0.0128
  up_bar_frac                r=-0.0594  p=0.0605
  last_bar_up                r=-0.0017  p=0.9575
  wick_ratio                 r=+0.0703  p=0.0263
  n_bars                     r=+nan  p=nan
  accel                      r=+0.0049  p=0.8776

=== Quintile Sharpe by cum_return ===
  Q1  mean_ret=+0.00541  sharpe=3.688  n=200
  Q2  mean_ret=+0.00275  sharpe=2.260  n=200
  Q3  mean_ret=+0.00437  sharpe=3.613  n=200
  Q4  mean_ret=+0.00376  sharpe=3.181  n=200
  Q5  mean_ret=+0.00135  sharpe=1.052  n=200


/tmp/ipykernel_3288/2440219881.py:48: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = stats.pearsonr(df[col], df['return'])


In [24]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# Features that are significant (p < 0.05)
sig_features = ['cum_return', 'slope', 'realized_vol', 'range_mean', 'wick_ratio']

X = df[sig_features].values
y = df['return'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Ridge regression — regularize heavily given weak signal
ridge = Ridge(alpha=100)

# Cross-validate Sharpe directly
def sharpe_scorer(estimator, X, y):
    pred = estimator.predict(X)
    # Position = prediction (go long/short proportional to signal)
    pnl = pred * y  # not right — see below
    return np.mean(pnl) / np.std(pnl) * 16

# Correct Sharpe CV: position proportional to prediction, PnL is position * actual_return
def sharpe_cv(estimator, X_tr, y_tr, X_val, y_val):
    estimator.fit(X_tr, y_tr)
    positions = estimator.predict(X_val)
    # Normalize positions to unit scale
    positions = positions / np.std(positions)
    pnl = positions * y_val
    return np.mean(pnl) / np.std(pnl) * 16

# Manual walk-free CV (sessions are independent so random split is fine)
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
sharpes = []
for train_idx, val_idx in kf.split(X_scaled):
    ridge.fit(X_scaled[train_idx], y[train_idx])
    positions = ridge.predict(X_scaled[val_idx])
    positions = positions / positions.std()
    pnl = positions * y[val_idx]
    sharpes.append(np.mean(pnl) / np.std(pnl) * 16)

print(f"CV Sharpe: {np.mean(sharpes):.4f} ± {np.std(sharpes):.4f}")
print(f"Baseline:  2.7647")

CV Sharpe: 2.6872 ± 0.5901
Baseline:  2.7647


In [25]:
# def make_positions(predictions, method='proportional', clip=3.0):
#     z = (predictions - predictions.mean()) / predictions.std()
#     z = np.clip(z, -clip, clip)
    
#     if method == 'proportional':
#         # Scale around the always-long baseline
#         # Never go short (drift is strongly positive)
#         return 1.0 + 0.3 * z          # tune the 0.3 multiplier
    
#     elif method == 'long_only_sized':
#         # Go long always, size by conviction
#         return np.maximum(0.1, 1.0 + 0.5 * z)
    
#     elif method == 'full_ls':
#         # Allow shorts on strong negative signals
#         return z                        # risky given strong positive drift
    
# make_positions(pred)


In [26]:
ALPHA=40000
SENSITIVITY=1.0
CLIP=0.5


import os
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

DATA_DIR = "../data"

# ── 1. Load training data ─────────────────────────────────────────────────────

seen_train    = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_train.parquet"), engine="fastparquet")
unseen_train  = pd.read_parquet(os.path.join(DATA_DIR, "bars_unseen_train.parquet"), engine="fastparquet")

# ── 2. Feature engineering ────────────────────────────────────────────────────

def build_features(bars: pd.DataFrame) -> pd.DataFrame:
    """Extract session-level features from seen OHLC bars."""

    def session_feats(grp):
        grp   = grp.sort_values("bar_ix")
        c     = grp["close"].values
        o     = grp["open"].values
        h     = grp["high"].values
        lo    = grp["low"].values
        n     = len(c)
        br    = np.diff(c) / c[:-1] if n > 1 else np.array([0.0])

        return pd.Series({
            "cum_return":   c[-1] / c[0] - 1,
            "last3_return": c[-1] / c[max(0, n - 4)] - 1,
            "slope":        np.polyfit(np.arange(n), c, 1)[0] / c.mean(),
            "realized_vol": np.std(br),
            "range_mean":   np.mean((h - lo) / c),
            "up_bar_frac":  np.mean(c > o),
            "wick_ratio":   np.mean((h - c) / (h - lo + 1e-8)),
        })

    return bars.groupby("session").apply(session_feats)

FEATURES = ["cum_return", "last3_return", "slope",
            "realized_vol", "range_mean", "up_bar_frac", "wick_ratio"]

# ── 3. Build training labels ──────────────────────────────────────────────────

mid_price = seen_train.groupby("session")["close"].last()
end_price = unseen_train.groupby("session")["close"].last()
y_train   = (end_price / mid_price - 1).rename("return")

X_train_raw = build_features(seen_train)[FEATURES]

# Align index (drop any sessions missing from either side)
idx = X_train_raw.index.intersection(y_train.index)
X_train_raw = X_train_raw.loc[idx]
y_train     = y_train.loc[idx]

# ── 4. Cross-validate to confirm we beat the baseline ────────────────────────

from sklearn.model_selection import KFold

def compute_sharpe(positions, returns):
    pnl = positions * returns
    return np.mean(pnl) / np.std(pnl) * 16

def make_positions(raw_preds, long_bias=1.0, sensitivity=0.3, clip=2.5):
    """
    Never go short (strong positive drift).
    Modulate position size around the long bias using model signal.
    position = long_bias + sensitivity * z_score(prediction)
    Clipped so minimum position is always > 0.
    """
    z = (raw_preds - raw_preds.mean()) / (raw_preds.std() + 1e-8)
    z = np.clip(z, -clip, clip)
    positions = long_bias + sensitivity * z
    positions = np.maximum(positions, 0.05)   # never fully flat
    return positions

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train_raw)
y        = y_train.values

kf      = KFold(n_splits=5, shuffle=True, random_state=42)
cv_sharpes_model    = []
cv_sharpes_baseline = []

for train_idx, val_idx in kf.split(X_scaled):
    model = Ridge(alpha=ALPHA)
    model.fit(X_scaled[train_idx], y[train_idx])

    preds     = model.predict(X_scaled[val_idx])
    positions = make_positions(preds, sensitivity=SENSITIVITY, clip=CLIP)
    y_val     = y[val_idx]

    cv_sharpes_model.append(compute_sharpe(positions, y_val))
    cv_sharpes_baseline.append(compute_sharpe(np.ones(len(y_val)), y_val))

print(f"Baseline CV Sharpe : {np.mean(cv_sharpes_baseline):.4f} ± {np.std(cv_sharpes_baseline):.4f}")
print(f"Model    CV Sharpe : {np.mean(cv_sharpes_model):.4f} ± {np.std(cv_sharpes_model):.4f}")

# ── 5. Fit final model on all training data ───────────────────────────────────

final_model = Ridge(alpha=ALPHA)
final_scaler = StandardScaler()
X_final = final_scaler.fit_transform(X_train_raw)
final_model.fit(X_final, y)

print("\nFeature coefficients:")
for feat, coef in zip(FEATURES, final_model.coef_):
    print(f"  {feat:20s}  {coef:+.6f}")

# ── 6. Generate predictions for both test sets ────────────────────────────────

def predict_positions(bars: pd.DataFrame) -> pd.Series:
    feats    = build_features(bars)[FEATURES]
    X        = final_scaler.transform(feats)
    preds    = final_model.predict(X)
    positions = make_positions(preds)
    return pd.Series(positions, index=feats.index, name="target_position")

public_bars  = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_public_test.parquet"),  engine="fastparquet")
private_bars = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_private_test.parquet"), engine="fastparquet")

public_positions  = predict_positions(public_bars)
private_positions = predict_positions(private_bars)

# ── 7. Write submissions ──────────────────────────────────────────────────────

def write_submission(positions: pd.Series, path: str):
    sub = positions.reset_index()
    sub.columns = ["session", "target_position"]
    sub = sub.sort_values("session")
    sub.to_csv(path, index=False)
    print(f"Saved {len(sub)} rows → {path}")
    print(sub.describe())

write_submission(public_positions,  "submission_public.csv")
write_submission(private_positions, "submission_private.csv")

Baseline CV Sharpe : 2.7838 ± 0.5857
Model    CV Sharpe : 2.9824 ± 0.5270

Feature coefficients:
  cum_return            -0.000033
  last3_return          +0.000022
  slope                 -0.000032
  realized_vol          +0.000035
  range_mean            +0.000038
  up_bar_frac           -0.000028
  wick_ratio            +0.000033
Saved 10000 rows → submission_public.csv
           session  target_position
count  10000.00000     10000.000000
mean    5999.50000         0.999067
std     2886.89568         0.296346
min     1000.00000         0.250000
25%     3499.75000         0.790253
50%     5999.50000         0.983981
75%     8499.25000         1.200229
max    10999.00000         1.750000
Saved 10000 rows → submission_private.csv
           session  target_position
count  10000.00000     10000.000000
mean   15999.50000         0.999158
std     2886.89568         0.296497
min    11000.00000         0.250000
25%    13499.75000         0.794812
50%    15999.50000         0.985653
75%   

In [27]:
for s in [0.0, 0.1, 0.2, 0.3, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]:
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_scaled):
        m = Ridge(alpha=100).fit(X_scaled[train_idx], y[train_idx])
        p = make_positions(m.predict(X_scaled[val_idx]), sensitivity=s)
        fold_sharpes.append(compute_sharpe(p, y[val_idx]))
    print(f"sensitivity={s:.1f}  sharpe={np.mean(fold_sharpes):.4f}")


sensitivity=0.0  sharpe=2.7838
sensitivity=0.1  sharpe=2.8236
sensitivity=0.2  sharpe=2.8356
sensitivity=0.3  sharpe=2.8247
sensitivity=0.5  sharpe=2.7568
sensitivity=0.6  sharpe=2.7184
sensitivity=0.7  sharpe=2.6759
sensitivity=0.8  sharpe=2.6309
sensitivity=0.9  sharpe=2.5831
sensitivity=1.0  sharpe=2.5326
sensitivity=1.1  sharpe=2.4841
sensitivity=1.2  sharpe=2.4430
sensitivity=1.3  sharpe=2.4059
sensitivity=1.4  sharpe=2.3721
sensitivity=1.5  sharpe=2.3408


In [28]:
from itertools import product

best, best_params = -np.inf, {}

for alpha, sensitivity, clip in product(
    [10, 50, 100, 200, 500, 400, 300, 600, 700, 800, 900, 1000, 1250, 1500, 1750, 2000, 2250, 2500, 2750, 3000, 5000, 7500, 10000, 20000, 30000, 40000],   # ridge regularization
    [0.15, 0.18, 0.20, 0.22, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5],  # tight grid around optimum
    [0.25, 0.5, 0.75, 1.0, 1.2, 1.5, 2.0, 2.5, 3.0],           # outlier clipping
):
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_scaled):
        m = Ridge(alpha=alpha).fit(X_scaled[train_idx], y[train_idx])
        p = make_positions(
            m.predict(X_scaled[val_idx]),
            sensitivity=sensitivity,
            clip=clip
        )
        fold_sharpes.append(compute_sharpe(p, y[val_idx]))
    
    s = np.mean(fold_sharpes)
    if s > best:
        best, best_params = s, dict(alpha=alpha, sensitivity=sensitivity, clip=clip)

print(f"Best CV Sharpe : {best:.4f}")
print(f"Best params    : {best_params}")

Best CV Sharpe : 2.9824
Best params    : {'alpha': 40000, 'sensitivity': 1.0, 'clip': 0.5}


In [29]:
def make_positions_vol_scaled(raw_preds, vol, sensitivity=0.2, vol_power=0.5):
    """
    Size positions by both signal AND volatility.
    vol_power=0.5 means sqrt scaling — aggressive but not linear.
    vol_power=0.0 recovers the original approach.
    """
    z = (raw_preds - raw_preds.mean()) / (raw_preds.std() + 1e-8)
    z = np.clip(z, -2.5, 2.5)
    
    # Normalize vol to have mean 1
    vol_norm = vol / vol.mean()
    vol_scale = vol_norm ** vol_power
    
    positions = (1.0 + sensitivity * z) * vol_scale
    positions = np.maximum(positions, 0.05)
    return positions

# Test across vol_power values
vol_train = X_train_raw['realized_vol'].values

for vp in [2.0, 0.0, 0.25, 0.5, 0.75, 1.0]:
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_scaled):
        m = Ridge(alpha=100).fit(X_scaled[train_idx], y[train_idx])
        p = make_positions_vol_scaled(
            m.predict(X_scaled[val_idx]),
            vol_train[val_idx],
            sensitivity=0.2,
            vol_power=vp
        )
        fold_sharpes.append(compute_sharpe(p, y[val_idx]))
    print(f"vol_power={vp:.2f}  sharpe={np.mean(fold_sharpes):.4f}")

vol_power=2.00  sharpe=2.6349
vol_power=0.00  sharpe=2.8356
vol_power=0.25  sharpe=2.8282
vol_power=0.50  sharpe=2.8141
vol_power=0.75  sharpe=2.7942
vol_power=1.00  sharpe=2.7693


In [30]:
def make_positions_kelly(raw_preds, 
                          session_vols,
                          kelly_fraction=0.3):
    """
    Kelly-inspired: size ∝ predicted_return / realized_variance
    kelly_fraction < 1 = fractional Kelly (safer)
    """
    # Normalize predicted returns to be on same scale as actual
    pred_scaled = raw_preds / raw_preds.std() * y.std()
    
    # Kelly position: mu / sigma^2
    variance    = session_vols ** 2 + 1e-8
    kelly_pos   = pred_scaled / variance
    
    # Normalize and apply fraction
    kelly_pos   = kelly_pos / kelly_pos.std()
    positions   = 1.0 + kelly_fraction * kelly_pos
    positions   = np.maximum(positions, 0.05)
    return positions

vol_train = X_train_raw['realized_vol'].values

for kf_frac in [0.1, 0.2, 0.3, 0.4, 0.5]:
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_scaled):
        m = Ridge(alpha=100).fit(X_scaled[train_idx], y[train_idx])
        p = make_positions_kelly(
            m.predict(X_scaled[val_idx]),
            vol_train[val_idx],
            kelly_fraction=kf_frac
        )
        fold_sharpes.append(compute_sharpe(p, y[val_idx]))
    print(f"kelly_fraction={kf_frac:.1f}  sharpe={np.mean(fold_sharpes):.4f}")

kelly_fraction=0.1  sharpe=2.8218
kelly_fraction=0.2  sharpe=2.8362
kelly_fraction=0.3  sharpe=2.8392
kelly_fraction=0.4  sharpe=2.8351
kelly_fraction=0.5  sharpe=2.8279


In [31]:
def make_positions_threshold(raw_preds, 
                              long_threshold=0.3,
                              short_threshold=-0.3,
                              long_size=1.5,
                              base_size=1.0,
                              short_size=0.1):
    """
    Three-regime positioning:
    - Strong positive signal → size up
    - Weak signal either way → base position  
    - Strong negative signal → near-flat (don't short, drift too strong)
    """
    z = (raw_preds - raw_preds.mean()) / (raw_preds.std() + 1e-8)
    
    positions = np.where(
        z >  long_threshold,  long_size,
        np.where(
        z < short_threshold,  short_size,
                              base_size
        )
    )
    return positions

# Grid search thresholds
from itertools import product

best_sharpe, best_params = -np.inf, {}
for lt, st, ls, ss, alpha in product(
    [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.5],   # long threshold
    [-0.05, -0.1, -0.15, -0.2, -0.25, -0.3 -0.5], # short threshold  
    [0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.3, 1.5, 2.0, 3.0, 4.0],   # long size
    [0.05, 0.1, 0.2, 0.3, 0.4, 0.5],  # short (near-flat) size
    [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 12000, 14000, 16000, 18000, 20000]
):
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_scaled):
        m = Ridge(alpha=alpha).fit(X_scaled[train_idx], y[train_idx])
        p = make_positions_threshold(
            m.predict(X_scaled[val_idx]),
            long_threshold=lt,
            short_threshold=st,
            long_size=ls,
            short_size=ss
        )
        fold_sharpes.append(compute_sharpe(p, y[val_idx]))
    
    s = np.mean(fold_sharpes)
    if s > best_sharpe:
        best_sharpe = s
        best_params = dict(lt=lt, st=st, ls=ls, ss=ss, alpha=alpha)

print(f"Best threshold Sharpe: {best_sharpe:.4f}")
print(f"Best params: {best_params}")
print(f"Current best: 2.8356")

KeyboardInterrupt: 

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

DATA_DIR = "../data"

# ── Best parameters from optimisation ────────────────────────────────────────
BEST_PARAMS = {
    'lt':    0.5,    # long threshold
    'st':   -0.15,   # short threshold
    'ls':    0.6,    # long size
    'ss':    0.2,    # short (near-flat) size
    'alpha': 1000,   # Ridge regularisation
}

FEATURES = [
    "cum_return",
    "last3_return",
    "slope",
    "realized_vol",
    "range_mean",
    "up_bar_frac",
    "wick_ratio",
]

# ── Feature engineering ───────────────────────────────────────────────────────

def build_features(bars: pd.DataFrame) -> pd.DataFrame:
    def session_feats(grp):
        grp = grp.sort_values("bar_ix")
        c   = grp["close"].values
        o   = grp["open"].values
        h   = grp["high"].values
        lo  = grp["low"].values
        n   = len(c)
        br  = np.diff(c) / c[:-1] if n > 1 else np.array([0.0])

        return pd.Series({
            "cum_return":   c[-1] / c[0] - 1,
            "last3_return": c[-1] / c[max(0, n-4)] - 1,
            "slope":        np.polyfit(np.arange(n), c, 1)[0] / c.mean(),
            "realized_vol": np.std(br),
            "range_mean":   np.mean((h - lo) / c),
            "up_bar_frac":  np.mean(c > o),
            "wick_ratio":   np.mean((h - c) / (h - lo + 1e-8)),
        })

    return bars.groupby("session").apply(session_feats)

# ── Position sizing ───────────────────────────────────────────────────────────

def make_positions_threshold(raw_preds, lt, st, ls, ss):
    """
    Three-regime positioning:
      z >  lt  →  size up (long_size)
      z <  st  →  near-flat (short_size) — never truly short given positive drift
      else     →  base position 1.0
    """
    z = (raw_preds - raw_preds.mean()) / (raw_preds.std() + 1e-8)

    positions = np.where(
        z >  lt, ls,
        np.where(
        z <  st, ss,
                 1.0   # base position
        )
    )
    return positions

def compute_sharpe(positions, returns):
    pnl = positions * returns
    return np.mean(pnl) / np.std(pnl) * 16

# ── Load & fit on full training data ─────────────────────────────────────────

seen_train   = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_train.parquet"),   engine="fastparquet")
unseen_train = pd.read_parquet(os.path.join(DATA_DIR, "bars_unseen_train.parquet"), engine="fastparquet")

mid_price = seen_train.groupby("session")["close"].last()
end_price = unseen_train.groupby("session")["close"].last()
y_train   = (end_price / mid_price - 1).rename("return")

X_train_raw = build_features(seen_train)[FEATURES]
idx         = X_train_raw.index.intersection(y_train.index)
X_train_raw = X_train_raw.loc[idx]
y_train     = y_train.loc[idx]

scaler      = StandardScaler()
X_scaled    = scaler.fit_transform(X_train_raw)
y           = y_train.values

model = Ridge(alpha=BEST_PARAMS['alpha'])
model.fit(X_scaled, y)

print("=== Final model fitted on full training data ===")
print(f"Alpha: {BEST_PARAMS['alpha']}")
print("\nFeature coefficients:")
for feat, coef in zip(FEATURES, model.coef_):
    print(f"  {feat:20s}  {coef:+.6f}")

# Quick sanity-check: in-sample Sharpe with threshold sizing
train_preds     = model.predict(X_scaled)
train_positions = make_positions_threshold(
    train_preds,
    lt=BEST_PARAMS['lt'],
    st=BEST_PARAMS['st'],
    ls=BEST_PARAMS['ls'],
    ss=BEST_PARAMS['ss'],
)
print(f"\nIn-sample Sharpe (sanity check): {compute_sharpe(train_positions, y):.4f}")
print(f"Position distribution:")
print(f"  Long  ({BEST_PARAMS['ls']:.1f}): {np.mean(train_positions == BEST_PARAMS['ls']):.3f} of sessions")
print(f"  Base  (1.0): {np.mean(train_positions == 1.0):.3f} of sessions")
print(f"  Flat  ({BEST_PARAMS['ss']:.1f}): {np.mean(train_positions == BEST_PARAMS['ss']):.3f} of sessions")

# ── Submission function ───────────────────────────────────────────────────────

def make_submission(bars_path: str, output_path: str) -> pd.DataFrame:
    """
    Load seen bars, build features, predict positions, write CSV.
    Returns the submission dataframe.
    """
    bars      = pd.read_parquet(bars_path, engine="fastparquet")
    feats     = build_features(bars)[FEATURES]
    X         = scaler.transform(feats)
    preds     = model.predict(X)

    positions = make_positions_threshold(
        preds,
        lt=BEST_PARAMS['lt'],
        st=BEST_PARAMS['st'],
        ls=BEST_PARAMS['ls'],
        ss=BEST_PARAMS['ss'],
    )

    sub = pd.DataFrame({
        "session":         feats.index,
        "target_position": positions,
    }).sort_values("session")

    sub.to_csv(output_path, index=False)

    print(f"\n── {output_path} ──")
    print(f"  Sessions:     {len(sub)}")
    print(f"  Long  ({BEST_PARAMS['ls']:.1f}): {np.mean(positions == BEST_PARAMS['ls']):.3f}")
    print(f"  Base  (1.0): {np.mean(positions == 1.0):.3f}")
    print(f"  Flat  ({BEST_PARAMS['ss']:.1f}): {np.mean(positions == BEST_PARAMS['ss']):.3f}")
    print(sub.head())

    return sub

# ── Generate submissions ──────────────────────────────────────────────────────

public_sub  = make_submission(
    bars_path   = os.path.join(DATA_DIR, "bars_seen_public_test.parquet"),
    output_path = "submission_public_threshold.csv",
)

private_sub = make_submission(
    bars_path   = os.path.join(DATA_DIR, "bars_seen_private_test.parquet"),
    output_path = "submission_private_threshold.csv",
)

print("\n=== Done ===")
print("submission_public_threshold.csv  → upload for public leaderboard")
print("submission_private_threshold.csv → upload for private leaderboard")

=== Final model fitted on full training data ===
Alpha: 1000

Feature coefficients:
  cum_return            -0.000403
  last3_return          +0.000536
  slope                 -0.000272
  realized_vol          +0.000446
  range_mean            +0.000579
  up_bar_frac           -0.000245
  wick_ratio            +0.000395

In-sample Sharpe (sanity check): 3.2310
Position distribution:
  Long  (0.6): 0.290 of sessions
  Base  (1.0): 0.249 of sessions
  Flat  (0.2): 0.461 of sessions

── submission_public_threshold.csv ──
  Sessions:     10000
  Long  (0.6): 0.300
  Base  (1.0): 0.238
  Flat  (0.2): 0.462
   session  target_position
0     1000              0.6
1     1001              0.2
2     1002              0.2
3     1003              0.2
4     1004              0.2

── submission_private_threshold.csv ──
  Sessions:     10000
  Long  (0.6): 0.295
  Base  (1.0): 0.242
  Flat  (0.2): 0.463
   session  target_position
0    11000              0.2
1    11001              1.0
2    11002    

In [ ]:
# Variant: pure always-long, no model adjustment at all
# This is sensitivity=0.0, which got CV 2.7838
# Your public score should be close to 2.76 baseline

# vs your original sensitivity=0.2 which got public 2.40
# Something is off — CV said 2.84 but public gave 2.40

# The gap (2.84 CV vs 2.40 public) suggests even your 
# original model is slightly overfit. Check this:

for sensitivity in [0.0, 0.1, 0.2]:
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_scaled):
        m = Ridge(alpha=100).fit(X_scaled[train_idx], y[train_idx])
        p = make_positions(
            m.predict(X_scaled[val_idx]),
            sensitivity=sensitivity
        )
        fold_sharpes.append(compute_sharpe(p, y[val_idx]))
    print(f"sensitivity={sensitivity:.1f}  "
          f"cv_sharpe={np.mean(fold_sharpes):.4f}  "
          f"cv_std={np.std(fold_sharpes):.4f}")

sensitivity=0.0  cv_sharpe=2.7838  cv_std=0.5857
sensitivity=0.1  cv_sharpe=2.8236  cv_std=0.5560
sensitivity=0.2  cv_sharpe=2.8356  cv_std=0.5296


In [ ]:
import re

seen_train     = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_train.parquet"),
                                  engine="fastparquet")
headlines_train = pd.read_parquet(os.path.join(DATA_DIR, "headlines_seen_train.parquet"),
                                   engine="fastparquet")

def extract_secures_features(headlines_df: pd.DataFrame) -> pd.DataFrame:
    records = []
    for session_id, grp in headlines_df.groupby('session'):
        amounts = []
        for _, row in grp.iterrows():
            h = row['headline'].lower()
            if 'secures' not in h:
                continue
            m = re.search(
                r'\$?([\d,]+(?:\.\d+)?)\s*(million|billion|m\b|b\b)',
                row['headline'], re.IGNORECASE
            )
            if m:
                amt = float(m.group(1).replace(',', ''))
                if m.group(2).lower() in ('billion', 'b'):
                    amt *= 1000
                amounts.append(amt)

        records.append({
            'session':        session_id,
            'has_secures':    float(len(amounts) > 0),
            'secures_amount': np.log1p(max(amounts)) if amounts else 0.0,
            'secures_count':  len(amounts),
        })

    return pd.DataFrame(records).set_index('session')

secures = extract_secures_features(headlines_train)

print("=== Secures feature correlations ===")
for col in ['has_secures', 'secures_amount', 'secures_count']:
    vals = secures.reindex(y_train.index)[col].fillna(0)
    r, p = stats.pearsonr(vals, y_train)
    print(f"  {col:20s}  r={r:+.4f}  p={p:.4f}")

# Also check: among sessions WITH a secures headline,
# does the amount predict the return?
has_secures_idx = secures[secures['has_secures'] == 1].index
has_secures_idx = has_secures_idx.intersection(y_train.index)
r, p = stats.pearsonr(
    secures.loc[has_secures_idx, 'secures_amount'],
    y_train.loc[has_secures_idx]
)
print(f"\n  secures_amount (conditional on has_secures, n={len(has_secures_idx)}):")
print(f"  r={r:+.4f}  p={p:.4f}")

=== Secures feature correlations ===
  has_secures           r=-0.0327  p=0.3012
  secures_amount        r=-0.0175  p=0.5811
  secures_count         r=-0.0206  p=0.5146

  secures_amount (conditional on has_secures, n=789):
  r=+0.0559  p=0.1166


In [ ]:
import re
import numpy as np
import pandas as pd
from scipy import stats

def extract_secures_timed(bars_df: pd.DataFrame, 
                           headlines_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each session, extract secures headlines with their bar_ix,
    then weight the deal amount by time remaining until session midpoint.
    
    time_weight = (max_bar_ix - bar_ix) / max_bar_ix
    → headline at bar 0:  full weight (1.0)
    → headline at bar 50: zero weight (already at midpoint)
    """
    # Get max bar_ix per session (the midpoint bar)
    max_bar = bars_df.groupby('session')['bar_ix'].max().rename('max_bar')
    
    records = []
    for session_id, grp in headlines_df.groupby('session'):
        session_max_bar = max_bar.get(session_id, None)
        if session_max_bar is None or session_max_bar == 0:
            records.append({
                'session':                  session_id,
                'secures_time_weighted':    0.0,
                'secures_amount_raw':       0.0,
                'secures_early':            0.0,  # amount in first third
                'secures_late':             0.0,  # amount in last third
                'secures_time_of_max':      0.5,  # normalised bar position
            })
            continue

        session_secures = []
        for _, row in grp.iterrows():
            h = row['headline']
            if 'secures' not in h.lower():
                continue

            # Extract dollar amount
            m = re.search(
                r'\$?([\d,]+(?:\.\d+)?)\s*(million|billion|m\b|b\b)',
                h, re.IGNORECASE
            )
            if not m:
                continue

            amt = float(m.group(1).replace(',', ''))
            if m.group(2).lower() in ('billion', 'b'):
                amt *= 1000  # normalise to millions

            bar_ix       = row['bar_ix']
            time_remaining = (session_max_bar - bar_ix) / session_max_bar
            time_elapsed   = bar_ix / session_max_bar

            session_secures.append({
                'amount':           amt,
                'log_amount':       np.log1p(amt),
                'bar_ix':           bar_ix,
                'time_remaining':   time_remaining,  # 1.0=start, 0.0=midpoint
                'time_elapsed':     time_elapsed,
                'time_weighted':    np.log1p(amt) * time_remaining,
            })

        if not session_secures:
            records.append({
                'session':               session_id,
                'secures_time_weighted': 0.0,
                'secures_amount_raw':    0.0,
                'secures_early':         0.0,
                'secures_late':          0.0,
                'secures_time_of_max':   0.5,
            })
            continue

        df_s = pd.DataFrame(session_secures)
        third = session_max_bar / 3

        records.append({
            'session': session_id,

            # Core signal: sum of log(amount) × time_remaining
            # Large early deal >> small late deal
            'secures_time_weighted': df_s['time_weighted'].sum(),

            # Raw amount (log) ignoring timing
            'secures_amount_raw':    df_s['log_amount'].sum(),

            # Early vs late split (is timing actually informative?)
            'secures_early':         df_s[df_s['bar_ix'] <= third]['log_amount'].sum(),
            'secures_late':          df_s[df_s['bar_ix'] >  third]['log_amount'].sum(),

            # When did the biggest deal hit? (0=start, 1=midpoint)
            'secures_time_of_max':   df_s.loc[df_s['log_amount'].idxmax(), 'time_elapsed'],
        })

    return pd.DataFrame(records).set_index('session')


# ── Run it ────────────────────────────────────────────────────────────────────

seen_train     = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_train.parquet"),
                                  engine="fastparquet")
headlines_train = pd.read_parquet(os.path.join(DATA_DIR, "headlines_seen_train.parquet"),
                                   engine="fastparquet")

secures_timed = extract_secures_timed(seen_train, headlines_train)

print("=== Timed secures correlations ===")
for col in secures_timed.columns:
    vals = secures_timed.reindex(y_train.index)[col].fillna(0)
    r, p = stats.pearsonr(vals, y_train)
    stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    print(f"  {col:28s}  r={r:+.4f}  p={p:.4f}  {stars}")

# ── Inspect the distribution ──────────────────────────────────────────────────
print("\n=== Secures time-weighted signal analysis ===")
df_q = secures_timed.reindex(y_train.index).fillna(0).copy()
df_q['return'] = y_train

# Split into zero (no secures) vs non-zero (has secures)
no_secures  = df_q[df_q['secures_time_weighted'] == 0]
has_secures = df_q[df_q['secures_time_weighted'] >  0]

print(f"Sessions without secures: {len(no_secures)} "
      f"({len(no_secures)/len(df_q):.1%})")
print(f"Sessions with secures:    {len(has_secures)} "
      f"({len(has_secures)/len(df_q):.1%})")

# Sharpe for each population
def sharpe(returns):
    return returns.mean() / returns.std() * 16

print(f"\nSharpe — no secures:       {sharpe(no_secures['return']):.4f}")
print(f"Sharpe — has secures:      {sharpe(has_secures['return']):.4f}")

# Among sessions WITH secures, quintile by signal strength
print("\n=== Quintile Sharpe (sessions WITH secures only) ===")
has_secures = has_secures.copy()
has_secures['quintile'] = pd.qcut(
    has_secures['secures_time_weighted'], 5, labels=False
)
for q, grp in has_secures.groupby('quintile'):
    print(f"  Q{q+1}  mean_ret={grp['return'].mean():+.5f}  "
          f"sharpe={sharpe(grp['return']):.3f}  "
          f"n={len(grp)}  "
          f"mean_signal={grp['secures_time_weighted'].mean():.3f}")

# t-test: are secures sessions meaningfully different from non-secures?
from scipy import stats
t, p = stats.ttest_ind(has_secures['return'], no_secures['return'])
print(f"\nt-test (has vs no secures):  t={t:+.4f}  p={p:.4f}")

# Correlation within secures sessions only
r, p = stats.pearsonr(
    has_secures['secures_time_weighted'],
    has_secures['return']
)
print(f"Correlation within secures:  r={r:+.4f}  p={p:.4f}")

=== Timed secures correlations ===
  secures_time_weighted         r=-0.0022  p=0.9453  
  secures_amount_raw            r=-0.0177  p=0.5771  
  secures_early                 r=+0.0223  p=0.4819  
  secures_late                  r=-0.0371  p=0.2406  
  secures_time_of_max           r=-0.0356  p=0.2611  

=== Secures time-weighted signal analysis ===
Sessions without secures: 220 (22.0%)
Sessions with secures:    780 (78.0%)

Sharpe — no secures:       3.7230
Sharpe — has secures:      2.5033

=== Quintile Sharpe (sessions WITH secures only) ===
  Q1  mean_ret=+0.00183  sharpe=1.382  n=156  mean_signal=1.324
  Q2  mean_ret=+0.00315  sharpe=2.365  n=156  mean_signal=3.399
  Q3  mean_ret=+0.00300  sharpe=2.495  n=156  mean_signal=5.075
  Q4  mean_ret=+0.00430  sharpe=3.298  n=156  mean_signal=7.115
  Q5  mean_ret=+0.00381  sharpe=2.994  n=156  mean_signal=11.480

t-test (has vs no secures):  t=-0.9052  p=0.3656
Correlation within secures:  r=+0.0202  p=0.5740


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# ── Confidence-weighted position sizing ───────────────────────────────────────

def make_positions_confident(raw_preds, 
                              sensitivity=0.2,
                              confidence_power=1.0):
    """
    Scale position by both direction AND confidence.
    
    Standard approach:  position = 1 + s * z
    This approach:      position = 1 + s * z * |z|^power
    
    power=0.0  → standard linear (your original model)
    power=0.5  → mild concentration on high-confidence sessions
    power=1.0  → quadratic — doubles down strongly on extremes
    power=2.0  → cubic — very aggressive concentration
    
    Effect: weak signals get shrunk toward 1.0 (baseline),
    strong signals get amplified. Middle-of-the-road sessions
    contribute less noise to the portfolio.
    """
    z = (raw_preds - raw_preds.mean()) / (raw_preds.std() + 1e-8)
    z = np.clip(z, -2.5, 2.5)
    
    # Amplify by confidence (magnitude of z)
    confidence_weight = np.abs(z) ** confidence_power
    # Normalise so mean weight stays ~1 (don't change overall exposure)
    confidence_weight = confidence_weight / confidence_weight.mean()
    
    positions = 1.0 + sensitivity * z * confidence_weight
    positions = np.maximum(positions, 0.05)
    return positions

# ── Grid search over sensitivity × confidence_power ──────────────────────────

print("=== Confidence-weighted position sizing ===\n")
print(f"{'sensitivity':>12} {'power':>8} {'cv_sharpe':>12} {'cv_std':>10}")
print("-" * 46)

best_sharpe, best_params = -np.inf, {}

for sensitivity in [0.1, 0.15, 0.2, 0.25, 0.3, 0.4]:
    for power in [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0]:
        for alpha in range(100, 1100, 100):
            fold_sharpes = []
            for train_idx, val_idx in kf.split(X_scaled):
                m = Ridge(alpha=alpha).fit(X_scaled[train_idx], y[train_idx])
                p = make_positions_confident(
                    m.predict(X_scaled[val_idx]),
                    sensitivity=sensitivity,
                    confidence_power=power,
                )
                fold_sharpes.append(compute_sharpe(p, y[val_idx]))

            mean_sh = np.mean(fold_sharpes)
            std_sh  = np.std(fold_sharpes)
            print(f"{sensitivity:>12.2f} {power:>8.2f} {mean_sh:>12.4f} {std_sh:>10.4f} alpha: {alpha}")

            if mean_sh > best_sharpe:
                best_sharpe = mean_sh
                best_params = dict(sensitivity=sensitivity, 
                                power=power, 
                                std=std_sh,
                                alpha=alpha)

print(f"\nBest CV Sharpe: {best_sharpe:.4f} ± {best_params['std']:.4f}")
print(f"Best params:    sensitivity={best_params['sensitivity']}  "
      f"power={best_params['power']}")
print(f"Baseline:       2.8356  (sensitivity=0.2, power=0.0)")
print(f"Improvement:   +{best_sharpe - 2.8356:.4f}")

=== Confidence-weighted position sizing ===

 sensitivity    power    cv_sharpe     cv_std
----------------------------------------------
        0.10     0.00       2.8236     0.5560 alpha: 100
        0.10     0.00       2.8289     0.5585 alpha: 200
        0.10     0.00       2.8313     0.5597 alpha: 300
        0.10     0.00       2.8326     0.5604 alpha: 400
        0.10     0.00       2.8334     0.5610 alpha: 500
        0.10     0.00       2.8340     0.5614 alpha: 600
        0.10     0.00       2.8344     0.5618 alpha: 700
        0.10     0.00       2.8347     0.5622 alpha: 800
        0.10     0.00       2.8350     0.5625 alpha: 900
        0.10     0.00       2.8352     0.5627 alpha: 1000
        0.10     0.25       2.8194     0.5500 alpha: 100
        0.10     0.25       2.8250     0.5533 alpha: 200
        0.10     0.25       2.8274     0.5549 alpha: 300
        0.10     0.25       2.8287     0.5560 alpha: 400
        0.10     0.25       2.8295     0.5568 alpha: 500
      

In [ ]:
def build_keyword_transitions(headlines_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each session, analyse the sequence of keyword sentiments
    across bar_ix to detect transitions and patterns.
    """
    # Simple directional mapping — avoid label leakage
    keyword_direction = {
        # Positive
        "beats analyst expectations with strong earnings growth": +1,
        "reports record quarterly":                              +1,
        "raises full-year guidance citing robust demand":        +1,
        "achieves key regulatory milestone ahead of schedule":   +1,
        "announces breakthrough in":                             +1,
        "margin improvement":                                    +1,
        "increase in customer acquisition":                      +1,
        "reports strong demand in":                              +1,
        "completes strategic acquisition":                       +1,
        "signs multi-year partnership with a":                   +1,
        "secures":                                               +1,
        "enters joint venture":                                  +1,
        "expands operations into":                               +1,
        "launches next-generation":                              +1,
        "share buyback program":                                 +1,
        "in talks for potential merger, details undisclosed":    +1,
        "revises long-term strategy with focus on":              +1,
        "wins industry award":                                   +1,
        "opens new office":                                      +1,
        "completes planned facility upgrade":                    +1,
        "confirms participation":                                +1,

        # Negative
        "lowers full-year guidance amid softening demand":       -1,
        "misses quarterly revenue estimates":                    -1,
        "reports unexpected decline in":                         -1,
        "decline in operating income":                           -1,
        "faces class action":                                    -1,
        "faces regulatory review":                               -1,
        "files for regulatory":                                  -1,
        "loses key contract":                                    -1,
        "recalls products in":                                   -1,
        "withdraws from":                                        -1,
        "steps down unexpectedly":                               -1,
        "warns of supply chain disruptions affecting":           -1,
        "reports rising costs pressuring margins":               -1,
        "delays product launch":                                 -1,
        "drop in new customer orders this quarter":              -1,
        "explores strategic alternatives":                       -1,
        "announces restructuring plan":                          -1,
        "announces major organizational restructuring":          -1,
        "faces regulatory review":                               -1,

        # Neutral
        "to present at":                                          0,
        "board meeting":                                          0,
        "publishes annual sustainability report":                 0,
        "schedules annual shareholder meeting for next month":    0,
        "files routine":                                          0,
        "begins scheduled maintenance":                           0,
        "to host investor day focused on":                        0,
        "names new":                                              0,
        "confirms participation":                                  0,
        "sees mixed results in":                                   0,
        "addresses investor concerns in open letter":              0,
        "announces significant capital expenditure plan for":      0,
        "to present at":                                           0,
    }

    records = []
    for session_id, grp in headlines_df.groupby('session'):
        # Sort by bar_ix to get temporal sequence
        grp = grp.sort_values('bar_ix')

        # Score each headline
        scored = []
        for _, row in grp.iterrows():
            h_lower = row['headline'].lower()
            direction = 0
            for kw, d in keyword_direction.items():
                if kw in h_lower:
                    direction = d
                    break
            scored.append({
                'bar_ix':    row['bar_ix'],
                'direction': direction,
            })

        df_s = pd.DataFrame(scored)

        # Split into first half and second half of seen session
        mid_bar  = df_s['bar_ix'].median()
        early    = df_s[df_s['bar_ix'] <= mid_bar]['direction'].values
        late     = df_s[df_s['bar_ix'] >  mid_bar]['direction'].values

        # Early and late sentiment
        early_sent = np.mean(early) if len(early) > 0 else 0.0
        late_sent  = np.mean(late)  if len(late)  > 0 else 0.0

        # Transition: did sentiment direction change?
        transition  = late_sent - early_sent  # + means improving, - means deteriorating

        # Consistency: same sign throughout?
        all_dirs    = df_s['direction'].values
        nonzero     = all_dirs[all_dirs != 0]
        consistency = (np.all(nonzero > 0) or np.all(nonzero < 0)) if len(nonzero) > 0 else True

        # Count sign changes in sequence
        sign_changes = 0
        prev = 0
        for d in all_dirs:
            if d != 0:
                if prev != 0 and d != prev:
                    sign_changes += 1
                prev = d

        # Momentum: is late sentiment stronger than early?
        sent_momentum = np.abs(late_sent) - np.abs(early_sent)

        # Transition regimes
        # neg→pos: started bad, ended good (recovery signal?)
        # pos→neg: started good, ended bad (deterioration signal?)
        # pos→pos: consistently positive
        # neg→neg: consistently negative
        if early_sent > 0 and late_sent > 0:
            regime = 'pos_pos'
        elif early_sent < 0 and late_sent < 0:
            regime = 'neg_neg'
        elif early_sent <= 0 and late_sent > 0:
            regime = 'neg_pos'
        elif early_sent >= 0 and late_sent < 0:
            regime = 'pos_neg'
        else:
            regime = 'neutral'

        records.append({
            'session':        session_id,
            'early_sent':     early_sent,
            'late_sent':      late_sent,
            'transition':     transition,       # late - early
            'consistency':    float(consistency),
            'sign_changes':   sign_changes,
            'sent_momentum':  sent_momentum,    # |late| - |early|
            'regime':         regime,
        })

    return pd.DataFrame(records).set_index('session')

# ── Run it ────────────────────────────────────────────────────────────────────

transitions = build_keyword_transitions(headlines_train)

print("=== Keyword transition correlations ===\n")
print(f"{'feature':>20}  {'r':>8}  {'p':>8}  {'sig':>5}")
print("-" * 48)

for col in ['early_sent', 'late_sent', 'transition', 
            'consistency', 'sign_changes', 'sent_momentum']:
    vals = transitions.reindex(y_train.index)[col].fillna(0)
    r, p = stats.pearsonr(vals, y_train)
    stars = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "—"
    print(f"  {col:20s}  {r:>+8.4f}  {p:>8.4f}  {stars}")

# ── Regime analysis ───────────────────────────────────────────────────────────
print("\n=== Sharpe by sentiment regime ===\n")
print(f"{'regime':>12}  {'n':>6}  {'mean_ret':>10}  {'sharpe':>8}")
print("-" * 42)

df_reg = transitions.reindex(y_train.index).copy()
df_reg['return'] = y_train

for regime, grp in df_reg.groupby('regime'):
    sh = sharpe(grp['return'])
    print(f"  {regime:>12}  {len(grp):>6}  "
          f"{grp['return'].mean():>+10.5f}  {sh:>8.4f}")

# ── CV test if any feature clears p < 0.05 ───────────────────────────────────
sig_cols = []
for col in ['early_sent', 'late_sent', 'transition',
            'consistency', 'sign_changes', 'sent_momentum']:
    vals = transitions.reindex(y_train.index)[col].fillna(0)
    r, p = stats.pearsonr(vals, y_train)
    if p < 0.05:
        sig_cols.append(col)

if sig_cols:
    print(f"\n=== CV Sharpe with significant transition features: {sig_cols} ===")

    trans_feats = transitions[sig_cols].reindex(y_train.index).fillna(0)
    X_combined  = pd.concat([X_train_raw, trans_feats], axis=1)
    feat_names  = FEATURES + sig_cols

    scaler_t  = StandardScaler()
    X_scaled_t = scaler_t.fit_transform(X_combined[feat_names])

    for label, X_cv in [("OHLC only",            X_scaled),
                         ("OHLC + transitions",   X_scaled_t)]:
        fold_sharpes = []
        for train_idx, val_idx in kf.split(X_cv):
            m = Ridge(alpha=100).fit(X_cv[train_idx], y[train_idx])
            p = make_positions(
                m.predict(X_cv[val_idx]),
                sensitivity=0.2
            )
            fold_sharpes.append(compute_sharpe(p, y[val_idx]))
        print(f"  {label:25s}  "
              f"sharpe={np.mean(fold_sharpes):.4f}  "
              f"std={np.std(fold_sharpes):.4f}")
else:
    print("\nNo transition features cleared p<0.05.")
    print("Submit original model.")

=== Keyword transition correlations ===

             feature         r         p    sig
------------------------------------------------
  early_sent             +0.0102    0.7466  —
  late_sent              +0.0496    0.1170  —
  transition             +0.0300    0.3425  —
  consistency            +0.0212    0.5034  —
  sign_changes           -0.0569    0.0721  —
  sent_momentum          +0.0154    0.6273  —

=== Sharpe by sentiment regime ===

      regime       n    mean_ret    sharpe
------------------------------------------
       neg_neg      64    +0.00232    1.7685
       neg_pos     249    +0.00329    2.5709
       neutral     184    +0.00139    1.0490
       pos_neg     197    +0.00309    2.3859
       pos_pos     306    +0.00555    4.5798

No transition features cleared p<0.05.
Submit original model.


In [ ]:
def analyse_cluster_correlation(cluster_df: pd.DataFrame,
                                 y_train: pd.Series) -> None:
    """
    For each cluster, test whether sessions are:
    - Positively correlated (same headlines → same direction)
    - Negatively correlated (same headlines → opposite direction)  
    - Uncorrelated (same headlines → random direction)
    """
    pos_corr    = []  # clusters where returns are positively consistent
    neg_corr    = []  # clusters where returns are negatively consistent
    uncorr      = []  # clusters where returns are random
    
    all_within_corrs = []
    
    for _, row in cluster_df.iterrows():
        sessions = row['cluster_sessions']
        if len(sessions) < 2:
            continue
        
        returns = y_train.loc[sessions].values
        mean_r  = np.mean(returns)
        
        # For pairs within cluster: compute all pairwise products
        # positive product = same direction, negative = opposite
        n = len(returns)
        pair_products = []
        for i in range(n):
            for j in range(i+1, n):
                pair_products.append(returns[i] * returns[j])
        
        mean_pair_product = np.mean(pair_products)
        all_within_corrs.append(mean_pair_product)
        
        if mean_pair_product > 0:
            pos_corr.append(row)
        elif mean_pair_product < 0:
            neg_corr.append(row)
        else:
            uncorr.append(row)
    
    print("=== Within-cluster return correlation structure ===\n")
    print(f"Positively correlated clusters:  {len(pos_corr):4d} "
          f"(same headlines → same direction)")
    print(f"Negatively correlated clusters:  {len(neg_corr):4d} "
          f"(same headlines → opposite direction)")
    print(f"Uncorrelated clusters:           {len(uncorr):4d} "
          f"(same headlines → random)")
    
    print(f"\nMean pairwise return product:    "
          f"{np.mean(all_within_corrs):+.8f}")
    print(f"Std pairwise return product:     "
          f"{np.std(all_within_corrs):.8f}")
    
    # t-test: is mean pairwise product significantly different from zero?
    t, p = scipy_stats.ttest_1samp(all_within_corrs, 0)
    print(f"t-test vs zero:                  t={t:+.4f}  p={p:.4f}")
    
    if p < 0.05:
        if np.mean(all_within_corrs) > 0:
            print("\n→ POSITIVE correlation confirmed.")
            print("  Same headlines predict same direction.")
            print("  Use mean cluster return as directional signal.")
        else:
            print("\n→ NEGATIVE (ANTI) correlation confirmed.")
            print("  Same headlines predict OPPOSITE directions.")
            print("  Use mean cluster return with FLIPPED sign as signal.")
    else:
        print("\n→ No significant correlation in either direction.")
        print("  Headline overlap does not predict return direction.")

    # ── Deeper: does cluster size moderate the effect? ────────────────────────
    print("\n=== Pairwise correlation by cluster size ===\n")
    print(f"{'cluster_size':>14}  {'n_clusters':>12}  "
          f"{'mean_pair_prod':>16}  {'p_value':>10}")
    print("-" * 58)
    
    size_buckets = [(2,2), (3,5), (6,10), (11,50), (51,9999)]
    for lo, hi in size_buckets:
        bucket = [
            r for _, r in cluster_df.iterrows()
            if lo <= r['n'] <= hi
        ]
        if not bucket:
            continue
        
        bucket_products = []
        for row in bucket:
            returns = y_train.loc[row['cluster_sessions']].values
            n = len(returns)
            for i in range(n):
                for j in range(i+1, n):
                    bucket_products.append(returns[i] * returns[j])
        
        if len(bucket_products) < 2:
            continue
            
        t, p = scipy_stats.ttest_1samp(bucket_products, 0)
        print(f"  n={lo:3d}–{hi:4d}:        "
              f"{len(bucket):>12}  "
              f"{np.mean(bucket_products):>+16.8f}  "
              f"{p:>10.4f}")

    # ── Visualise: histogram of pairwise products ─────────────────────────────
    print(f"\n=== Distribution of pairwise return products ===")
    print(f"(positive = correlated, negative = anti-correlated)\n")
    
    flat_products = []
    for _, row in cluster_df.iterrows():
        returns = y_train.loc[row['cluster_sessions']].values
        n = len(returns)
        for i in range(n):
            for j in range(i+1, n):
                flat_products.append(returns[i] * returns[j])
    
    flat_products = np.array(flat_products)
    pct_positive  = np.mean(flat_products > 0)
    pct_negative  = np.mean(flat_products < 0)
    
    print(f"  % positive pairs (correlated):      {pct_positive:.3f}")
    print(f"  % negative pairs (anti-correlated): {pct_negative:.3f}")
    print(f"  Ratio pos/neg:                      {pct_positive/pct_negative:.3f}")
    print(f"  (1.0 = random, >1 = correlated, <1 = anti-correlated)")

    # ── Final signal construction based on finding ────────────────────────────
    print("\n=== Signal construction recommendation ===")
    mean_prod = np.mean(all_within_corrs)
    t, p      = scipy_stats.ttest_1samp(all_within_corrs, 0)
    
    if p >= 0.05:
        print("No signal. Do not use cluster membership for trading.")
    elif mean_prod > 0:
        print("Use LOO cluster mean return as LONG signal.")
        print("Bigger position when cluster mean is strongly positive.")
    else:
        print("Use LOO cluster mean return as SHORT signal.")  
        print("Flip sign: negative cluster mean → go long.")
        print("Positive cluster mean → reduce position.")

analyse_cluster_correlation(cluster_df, y_train)

=== Within-cluster return correlation structure ===

Positively correlated clusters:     0 (same headlines → same direction)
Negatively correlated clusters:     0 (same headlines → opposite direction)
Uncorrelated clusters:              0 (same headlines → random)

Mean pairwise return product:    +nan
Std pairwise return product:     nan
t-test vs zero:                  t=+nan  p=nan

→ No significant correlation in either direction.
  Headline overlap does not predict return direction.

=== Pairwise correlation by cluster size ===

  cluster_size    n_clusters    mean_pair_prod     p_value
----------------------------------------------------------

=== Distribution of pairwise return products ===
(positive = correlated, negative = anti-correlated)

  % positive pairs (correlated):      nan
  % negative pairs (anti-correlated): nan
  Ratio pos/neg:                      nan
  (1.0 = random, >1 = correlated, <1 = anti-correlated)

=== Signal construction recommendation ===
Use LOO clust

/home/egork/Coding/hrt-datathon/.venv/lib/python3.14/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/egork/Coding/hrt-datathon/.venv/lib/python3.14/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/egork/Coding/hrt-datathon/.venv/lib/python3.14/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/egork/Coding/hrt-datathon/.venv/lib/python3.14/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/egork/Coding/hrt-datathon/.venv/lib/python3.14/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/tmp/ipykernel_953/31

In [ ]:
# Quick diagnostic to understand the structure
print("=== Headline repetition structure ===\n")

# How many unique headlines exist total?
all_headlines = headlines_train['headline'].unique()
print(f"Total unique headlines:     {len(all_headlines)}")
print(f"Total headline occurrences: {len(headlines_train)}")
print(f"Sessions:                   {headlines_train['session'].nunique()}")
print(f"Mean headlines per session: {len(headlines_train)/headlines_train['session'].nunique():.1f}")

# How often does each individual headline repeat?
headline_counts = headlines_train['headline'].value_counts()
print(f"\nIndividual headline repetition:")
print(f"  Appears exactly once: {(headline_counts == 1).sum()}")
print(f"  Appears 2–5 times:    {((headline_counts >= 2) & (headline_counts <= 5)).sum()}")
print(f"  Appears 6–10 times:   {((headline_counts >= 6) & (headline_counts <= 10)).sum()}")
print(f"  Appears 11–50 times:  {((headline_counts >= 11) & (headline_counts <= 50)).sum()}")
print(f"  Appears 50+ times:    {(headline_counts >= 50).sum()}")

# How much overlap exists between session pairs?
# Sample 500 session pairs and measure Jaccard similarity
sessions = headlines_train['session'].unique()
session_headline_sets = {
    s: set(headlines_train[headlines_train['session'] == s]['headline'])
    for s in sessions
}

np.random.seed(42)
sample_sessions = np.random.choice(sessions, min(100, len(sessions)), replace=False)

jaccard_scores = []
for i in range(len(sample_sessions)):
    for j in range(i+1, len(sample_sessions)):
        s1 = session_headline_sets[sample_sessions[i]]
        s2 = session_headline_sets[sample_sessions[j]]
        intersection = len(s1 & s2)
        union        = len(s1 | s2)
        jaccard_scores.append(intersection / union if union > 0 else 0)

print(f"\nPairwise Jaccard similarity (sample of session pairs):")
print(f"  Mean:    {np.mean(jaccard_scores):.4f}")
print(f"  Max:     {np.max(jaccard_scores):.4f}")
print(f"  Median:  {np.median(jaccard_scores):.4f}")
print(f"  % pairs with any overlap: "
      f"{np.mean(np.array(jaccard_scores) > 0):.3f}")

=== Headline repetition structure ===

Total unique headlines:     8463
Total headline occurrences: 9740
Sessions:                   1000
Mean headlines per session: 9.7

Individual headline repetition:
  Appears exactly once: 7305
  Appears 2–5 times:    1158
  Appears 6–10 times:   0
  Appears 11–50 times:  0
  Appears 50+ times:    0

Pairwise Jaccard similarity (sample of session pairs):
  Mean:    0.0001
  Max:     0.0667
  Median:  0.0000
  % pairs with any overlap: 0.003


In [ ]:
from scipy.sparse import csr_matrix
from sklearn.preprocessing import binarize

def build_partial_overlap_clusters(headlines_df: pd.DataFrame,
                                    y_train: pd.Series,
                                    min_overlap: int = 1) -> pd.DataFrame:
    """
    Cluster sessions that share at least min_overlap headlines.
    Uses a graph-based approach: sessions are nodes, 
    shared headlines are edges.
    """
    # Build session → headline set mapping
    session_sets = {}
    for session_id, grp in headlines_df.groupby('session'):
        session_sets[session_id] = set(grp['headline'].values)
    
    sessions = list(session_sets.keys())
    n        = len(sessions)
    
    # Build headline → sessions lookup for efficiency
    headline_to_sessions = defaultdict(set)
    for session_id, headlines in session_sets.items():
        for h in headlines:
            headline_to_sessions[h].add(session_id)
    
    # Find all session pairs with at least min_overlap shared headlines
    print(f"Finding session pairs with >= {min_overlap} shared headline(s)...")
    
    pair_overlaps = defaultdict(int)
    for h, h_sessions in headline_to_sessions.items():
        h_sessions = list(h_sessions)
        if len(h_sessions) < 2:
            continue
        # All pairs sharing this headline
        for i in range(len(h_sessions)):
            for j in range(i+1, len(h_sessions)):
                key = (min(h_sessions[i], h_sessions[j]),
                       max(h_sessions[i], h_sessions[j]))
                pair_overlaps[key] += 1
    
    # Filter to pairs meeting minimum overlap threshold
    valid_pairs = {pair: count for pair, count in pair_overlaps.items()
                   if count >= min_overlap}
    
    print(f"Total session pairs with >= {min_overlap} overlap: {len(valid_pairs)}")
    
    if not valid_pairs:
        print("No overlapping session pairs found.")
        return pd.DataFrame(), {}
    
    # Overlap distribution
    overlap_counts = list(valid_pairs.values())
    print(f"Shared headlines per pair:")
    print(f"  Mean:   {np.mean(overlap_counts):.2f}")
    print(f"  Max:    {np.max(overlap_counts)}")
    print(f"  Median: {np.median(overlap_counts):.1f}")
    
    # Build connected components (clusters) using union-find
    parent = {s: s for s in sessions}
    
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    
    def union(x, y):
        parent[find(x)] = find(y)
    
    for (s1, s2) in valid_pairs:
        union(s1, s2)
    
    # Group sessions into clusters
    clusters = defaultdict(list)
    for s in sessions:
        clusters[find(s)].append(s)
    
    multi_session_clusters = {k: v for k, v in clusters.items() 
                               if len(v) > 1}
    
    print(f"\n=== Cluster structure ===")
    print(f"Total clusters (2+ sessions): {len(multi_session_clusters)}")
    
    sizes = sorted([len(v) for v in multi_session_clusters.values()], 
                   reverse=True)
    if sizes:
        print(f"Largest cluster:  {sizes[0]} sessions")
        print(f"Median size:      {np.median(sizes):.1f} sessions")
        print(f"Sessions covered: {sum(sizes)} "
              f"({sum(sizes)/n:.1%} of all sessions)")
        print(f"Size distribution:")
        for threshold in [2, 3, 5, 10, 20]:
            n_clusters = sum(1 for s in sizes if s >= threshold)
            print(f"  clusters with {threshold:>2}+ sessions: {n_clusters}")
    
    # ── Analyse within-cluster return correlation ─────────────────────────────
    print(f"\n=== Within-cluster pairwise return correlation ===")
    
    all_pair_products = []
    cluster_records   = []
    
    for root, cluster_sessions in multi_session_clusters.items():
        valid = [s for s in cluster_sessions if s in y_train.index]
        if len(valid) < 2:
            continue
        
        returns = y_train.loc[valid].values
        
        pair_products = []
        for i in range(len(returns)):
            for j in range(i+1, len(returns)):
                pp = returns[i] * returns[j]
                pair_products.append(pp)
                all_pair_products.append(pp)
        
        cluster_records.append({
            'sessions':         valid,
            'n':                len(valid),
            'mean_return':      np.mean(returns),
            'std_return':       np.std(returns),
            'mean_pair_prod':   np.mean(pair_products),
            'consistency':      np.mean(
                                    np.sign(returns) == np.sign(np.mean(returns))
                                ),
        })
    
    cluster_df = pd.DataFrame(cluster_records)
    
    if len(all_pair_products) == 0:
        print("No valid pairs found.")
        return cluster_df, valid_pairs
    
    all_pair_products = np.array(all_pair_products)
    pct_pos = np.mean(all_pair_products > 0)
    pct_neg = np.mean(all_pair_products < 0)
    
    print(f"Total pairs analysed:           {len(all_pair_products)}")
    print(f"% positive (correlated):        {pct_pos:.3f}")
    print(f"% negative (anti-correlated):   {pct_neg:.3f}")
    print(f"Ratio pos/neg:                  {pct_pos/pct_neg:.3f}  "
          f"(1.0 = random)")
    print(f"Mean pairwise product:          {all_pair_products.mean():+.8f}")
    
    t, p = scipy_stats.ttest_1samp(all_pair_products, 0)
    print(f"t-test vs zero:                 t={t:+.4f}  p={p:.4f}")
    
    if p < 0.05:
        direction = "POSITIVE" if all_pair_products.mean() > 0 else "NEGATIVE"
        print(f"\n→ Significant {direction} correlation found!")
        print(f"  Sessions sharing headlines move in "
              f"{'same' if direction == 'POSITIVE' else 'opposite'} direction.")
    else:
        print(f"\n→ No significant correlation. p={p:.4f}")

    # ── LOO signal: use cluster-mates' returns as prior ───────────────────────
    print(f"\n=== Leave-one-out cluster signal ===")
    
    session_to_cluster = {}
    for root, cluster_sessions in multi_session_clusters.items():
        for s in cluster_sessions:
            session_to_cluster[s] = cluster_sessions
    
    loo_signal = {}
    for session_id, cluster_sessions in session_to_cluster.items():
        if session_id not in y_train.index:
            continue
        others = [s for s in cluster_sessions 
                  if s != session_id and s in y_train.index]
        if others:
            loo_signal[session_id] = y_train.loc[others].mean()
    
    signal = pd.Series(loo_signal, name='cluster_signal')
    n_covered = len(signal)
    
    print(f"Sessions with LOO signal: {n_covered} "
          f"({n_covered/len(y_train):.1%})")
    
    if n_covered > 10:
        aligned = signal.reindex(y_train.index).fillna(0)
        covered = signal.dropna()
        r, p_corr = scipy_stats.pearsonr(covered, y_train.loc[covered.index])
        print(f"LOO signal correlation:   r={r:+.4f}  p={p_corr:.4f}")
        
        # Sharpe comparison
        cluster_rets  = y_train.loc[covered.index]
        pos_long      = np.ones(len(cluster_rets))
        pos_signal    = np.sign(covered.values)
        pos_signal[pos_signal == 0] = 1
        
        print(f"\nSharpe — all sessions (always long):      "
              f"{sharpe(y_train):.4f}")
        print(f"Sharpe — clustered sessions, always long: "
              f"{sharpe(cluster_rets):.4f}")
        print(f"Sharpe — clustered sessions, LOO signal:  "
              f"{sharpe(pd.Series(pos_signal * cluster_rets.values)):.4f}")
    
    return cluster_df, valid_pairs

# ── Run with min_overlap=1 ────────────────────────────────────────────────────

cluster_df, valid_pairs = build_partial_overlap_clusters(
    headlines_train, y_train, min_overlap=1
)

Finding session pairs with >= 1 shared headline(s)...
Total session pairs with >= 1 overlap: 1356
Shared headlines per pair:
  Mean:   1.01
  Max:    3
  Median: 1.0

=== Cluster structure ===
Total clusters (2+ sessions): 14
Largest cluster:  883 sessions
Median size:      2.0 sessions
Sessions covered: 912 (91.2% of all sessions)
Size distribution:
  clusters with  2+ sessions: 14
  clusters with  3+ sessions: 4
  clusters with  5+ sessions: 1
  clusters with 10+ sessions: 1
  clusters with 20+ sessions: 1

=== Within-cluster pairwise return correlation ===
Total pairs analysed:           389422
% positive (correlated):        0.502
% negative (anti-correlated):   0.489
Ratio pos/neg:                  1.027  (1.0 = random)
Mean pairwise product:          +0.00000990
t-test vs zero:                 t=+13.8344  p=0.0000

→ Significant POSITIVE correlation found!
  Sessions sharing headlines move in same direction.

=== Leave-one-out cluster signal ===
Sessions with LOO signal: 912 (91.

In [ ]:
# Verify: the giant cluster is an artifact of transitive chaining
# Check direct overlap between sessions in the giant cluster

giant_cluster = max(
    [v for v in 
     {root: sessions for root, sessions in 
      {s: [] for s in headlines_train['session'].unique()}.items()}.values()
    ],
    key=len
)

# Better approach: only use DIRECT pairs, ignore transitive connections
print("=== Direct pair signal (no transitive chaining) ===\n")

# For each session, find sessions with direct headline overlap
# and use ONLY those as signal — not the full connected component

def build_direct_pair_signal(headlines_df: pd.DataFrame,
                              y_train: pd.Series,
                              min_overlap: int = 2) -> pd.Series:
    """
    Skip union-find entirely. For each session, compute
    a signal directly from sessions with >= min_overlap 
    shared headlines — no transitive connections.
    
    min_overlap=2 filters out the weakest single-headline 
    connections that created the giant spurious cluster.
    """
    headline_to_sessions = defaultdict(set)
    for _, row in headlines_df.iterrows():
        headline_to_sessions[row['headline']].add(row['session'])
    
    # Count direct overlaps between all session pairs
    pair_overlaps = defaultdict(int)
    for h, h_sessions in headline_to_sessions.items():
        h_sessions = list(h_sessions)
        if len(h_sessions) < 2:
            continue
        for i in range(len(h_sessions)):
            for j in range(i+1, len(h_sessions)):
                key = (min(h_sessions[i], h_sessions[j]),
                       max(h_sessions[i], h_sessions[j]))
                pair_overlaps[key] += 1
    
    # Filter to pairs with sufficient direct overlap
    strong_pairs = {pair: count for pair, count in pair_overlaps.items()
                    if count >= min_overlap}
    
    print(f"Pairs with >= {min_overlap} shared headlines: {len(strong_pairs)}")
    
    if not strong_pairs:
        print("No strong pairs found.")
        return pd.Series(dtype=float)
    
    # Build direct LOO signal — no chaining
    session_direct_signal = defaultdict(list)
    for (s1, s2), overlap in strong_pairs.items():
        weight = overlap  # weight by number of shared headlines
        if s1 in y_train.index:
            session_direct_signal[s2].append((y_train[s1], weight))
        if s2 in y_train.index:
            session_direct_signal[s1].append((y_train[s2], weight))
    
    # Weighted mean of direct neighbours' returns
    loo_signal = {}
    for session_id, neighbour_returns in session_direct_signal.items():
        if session_id not in y_train.index:
            continue
        returns, weights = zip(*neighbour_returns)
        loo_signal[session_id] = np.average(returns, weights=weights)
    
    signal = pd.Series(loo_signal, name='direct_pair_signal')
    n_covered = len(signal)
    
    print(f"Sessions covered:           {n_covered} "
          f"({n_covered/len(y_train):.1%})")
    
    if n_covered < 5:
        print("Too few sessions covered to analyse.")
        return signal
    
    covered      = signal.dropna()
    actual       = y_train.loc[covered.index]
    r, p         = scipy_stats.pearsonr(covered, actual)
    print(f"Direct signal correlation:  r={r:+.4f}  p={p:.4f}")
    
    # Pairwise product test on direct pairs only
    direct_products = []
    for (s1, s2) in strong_pairs:
        if s1 in y_train.index and s2 in y_train.index:
            direct_products.append(y_train[s1] * y_train[s2])
    
    if direct_products:
        direct_products = np.array(direct_products)
        pct_pos = np.mean(direct_products > 0)
        t, p_t  = scipy_stats.ttest_1samp(direct_products, 0)
        print(f"\nDirect pair products:")
        print(f"  n pairs:        {len(direct_products)}")
        print(f"  % positive:     {pct_pos:.3f}")
        print(f"  mean product:   {direct_products.mean():+.8f}")
        print(f"  t={t:+.4f}  p={p_t:.4f}")
    
    return signal

print("=== min_overlap=2 (at least 2 shared headlines) ===")
signal_2 = build_direct_pair_signal(headlines_train, y_train, min_overlap=2)

print("\n=== min_overlap=3 (at least 3 shared headlines) ===")
signal_3 = build_direct_pair_signal(headlines_train, y_train, min_overlap=3)

=== Direct pair signal (no transitive chaining) ===

=== min_overlap=2 (at least 2 shared headlines) ===
Pairs with >= 2 shared headlines: 8
Sessions covered:           15 (1.5%)
Direct signal correlation:  r=+0.1309  p=0.6419

Direct pair products:
  n pairs:        8
  % positive:     0.750
  mean product:   +0.00011107
  t=+1.1636  p=0.2827

=== min_overlap=3 (at least 3 shared headlines) ===
Pairs with >= 3 shared headlines: 1
Sessions covered:           2 (0.2%)
Too few sessions covered to analyse.


In [ ]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict
from scipy import stats as scipy_stats

# ── Load everything ───────────────────────────────────────────────────────────

seen_train      = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_train.parquet"),       engine="fastparquet")
unseen_train    = pd.read_parquet(os.path.join(DATA_DIR, "bars_unseen_train.parquet"),     engine="fastparquet")
headlines_train = pd.read_parquet(os.path.join(DATA_DIR, "headlines_seen_train.parquet"),  engine="fastparquet")

public_bars     = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_public_test.parquet"),     engine="fastparquet")
private_bars    = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_private_test.parquet"),    engine="fastparquet")
public_hl       = pd.read_parquet(os.path.join(DATA_DIR, "headlines_seen_public_test.parquet"),  engine="fastparquet")
private_hl      = pd.read_parquet(os.path.join(DATA_DIR, "headlines_seen_private_test.parquet"), engine="fastparquet")

# Training returns
mid_price = seen_train.groupby("session")["close"].last()
end_price = unseen_train.groupby("session")["close"].last()
y_train   = (end_price / mid_price - 1).rename("return")

# ── Build lookup structures ───────────────────────────────────────────────────

# Full headline set per session
train_session_headlines = (
    headlines_train
    .groupby("session")["headline"]
    .apply(set)
    .to_dict()
)

def get_test_session_headlines(hl_df):
    return (
        hl_df
        .groupby("session")["headline"]
        .apply(set)
        .to_dict()
    )

public_session_hl  = get_test_session_headlines(public_hl)
private_session_hl = get_test_session_headlines(private_hl)

# Inverted index: headline → training sessions containing it
headline_to_train_sessions = defaultdict(set)
for session_id, hl_set in train_session_headlines.items():
    for h in hl_set:
        headline_to_train_sessions[h].add(session_id)

print(f"Train sessions:   {len(train_session_headlines)}")
print(f"Public test:      {len(public_session_hl)}")
print(f"Private test:     {len(private_session_hl)}")
print(f"Unique train headlines: {len(headline_to_train_sessions)}")

# ── Approach 1: Exact full-set match ─────────────────────────────────────────
# Test session has IDENTICAL headline set to a training session

print("\n" + "="*60)
print("APPROACH 1: Exact full headline set match")
print("="*60)

def exact_match(test_session_hl, train_session_headlines, y_train):
    results = {}
    # Invert train: frozenset → session
    train_set_to_session = {
        frozenset(hl_set): sid
        for sid, hl_set in train_session_headlines.items()
    }
    for test_sid, test_hl in test_session_hl.items():
        key = frozenset(test_hl)
        if key in train_set_to_session:
            train_sid = train_set_to_session[key]
            results[test_sid] = {
                'train_session':  train_sid,
                'train_return':   y_train[train_sid],
                'match_type':     'exact',
                'n_shared':       len(test_hl),
                'n_test':         len(test_hl),
                'n_train':        len(test_hl),
                'jaccard':        1.0,
            }
    return results

pub_exact  = exact_match(public_session_hl,  train_session_headlines, y_train)
priv_exact = exact_match(private_session_hl, train_session_headlines, y_train)
print(f"Public  exact matches:  {len(pub_exact)}/{len(public_session_hl)}")
print(f"Private exact matches:  {len(priv_exact)}/{len(private_session_hl)}")

# ── Approach 2: Majority overlap ──────────────────────────────────────────────
# Test session shares >50% of its headlines with a training session

print("\n" + "="*60)
print("APPROACH 2: Majority overlap (Jaccard > 0.5)")
print("="*60)

def majority_match(test_session_hl, train_session_headlines, 
                   y_train, jaccard_threshold=0.5):
    results = {}
    for test_sid, test_hl in test_session_hl.items():
        best_jaccard  = 0
        best_train    = None
        best_shared   = 0

        # Find candidate train sessions via inverted index
        candidate_trains = defaultdict(int)
        for h in test_hl:
            for train_sid in headline_to_train_sessions.get(h, []):
                candidate_trains[train_sid] += 1

        for train_sid, n_shared in candidate_trains.items():
            train_hl = train_session_headlines[train_sid]
            union    = len(test_hl | train_hl)
            jaccard  = n_shared / union if union > 0 else 0

            if jaccard > best_jaccard:
                best_jaccard = jaccard
                best_train   = train_sid
                best_shared  = n_shared

        if best_jaccard >= jaccard_threshold:
            results[test_sid] = {
                'train_session': best_train,
                'train_return':  y_train[best_train],
                'match_type':    'majority',
                'n_shared':      best_shared,
                'n_test':        len(test_hl),
                'n_train':       len(train_session_headlines[best_train]),
                'jaccard':       best_jaccard,
            }
    return results

for threshold in [0.9, 0.7, 0.5, 0.3]:
    pub_maj = majority_match(
        public_session_hl, train_session_headlines, 
        y_train, threshold
    )
    priv_maj = majority_match(
        private_session_hl, train_session_headlines,
        y_train, threshold
    )
    print(f"  Jaccard>{threshold:.1f}:  "
          f"public={len(pub_maj):3d}/{len(public_session_hl)}  "
          f"private={len(priv_maj):3d}/{len(private_session_hl)}")

# ── Approach 3: Any shared headline, weighted by overlap ─────────────────────
# Even 1 shared headline gives us a weak prior

print("\n" + "="*60)
print("APPROACH 3: Weighted signal from all overlapping train sessions")
print("="*60)

def weighted_overlap_signal(test_session_hl, train_session_headlines,
                             y_train, min_shared=1):
    """
    For each test session, find ALL training sessions with
    any headline overlap. Weight each training session's return
    by number of shared headlines. Aggregate into a single signal.
    """
    results = {}
    for test_sid, test_hl in test_session_hl.items():
        candidate_trains = defaultdict(int)
        for h in test_hl:
            for train_sid in headline_to_train_sessions.get(h, []):
                candidate_trains[train_sid] += 1

        # Filter by minimum shared headlines
        valid = {s: c for s, c in candidate_trains.items()
                 if c >= min_shared}

        if not valid:
            continue

        # Weighted mean return
        weights = np.array(list(valid.values()), dtype=float)
        returns = np.array([y_train[s] for s in valid.keys()])
        weighted_return = np.average(returns, weights=weights)
        total_shared    = sum(valid.values())
        best_match      = max(valid, key=valid.get)

        results[test_sid] = {
            'signal':           weighted_return,
            'n_train_sessions': len(valid),
            'total_shared':     total_shared,
            'best_jaccard':     valid[best_match] / len(
                                    test_hl | train_session_headlines[best_match]
                                ),
            'best_train':       best_match,
            'best_return':      y_train[best_match],
        }

    return results

pub_weighted  = weighted_overlap_signal(public_session_hl,
                                         train_session_headlines, y_train)
priv_weighted = weighted_overlap_signal(private_session_hl,
                                         train_session_headlines, y_train)

print(f"Public  sessions with any overlap:  "
      f"{len(pub_weighted)}/{len(public_session_hl)} "
      f"({len(pub_weighted)/len(public_session_hl):.1%})")
print(f"Private sessions with any overlap:  "
      f"{len(priv_weighted)}/{len(private_session_hl)} "
      f"({len(priv_weighted)/len(private_session_hl):.1%})")

if pub_weighted:
    jaccards = [v['best_jaccard'] for v in pub_weighted.values()]
    shared   = [v['total_shared'] for v in pub_weighted.values()]
    print(f"\nPublic overlap stats:")
    print(f"  Mean best Jaccard:    {np.mean(jaccards):.4f}")
    print(f"  Max best Jaccard:     {np.max(jaccards):.4f}")
    print(f"  Mean shared headlines:{np.mean(shared):.2f}")

# ── Approach 4: OHLC fingerprint matching ────────────────────────────────────
# Same session = same price path. Find test sessions whose
# OHLC pattern most closely matches a training session.

print("\n" + "="*60)
print("APPROACH 4: OHLC price path fingerprint matching")
print("="*60)

def build_price_fingerprint(bars_df):
    """
    Represent each session as a normalised sequence of returns.
    Sessions with identical data generating process should have
    similar (or identical) fingerprints.
    """
    fingerprints = {}
    for session_id, grp in bars_df.groupby("session"):
        grp    = grp.sort_values("bar_ix")
        closes = grp["close"].values
        # Normalise to start at 1 — already done per problem statement
        # Use bar-to-bar returns as fingerprint
        if len(closes) > 1:
            returns = np.diff(closes) / closes[:-1]
            fingerprints[session_id] = returns
    return fingerprints

train_fps  = build_price_fingerprint(seen_train)
public_fps = build_price_fingerprint(public_bars)
priv_fps   = build_price_fingerprint(private_bars)

def find_closest_ohlc_match(test_fps, train_fps, y_train, 
                             top_n=3, correlation_threshold=0.95):
    """
    For each test session, find the training session whose
    return sequence is most correlated with it.
    Perfect match (r=1.0) = same session reused.
    """
    results = {}
    train_ids   = list(train_fps.keys())
    train_array = np.array([train_fps[s] for s in train_ids 
                             if len(train_fps[s]) > 0])

    for test_sid, test_ret in test_fps.items():
        if len(test_ret) == 0:
            continue

        # Correlate test fingerprint with all training fingerprints
        correlations = []
        for i, train_sid in enumerate(train_ids):
            train_ret = train_fps[train_sid]
            min_len   = min(len(test_ret), len(train_ret))
            if min_len < 3:
                correlations.append(0)
                continue
            r, _ = scipy_stats.pearsonr(
                test_ret[:min_len], train_ret[:min_len]
            )
            correlations.append(r if not np.isnan(r) else 0)

        correlations = np.array(correlations)
        best_idx     = np.argmax(correlations)
        best_corr    = correlations[best_idx]
        best_train   = train_ids[best_idx]

        if best_corr >= correlation_threshold:
            results[test_sid] = {
                'train_session': best_train,
                'correlation':   best_corr,
                'train_return':  y_train[best_train],
                'signal':        y_train[best_train],
            }

    return results

print("Matching public test OHLC fingerprints to training...")
for threshold in [1.0, 0.999, 0.99, 0.95]:
    pub_ohlc = find_closest_ohlc_match(
        public_fps, train_fps, y_train,
        correlation_threshold=threshold
    )
    print(f"  r>={threshold:.3f}:  "
          f"{len(pub_ohlc):3d}/{len(public_fps)} public sessions matched")

print("\nMatching private test OHLC fingerprints to training...")
for threshold in [1.0, 0.999, 0.99, 0.95]:
    priv_ohlc = find_closest_ohlc_match(
        priv_fps, train_fps, y_train,
        correlation_threshold=threshold
    )
    print(f"  r>={threshold:.3f}:  "
          f"{len(priv_ohlc):3d}/{len(priv_fps)} private sessions matched")

# ── Summary and best signal ───────────────────────────────────────────────────

print("\n" + "="*60)
print("SUMMARY: Match coverage across all approaches")
print("="*60)
print(f"{'Approach':<35} {'Public':>10} {'Private':>10}")
print("-"*55)
print(f"{'1. Exact headline set':<35} "
      f"{len(pub_exact):>10} {len(priv_exact):>10}")

pub_maj_best  = majority_match(public_session_hl,
                                train_session_headlines, y_train, 0.5)
priv_maj_best = majority_match(private_session_hl,
                                train_session_headlines, y_train, 0.5)
print(f"{'2. Majority overlap (J>0.5)':<35} "
      f"{len(pub_maj_best):>10} {len(priv_maj_best):>10}")
print(f"{'3. Any overlap (weighted)':<35} "
      f"{len(pub_weighted):>10} {len(priv_weighted):>10}")

pub_ohlc_exact = find_closest_ohlc_match(
    public_fps, train_fps, y_train, correlation_threshold=1.0
)
priv_ohlc_exact = find_closest_ohlc_match(
    priv_fps, train_fps, y_train, correlation_threshold=1.0
)
print(f"{'4. Exact OHLC fingerprint (r=1.0)':<35} "
      f"{len(pub_ohlc_exact):>10} {len(priv_ohlc_exact):>10}")

Train sessions:   1000
Public test:      10000
Private test:     9999
Unique train headlines: 8463

APPROACH 1: Exact full headline set match
Public  exact matches:  0/10000
Private exact matches:  0/9999

APPROACH 2: Majority overlap (Jaccard > 0.5)
  Jaccard>0.9:  public=  0/10000  private=  0/9999
  Jaccard>0.7:  public=  0/10000  private=  0/9999
  Jaccard>0.5:  public=  0/10000  private=  0/9999
  Jaccard>0.3:  public=  0/10000  private=  0/9999

APPROACH 3: Weighted signal from all overlapping train sessions
Public  sessions with any overlap:  9184/10000 (91.8%)
Private sessions with any overlap:  9107/9999 (91.1%)

Public overlap stats:
  Mean best Jaccard:    0.0542
  Max best Jaccard:     0.2000
  Mean shared headlines:3.07

APPROACH 4: OHLC price path fingerprint matching
Matching public test OHLC fingerprints to training...


KeyboardInterrupt: 

In [ ]:
import numpy as np
from scipy import stats as scipy_stats

def find_ohlc_matches_fast(test_fps, train_fps, y_train,
                            threshold=0.999):
    """
    Vectorised OHLC fingerprint matching.
    Replaces nested loop with matrix multiplication.
    Runs in seconds instead of hours.
    """
    # Align all sessions to same length (min common length)
    train_ids = [s for s in train_fps if len(train_fps[s]) > 1]
    test_ids  = [s for s in test_fps  if len(test_fps[s])  > 1]

    min_len = min(
        min(len(train_fps[s]) for s in train_ids),
        min(len(test_fps[s])  for s in test_ids),
    )

    # Build matrices — rows = sessions, cols = bar returns
    T = np.array([train_fps[s][:min_len] for s in train_ids])  # (n_train, L)
    X = np.array([test_fps[s][:min_len]  for s in test_ids])   # (n_test,  L)

    # Z-score each row (required for correlation via dot product)
    def row_zscore(M):
        mu  = M.mean(axis=1, keepdims=True)
        std = M.std(axis=1,  keepdims=True) + 1e-8
        return (M - mu) / std

    T_z = row_zscore(T)
    X_z = row_zscore(X)

    # Correlation matrix: (n_test, n_train)
    # corr[i,j] = pearson(test_i, train_j)
    corr_matrix = X_z @ T_z.T / min_len  # (n_test, n_train)

    print(f"Correlation matrix: {corr_matrix.shape}")
    print(f"Max correlation:    {corr_matrix.max():.6f}")
    print(f"Mean correlation:   {corr_matrix.mean():.6f}")

    # For each test session: best matching train session
    best_idx  = corr_matrix.argmax(axis=1)  # (n_test,)
    best_corr = corr_matrix.max(axis=1)     # (n_test,)

    # Distribution of best correlations
    print(f"\nBest match correlation distribution:")
    for pct in [50, 75, 90, 95, 99, 100]:
        print(f"  p{pct:3d}: {np.percentile(best_corr, pct):.6f}")

    # Sessions above threshold
    matched_mask = best_corr >= threshold
    print(f"\nMatched sessions (r>={threshold}):")
    print(f"  {matched_mask.sum()}/{len(test_ids)} "
          f"({matched_mask.mean():.1%})")

    # Build results
    results = {}
    for i, test_sid in enumerate(test_ids):
        if best_corr[i] >= threshold:
            train_sid = train_ids[best_idx[i]]
            results[test_sid] = {
                'train_session': train_sid,
                'correlation':   best_corr[i],
                'train_return':  y_train[train_sid],
            }

    return results, corr_matrix, test_ids, train_ids

# ── Run fast matching ─────────────────────────────────────────────────────────

print("=== Fast OHLC fingerprint matching ===\n")

import time
t0 = time.time()

pub_matches, pub_corr, pub_ids, train_ids = find_ohlc_matches_fast(
    public_fps, train_fps, y_train, threshold=0.999
)
print(f"Public  done in {time.time()-t0:.1f}s")

t0 = time.time()
priv_matches, priv_corr, priv_ids, _ = find_ohlc_matches_fast(
    priv_fps, train_fps, y_train, threshold=0.999
)
print(f"Private done in {time.time()-t0:.1f}s")

# ── What does the correlation structure look like? ────────────────────────────

print("\n=== Correlation structure analysis ===")

# Are there ANY perfect matches (r=1.0)?
pub_perfect  = (pub_corr  == 1.0).any(axis=1).sum()
priv_perfect = (priv_corr == 1.0).any(axis=1).sum()
print(f"Perfect matches (r=1.000): public={pub_perfect}, private={priv_perfect}")

# Sweep thresholds
print(f"\n{'threshold':>12}  {'public':>8}  {'private':>8}")
print("-" * 32)
for thresh in [1.000, 0.999, 0.998, 0.995, 0.990, 0.980, 0.950, 0.8, 0.75, 0.5]:
    pub_n  = (pub_corr.max(axis=1)  >= thresh).sum()
    priv_n = (priv_corr.max(axis=1) >= thresh).sum()
    print(f"  r>={thresh:.3f}:  {pub_n:>8}  {priv_n:>8}")

# ── If matches found: validate signal quality ─────────────────────────────────

if len(pub_matches) > 0 or len(priv_matches) > 0:
    print("\n=== Match signal quality ===")

    all_matches = {**pub_matches, **priv_matches}
    corrs   = [v['correlation']  for v in all_matches.values()]
    returns = [v['train_return'] for v in all_matches.values()]

    print(f"Total matched test sessions: {len(all_matches)}")
    print(f"Mean match correlation:      {np.mean(corrs):.6f}")
    print(f"Mean matched train return:   {np.mean(returns):+.5f}")
    print(f"% positive train returns:    {np.mean(np.array(returns)>0):.3f}")

    # If r=1.0 matches exist, the return IS the signal
    perfect = {k: v for k, v in all_matches.items() 
               if v['correlation'] > 0.9999}
    if perfect:
        print(f"\nPerfect matches: {len(perfect)}")
        print(f"These test sessions ARE training sessions reused.")
        print(f"Train return = exact future return for these sessions.")

=== Fast OHLC fingerprint matching ===

Correlation matrix: (10000, 1000)
Max correlation:    0.682960
Mean correlation:   -0.000061

Best match correlation distribution:
  p 50: 0.444461
  p 75: 0.475366
  p 90: 0.508012
  p 95: 0.527677
  p 99: 0.569828
  p100: 0.682960

Matched sessions (r>=0.999):
  0/10000 (0.0%)
Public  done in 0.1s
Correlation matrix: (10000, 1000)
Max correlation:    0.660435
Mean correlation:   0.000062

Best match correlation distribution:
  p 50: 0.445381
  p 75: 0.475645
  p 90: 0.505539
  p 95: 0.526335
  p 99: 0.571336
  p100: 0.660435

Matched sessions (r>=0.999):
  0/10000 (0.0%)
Private done in 0.1s

=== Correlation structure analysis ===
Perfect matches (r=1.000): public=0, private=0

   threshold    public   private
--------------------------------
  r>=1.000:         0         0
  r>=0.999:         0         0
  r>=0.998:         0         0
  r>=0.995:         0         0
  r>=0.990:         0         0
  r>=0.980:         0         0
  r>=0.950:  

In [ ]:
# Instead of threshold matching, use the correlation itself as a signal
# High correlation to a training session → weight that session's return

def build_correlation_signal(corr_matrix, test_ids, train_ids, 
                              y_train, top_n=5, min_corr=0.5):
    """
    For each test session, take the top_n most correlated
    training sessions and compute a weighted mean return.
    Weight = correlation^2 (higher correlation = more weight)
    
    This is a k-nearest-neighbours approach in price-path space.
    """
    results = {}
    
    for i, test_sid in enumerate(test_ids):
        row = corr_matrix[i]  # correlations to all train sessions
        
        # Get top_n matches above min_corr
        above_thresh = np.where(row >= min_corr)[0]
        if len(above_thresh) == 0:
            results[test_sid] = {
                'signal':    0.0,
                'n_matches': 0,
                'mean_corr': 0.0,
            }
            continue
        
        # Sort by correlation descending
        sorted_idx = above_thresh[np.argsort(row[above_thresh])[::-1]]
        top_idx    = sorted_idx[:top_n]
        
        top_corrs   = row[top_idx]
        top_returns = np.array([y_train[train_ids[j]] for j in top_idx])
        
        # Weight by correlation squared
        weights          = top_corrs ** 2
        weighted_return  = np.average(top_returns, weights=weights)
        
        results[test_sid] = {
            'signal':        weighted_return,
            'n_matches':     len(top_idx),
            'mean_corr':     top_corrs.mean(),
            'best_corr':     top_corrs[0],
            'best_return':   top_returns[0],
        }
    
    return results

# ── First validate on training data using LOO ─────────────────────────────────
# Build train-vs-train correlation matrix
# For each train session, find most similar OTHER train sessions
# and check if their returns predict its return

print("=== Validating KNN signal on training data (LOO) ===\n")

# Build train-train correlation matrix
train_ids_list = list(train_fps.keys())
min_len = min(len(train_fps[s]) for s in train_ids_list)

T = np.array([train_fps[s][:min_len] for s in train_ids_list])

def row_zscore(M):
    mu  = M.mean(axis=1, keepdims=True)
    std = M.std(axis=1,  keepdims=True) + 1e-8
    return (M - mu) / std

T_z = row_zscore(T)
train_corr = T_z @ T_z.T / min_len  # (n_train, n_train)

# Zero out self-correlation (diagonal)
np.fill_diagonal(train_corr, 0)

print(f"Train-train correlation matrix: {train_corr.shape}")
print(f"Max off-diagonal correlation:   {train_corr.max():.6f}")
print(f"Mean off-diagonal correlation:  {train_corr.mean():.6f}")

print(f"\nTop correlation distribution:")
# For each session, best matching OTHER session
best_train_corr = train_corr.max(axis=1)
for pct in [50, 75, 90, 95, 99, 100]:
    print(f"  p{pct:3d}: {np.percentile(best_train_corr, pct):.6f}")

# ── LOO KNN signal on training data ──────────────────────────────────────────

print("\n=== LOO KNN signal correlations ===\n")

y_array = y_train.loc[train_ids_list].values

for top_n in [1, 3, 5, 10]:
    for min_corr in [0.3, 0.4, 0.5, 0.6]:
        signals = []
        actuals = []
        
        for i in range(len(train_ids_list)):
            row = train_corr[i]
            above = np.where(row >= min_corr)[0]
            
            if len(above) == 0:
                signals.append(0.0)
            else:
                sorted_idx   = above[np.argsort(row[above])[::-1]]
                top_idx      = sorted_idx[:top_n]
                top_corrs    = row[top_idx]
                top_returns  = y_array[top_idx]
                weights      = top_corrs ** 2
                signals.append(np.average(top_returns, weights=weights))
            
            actuals.append(y_array[i])
        
        signals = np.array(signals)
        actuals = np.array(actuals)
        
        # Only correlate where we have a signal
        has_signal = signals != 0.0
        n_covered  = has_signal.sum()
        
        if n_covered < 10:
            continue
        
        r, p = scipy_stats.pearsonr(signals[has_signal], 
                                     actuals[has_signal])
        print(f"  top_n={top_n}  min_corr={min_corr:.1f}  "
              f"coverage={n_covered:4d}/1000  "
              f"r={r:+.4f}  p={p:.4f}")

# ── If signal is real, build CV Sharpe test ───────────────────────────────────

print("\n=== CV Sharpe with KNN signal (best params) ===\n")

# Use full correlation row as signal — weight all training sessions
# not just top_n — let correlation weight do the work
knn_signals_full = []
for i in range(len(train_ids_list)):
    row     = train_corr[i]
    # Only use positive correlations
    pos     = np.maximum(row, 0)
    weights = pos ** 2
    if weights.sum() == 0:
        knn_signals_full.append(0.0)
    else:
        knn_signals_full.append(
            np.average(y_array, weights=weights)
        )

knn_signal_series = pd.Series(
    knn_signals_full,
    index=train_ids_list,
    name='knn_signal'
)

r, p = scipy_stats.pearsonr(knn_signal_series, y_train.loc[train_ids_list])
print(f"Full weighted KNN signal:  r={r:+.4f}  p={p:.4f}")

# Add to feature set and CV
knn_feat    = knn_signal_series.reindex(y_train.index).fillna(0)
X_with_knn  = pd.concat([X_train_raw, knn_feat], axis=1)
feat_names  = FEATURES + ['knn_signal']

scaler_knn  = StandardScaler()
X_scaled_knn = scaler_knn.fit_transform(X_with_knn[feat_names])

for label, X_cv in [
    ("OHLC only",          X_scaled),
    ("OHLC + KNN signal",  X_scaled_knn),
    ("KNN signal only",    X_scaled_knn[:, [-1]]),
]:
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_cv):
        m = Ridge(alpha=100).fit(X_cv[train_idx], y[train_idx])
        p = make_positions(
            m.predict(X_cv[val_idx]),
            sensitivity=0.2
        )
        fold_sharpes.append(compute_sharpe(p, y[val_idx]))
    print(f"  {label:25s}  "
          f"sharpe={np.mean(fold_sharpes):.4f}  "
          f"std={np.std(fold_sharpes):.4f}")

=== Validating KNN signal on training data (LOO) ===

Train-train correlation matrix: (1000, 1000)
Max off-diagonal correlation:   0.645574
Mean off-diagonal correlation:  -0.000245

Top correlation distribution:
  p 50: 0.444652
  p 75: 0.473196
  p 90: 0.514460
  p 95: 0.535112
  p 99: 0.565305
  p100: 0.645574

=== LOO KNN signal correlations ===

  top_n=1  min_corr=0.3  coverage= 996/1000  r=+0.0943  p=0.0029
  top_n=1  min_corr=0.4  coverage= 892/1000  r=+0.1194  p=0.0003
  top_n=1  min_corr=0.5  coverage= 137/1000  r=-0.1294  p=0.1318
  top_n=3  min_corr=0.3  coverage=1000/1000  r=+0.0946  p=0.0028
  top_n=3  min_corr=0.4  coverage= 894/1000  r=+0.1299  p=0.0001
  top_n=3  min_corr=0.5  coverage= 137/1000  r=-0.1330  p=0.1212
  top_n=5  min_corr=0.3  coverage=1000/1000  r=+0.1000  p=0.0015
  top_n=5  min_corr=0.4  coverage= 894/1000  r=+0.1343  p=0.0001
  top_n=5  min_corr=0.5  coverage= 137/1000  r=-0.1330  p=0.1212
  top_n=10  min_corr=0.3  coverage=1000/1000  r=+0.0842  p=0.0

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def build_knn_signal(corr_matrix, source_ids, target_ids,
                     y_source, top_n=5, min_corr=0.4):
    """
    For each target session, find top_n most correlated
    source sessions above min_corr threshold.
    Weight by correlation squared.
    """
    signals = {}
    y_array = y_source.loc[source_ids].values

    for i, target_sid in enumerate(target_ids):
        row   = corr_matrix[i]
        above = np.where(row >= min_corr)[0]

        if len(above) == 0:
            signals[target_sid] = 0.0
            continue

        sorted_idx  = above[np.argsort(row[above])[::-1]]
        top_idx     = sorted_idx[:top_n]
        top_corrs   = row[top_idx]
        top_returns = y_array[top_idx]
        weights     = top_corrs ** 2

        signals[target_sid] = np.average(top_returns, weights=weights)

    return pd.Series(signals, name='knn_signal')

# ── Proper CV: build KNN signal using only training fold data ─────────────────

print("=== CV Sharpe with proper LOO KNN signal ===\n")
print(f"{'label':35s}  {'sharpe':>8}  {'std':>8}")
print("-" * 55)

train_ids_arr = np.array(train_ids_list)
y_array_full  = y_train.loc[train_ids_list].values

for top_n, min_corr in [(1,0.4),(3,0.4),(5,0.4),(10,0.4),(5,0.3)]:
    fold_sharpes_ohlc     = []
    fold_sharpes_knn      = []
    fold_sharpes_combined = []

    for train_idx, val_idx in kf.split(X_scaled):
        train_sids = train_ids_arr[train_idx]
        val_sids   = train_ids_arr[val_idx]

        # Build corr matrix: val × train (within fold)
        T_fold = T_z[train_idx]   # (n_train_fold, L)
        V_fold = T_z[val_idx]     # (n_val_fold,   L)

        # Val-to-train correlations only (no leakage)
        corr_fold = V_fold @ T_fold.T / min_len  # (n_val, n_train)

        # KNN signal for val sessions
        y_train_fold = y_train.loc[train_sids]
        knn_sig      = build_knn_signal(
            corr_fold,
            source_ids=list(train_sids),
            target_ids=list(val_sids),
            y_source=y_train_fold,
            top_n=top_n,
            min_corr=min_corr,
        )

        # Align
        knn_vals = knn_sig.reindex(val_sids).fillna(0).values
        y_val    = y_train.loc[val_sids].values

        # OHLC only
        m_ohlc = Ridge(alpha=100).fit(X_scaled[train_idx], y_array_full[train_idx])
        p_ohlc = make_positions(
            m_ohlc.predict(X_scaled[val_idx]),
            sensitivity=0.2
        )

        # KNN only — position proportional to signal
        p_knn = make_positions(knn_vals, sensitivity=0.2)

        # Combined — stack OHLC + KNN as features
        X_knn_train = np.column_stack([
            X_scaled[train_idx],
            build_knn_signal(
                # Train-train LOO correlations for training fold
                # Zero diagonal already set
                train_corr[np.ix_(train_idx, train_idx)],
                source_ids=list(train_sids),
                target_ids=list(train_sids),
                y_source=y_train.loc[train_sids],
                top_n=top_n,
                min_corr=min_corr,
            ).reindex(train_sids).fillna(0).values.reshape(-1,1)
        ])
        X_knn_val = np.column_stack([X_scaled[val_idx], knn_vals])

        m_comb = Ridge(alpha=100).fit(X_knn_train, y_array_full[train_idx])
        p_comb = make_positions(
            m_comb.predict(X_knn_val),
            sensitivity=0.2
        )

        fold_sharpes_ohlc.append(compute_sharpe(p_ohlc, y_val))
        fold_sharpes_knn.append(compute_sharpe(p_knn,  y_val))
        fold_sharpes_combined.append(compute_sharpe(p_comb, y_val))

    label = f"top_n={top_n} min_corr={min_corr}"
    print(f"\n  Params: {label}")
    print(f"  {'OHLC only':25s}  "
          f"{np.mean(fold_sharpes_ohlc):.4f}  "
          f"{np.std(fold_sharpes_ohlc):.4f}")
    print(f"  {'KNN only':25s}  "
          f"{np.mean(fold_sharpes_knn):.4f}  "
          f"{np.std(fold_sharpes_knn):.4f}")
    print(f"  {'OHLC + KNN':25s}  "
          f"{np.mean(fold_sharpes_combined):.4f}  "
          f"{np.std(fold_sharpes_combined):.4f}")

# ── Apply to test data ────────────────────────────────────────────────────────

print("\n=== Applying best KNN signal to test data ===\n")

# Public test: correlations already computed as pub_corr (n_test=10000, n_train=1000)
# Wait — 10000 test sessions? Recheck
print(f"pub_corr shape:  {pub_corr.shape}")
print(f"priv_corr shape: {priv_corr.shape}")
print(f"pub_ids length:  {len(pub_ids)}")
print(f"priv_ids length: {len(priv_ids)}")

=== CV Sharpe with proper LOO KNN signal ===

label                                  sharpe       std
-------------------------------------------------------

  Params: top_n=1 min_corr=0.4
  OHLC only                  2.8356  0.5296
  KNN only                   3.0298  0.5428
  OHLC + KNN                 2.8363  0.5295

  Params: top_n=3 min_corr=0.4
  OHLC only                  2.8356  0.5296
  KNN only                   2.9961  0.4845
  OHLC + KNN                 2.8360  0.5295

  Params: top_n=5 min_corr=0.4
  OHLC only                  2.8356  0.5296
  KNN only                   2.9984  0.4910
  OHLC + KNN                 2.8361  0.5295

  Params: top_n=10 min_corr=0.4
  OHLC only                  2.8356  0.5296
  KNN only                   2.9993  0.4899
  OHLC + KNN                 2.8361  0.5295

  Params: top_n=5 min_corr=0.3
  OHLC only                  2.8356  0.5296
  KNN only                   2.9690  0.5603
  OHLC + KNN                 2.8357  0.5296

=== Applying best KN

In [ ]:
# ── Rebuild fingerprints correctly ────────────────────────────────────────────

print("=== Test set session counts ===")
print(f"Public  test sessions:  {public_bars['session'].nunique()}")
print(f"Private test sessions:  {private_bars['session'].nunique()}")
print(f"Training sessions:      {seen_train['session'].nunique()}")

# ── Build final submission using KNN + OHLC fallback ─────────────────────────

def build_final_submission(bars_df, corr_matrix, test_ids, 
                            train_ids, y_train,
                            top_n=1, min_corr=0.4,
                            sensitivity_knn=0.2,
                            sensitivity_ohlc=0.2):
    """
    For each test session:
    - If KNN signal available (min_corr match exists): use KNN
    - Otherwise: fall back to OHLC model
    
    KNN signal is stronger (CV 3.03 vs 2.84) so prefer it.
    """
    # ── KNN signal ────────────────────────────────────────────────────────────
    y_array     = y_train.loc[train_ids].values
    knn_signals = {}
    knn_quality = {}

    for i, test_sid in enumerate(test_ids):
        row   = corr_matrix[i]
        above = np.where(row >= min_corr)[0]

        if len(above) == 0:
            knn_signals[test_sid] = None
            continue

        sorted_idx  = above[np.argsort(row[above])[::-1]]
        top_idx     = sorted_idx[:top_n]
        top_corrs   = row[top_idx]
        top_returns = y_array[top_idx]
        weights     = top_corrs ** 2

        knn_signals[test_sid] = np.average(top_returns, weights=weights)
        knn_quality[test_sid] = top_corrs[0]  # best correlation

    n_knn_covered = sum(1 for v in knn_signals.values() if v is not None)
    n_ohlc_needed = sum(1 for v in knn_signals.values() if v is None)

    print(f"Sessions using KNN signal:    {n_knn_covered:5d} "
          f"({n_knn_covered/len(test_ids):.1%})")
    print(f"Sessions using OHLC fallback: {n_ohlc_needed:5d} "
          f"({n_ohlc_needed/len(test_ids):.1%})")

    # ── OHLC signal for fallback sessions ────────────────────────────────────
    ohlc_feats  = build_features(bars_df)[FEATURES]
    X_test      = final_scaler.transform(
                      ohlc_feats.reindex(test_ids).fillna(0)[FEATURES]
                  )
    ohlc_preds  = final_model.predict(X_test)

    # Normalise both signals to same scale for fair combination
    knn_vals  = np.array([
        knn_signals[sid] if knn_signals[sid] is not None else np.nan
        for sid in test_ids
    ])
    ohlc_vals = ohlc_preds

    # Z-score KNN signal (using non-nan values)
    knn_nonnan = knn_vals[~np.isnan(knn_vals)]
    knn_mean   = knn_nonnan.mean()
    knn_std    = knn_nonnan.std() + 1e-8

    # Z-score OHLC signal
    ohlc_mean  = ohlc_vals.mean()
    ohlc_std   = ohlc_vals.std() + 1e-8

    positions = np.zeros(len(test_ids))

    for i, test_sid in enumerate(test_ids):
        if knn_signals[test_sid] is not None:
            # KNN: z-score then make position
            z = (knn_signals[test_sid] - knn_mean) / knn_std
            z = np.clip(z, -2.5, 2.5)
            positions[i] = 1.0 + sensitivity_knn * z
        else:
            # OHLC fallback
            z = (ohlc_vals[i] - ohlc_mean) / ohlc_std
            z = np.clip(z, -2.5, 2.5)
            positions[i] = 1.0 + sensitivity_ohlc * z

        positions[i] = max(positions[i], 0.05)

    return pd.Series(positions, index=test_ids, name='target_position')

# ── Fit final model on all training data ─────────────────────────────────────

final_scaler = StandardScaler()
X_final      = final_scaler.fit_transform(X_train_raw[FEATURES])
final_model  = Ridge(alpha=100).fit(X_final, y)

# ── Generate submissions ──────────────────────────────────────────────────────

print("\n=== Public test submission ===")
public_positions = build_final_submission(
    bars_df      = public_bars,
    corr_matrix  = pub_corr,
    test_ids     = pub_ids,
    train_ids    = train_ids_list,
    y_train      = y_train,
    top_n        = 1,
    min_corr     = 0.4,
    sensitivity_knn  = 0.2,
    sensitivity_ohlc = 0.2,
)

print("\n=== Private test submission ===")
private_positions = build_final_submission(
    bars_df      = private_bars,
    corr_matrix  = priv_corr,
    test_ids     = priv_ids,
    train_ids    = train_ids_list,
    y_train      = y_train,
    top_n        = 1,
    min_corr     = 0.4,
    sensitivity_knn  = 0.2,
    sensitivity_ohlc = 0.2,
)

# ── Write CSV ─────────────────────────────────────────────────────────────────

def write_submission(positions, path):
    sub = (positions
           .reset_index()
           .rename(columns={'index': 'session', 0: 'target_position'}))
    sub.columns = ['session', 'target_position']
    sub = sub.sort_values('session')
    sub.to_csv(path, index=False)
    print(f"\nSaved {len(sub)} rows → {path}")
    print(f"Position stats:")
    print(f"  Mean:  {sub['target_position'].mean():.4f}")
    print(f"  Std:   {sub['target_position'].std():.4f}")
    print(f"  Min:   {sub['target_position'].min():.4f}")
    print(f"  Max:   {sub['target_position'].max():.4f}")
    print(sub.head())
    return sub

write_submission(public_positions,  "submission_public_knn.csv")
write_submission(private_positions, "submission_private_knn.csv")

# ── Sanity check: sensitivity sweep on KNN CV ─────────────────────────────────

print("\n=== Sensitivity sweep for KNN signal ===\n")
for sens in [0.1, 0.2, 0.3, 0.4, 0.5]:
    fold_sharpes = []
    for train_idx, val_idx in kf.split(T_z):
        train_sids = train_ids_arr[train_idx]
        val_sids   = train_ids_arr[val_idx]

        corr_fold = T_z[val_idx] @ T_z[train_idx].T / min_len

        knn_sig = build_knn_signal(
            corr_fold,
            source_ids = list(train_sids),
            target_ids = list(val_sids),
            y_source   = y_train.loc[train_sids],
            top_n      = 1,
            min_corr   = 0.4,
        )

        knn_vals = knn_sig.reindex(val_sids).fillna(0).values
        y_val    = y_train.loc[val_sids].values

        p = make_positions(knn_vals, sensitivity=sens)
        fold_sharpes.append(compute_sharpe(p, y_val))

    print(f"  sensitivity={sens:.1f}  sharpe={np.mean(fold_sharpes):.4f}  "
          f"std={np.std(fold_sharpes):.4f}")

=== Test set session counts ===
Public  test sessions:  10000
Private test sessions:  10000
Training sessions:      1000

=== Public test submission ===
Sessions using KNN signal:     8959 (89.6%)
Sessions using OHLC fallback:  1041 (10.4%)

=== Private test submission ===
Sessions using KNN signal:     8975 (89.8%)
Sessions using OHLC fallback:  1025 (10.2%)

Saved 10000 rows → submission_public_knn.csv
Position stats:
  Mean:  1.0001
  Std:   0.1956
  Min:   0.5000
  Max:   1.5000
   session  target_position
0     1000         0.966750
1     1001         1.177210
2     1002         0.966470
3     1003         0.927925
4     1004         0.888342

Saved 10000 rows → submission_private_knn.csv
Position stats:
  Mean:  0.9992
  Std:   0.1952
  Min:   0.5000
  Max:   1.5000
   session  target_position
0    11000         0.814923
1    11001         1.213343
2    11002         0.875782
3    11003         1.120161
4    11004         1.006627

=== Sensitivity sweep for KNN signal ===

  sens

In [ ]:
# The most likely cause: the 10,000 test sessions vs 1,000 training sessions
# With 10x more test sessions, the correlation structure is different
# Most test sessions have NO good match (coverage at min_corr=0.4?)

# Check actual coverage on test data
above_04_public  = (pub_corr.max(axis=1) >= 0.4).sum()
above_04_private = (priv_corr.max(axis=1) >= 0.4).sum()

print(f"Public  sessions with min_corr>=0.4: {above_04_public}/10000 ({above_04_public/10000:.1%})")
print(f"Private sessions with min_corr>=0.4: {above_04_private}/10000 ({above_04_private/10000:.1%})")

# In training LOO, coverage was 894/1000 = 89%
# If test coverage is much lower, fallback to OHLC dominates
# but the signal mixing may have corrupted positions

# Check: what did positions actually look like?
print(f"\nPublic position distribution:")
print(f"  Mean:  {public_positions.mean():.4f}")
print(f"  Std:   {public_positions.std():.4f}")
print(f"  Min:   {public_positions.min():.4f}")
print(f"  Max:   {public_positions.max():.4f}")
print(f"  % at floor (0.05): {(public_positions <= 0.06).mean():.3f}")

Public  sessions with min_corr>=0.4: 8959/10000 (89.6%)
Private sessions with min_corr>=0.4: 8975/10000 (89.8%)

Public position distribution:
  Mean:  1.0001
  Std:   0.1956
  Min:   0.5000
  Max:   1.5000
  % at floor (0.05): 0.000


In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy import stats as scipy_stats

# ── Step 1: Build full correlation structure ───────────────────────────────────
# Train-train, test-test, and cross correlations

print("=== Building full session correlation structure ===\n")

# Combine train + public test + private test fingerprints
all_fps = {}
all_fps.update({f"train_{k}": v for k, v in train_fps.items()})
all_fps.update({f"pub_{k}":   v for k, v in public_fps.items()})
all_fps.update({f"priv_{k}":  v for k, v in priv_fps.items()})

# Align to common length
min_len = min(len(v) for v in all_fps.values())
all_ids = list(all_fps.keys())

M = np.array([all_fps[sid][:min_len] for sid in all_ids])

# Z-score rows
mu  = M.mean(axis=1, keepdims=True)
std = M.std(axis=1,  keepdims=True) + 1e-8
M_z = (M - mu) / std

print(f"Full session matrix: {M_z.shape}")
print(f"  Train sessions:   {len(train_fps)}")
print(f"  Public sessions:  {len(public_fps)}")
print(f"  Private sessions: {len(priv_fps)}")

# ── Step 2: PCA on price paths ────────────────────────────────────────────────
# Find the principal components of price path variation
# Sessions projecting strongly on the same component 
# share the same underlying dynamic

print("\n=== PCA on price paths ===\n")

pca = PCA(n_components=49)
M_pca = pca.fit_transform(M_z)

print(f"Variance explained by components:")
cumvar = np.cumsum(pca.explained_variance_ratio_)
for n in [1, 2, 3, 5, 10, 20, 49]:
    print(f"  {n:3d} components: {cumvar[n-1]:.3f}")

# Split PCA projections back into train/test
n_train = len(train_fps)
n_pub   = len(public_fps)
n_priv  = len(priv_fps)

train_pca = M_pca[:n_train]
pub_pca   = M_pca[n_train:n_train+n_pub]
priv_pca  = M_pca[n_train+n_pub:]

train_orig_ids = list(train_fps.keys())
pub_orig_ids   = list(public_fps.keys())
priv_orig_ids  = list(priv_fps.keys())

# ── Step 3: Do PCA components correlate with returns? ────────────────────────

print("\n=== PCA component correlations with training returns ===\n")
print(f"{'component':>12}  {'r':>8}  {'p':>8}  {'var_exp':>10}")
print("-" * 45)

sig_components = []
for i in range(49):
    r, p = scipy_stats.pearsonr(train_pca[:, i], y_train.loc[train_orig_ids])
    var  = pca.explained_variance_ratio_[i]
    stars = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else ""
    if p < 0.05:
        sig_components.append(i)
    if p < 0.1:  # show borderline too
        print(f"  PC{i:>3d}:       {r:>+8.4f}  {p:>8.4f}  "
              f"{var:>10.4f}  {stars}")

print(f"\nSignificant components (p<0.05): {sig_components}")

# ── Step 4: Use significant PCA components as signal ─────────────────────────

if sig_components:
    print("\n=== CV Sharpe using PCA price path signal ===\n")

    # Build feature matrix from significant PCA components
    X_pca_train = train_pca[:, sig_components]
    
    scaler_pca  = StandardScaler()
    X_pca_scaled = scaler_pca.fit_transform(X_pca_train)

    for label, X_cv in [
        ("OHLC only",             X_scaled),
        ("PCA price path only",   X_pca_scaled),
        ("OHLC + PCA",            np.hstack([X_scaled, X_pca_scaled])),
    ]:
        fold_sharpes = []
        for train_idx, val_idx in kf.split(X_cv):
            m = Ridge(alpha=100).fit(X_cv[train_idx], y[train_idx])
            p = make_positions(
                m.predict(X_cv[val_idx]),
                sensitivity=0.2
            )
            fold_sharpes.append(compute_sharpe(p, y[val_idx]))
        print(f"  {label:30s}  "
              f"sharpe={np.mean(fold_sharpes):.4f}  "
              f"std={np.std(fold_sharpes):.4f}")

# ── Step 5: Test-test correlation structure ───────────────────────────────────
# Key question: do test sessions cluster into groups?
# If so, sessions in the same cluster should get same-direction positions

print("\n=== Test session clustering via PCA ===\n")

# Public test PCA projections
pub_pca_df  = pd.DataFrame(pub_pca,  index=pub_orig_ids)
priv_pca_df = pd.DataFrame(priv_pca, index=priv_orig_ids)

# How similar are test sessions to each other?
# Compute test-test correlation on significant components only
if sig_components:
    pub_sig  = pub_pca[:,  sig_components]
    priv_sig = priv_pca[:, sig_components]
    train_sig = train_pca[:, sig_components]

    # Normalise
    pub_sig_z   = (pub_sig  - pub_sig.mean(axis=1, keepdims=True)) / (pub_sig.std(axis=1, keepdims=True) + 1e-8)
    priv_sig_z  = (priv_sig - priv_sig.mean(axis=1, keepdims=True)) / (priv_sig.std(axis=1, keepdims=True) + 1e-8)
    train_sig_z = (train_sig - train_sig.mean(axis=1, keepdims=True)) / (train_sig.std(axis=1, keepdims=True) + 1e-8)

    # Test-test similarity
    pub_pub_corr = pub_sig_z @ pub_sig_z.T / len(sig_components)
    np.fill_diagonal(pub_pub_corr, 0)

    print(f"Public test-test correlation (significant PCs only):")
    print(f"  Max:    {pub_pub_corr.max():.4f}")
    print(f"  Mean:   {pub_pub_corr.mean():.4f}")
    print(f"  % pairs > 0.5: {(pub_pub_corr > 0.5).mean():.4f}")
    print(f"  % pairs > 0.8: {(pub_pub_corr > 0.8).mean():.4f}")

# ── Step 6: Cross correlation — do test sessions match train clusters? ─────────

print("\n=== Cross correlation: test vs train (significant PCs) ===\n")

if sig_components:
    # Test-train correlation in PC space
    cross_corr = pub_sig_z @ train_sig_z.T / len(sig_components)
    
    print(f"Public test vs train correlation:")
    print(f"  Max:    {cross_corr.max():.4f}")
    print(f"  Mean:   {cross_corr.mean():.4f}")
    
    best_cross = cross_corr.max(axis=1)
    for pct in [50, 75, 90, 95, 99]:
        print(f"  p{pct:3d}: {np.percentile(best_cross, pct):.4f}")

    # Do test sessions with high cross-correlation to 
    # positive-return training sessions have better returns?
    # We can't validate this directly but can build the signal
    
    train_returns = y_train.loc[train_orig_ids].values
    
    # For each test session: weighted mean of training returns
    # weighted by PC-space similarity
    pos_cross = np.maximum(cross_corr, 0)
    weights   = pos_cross ** 2
    row_sums  = weights.sum(axis=1, keepdims=True) + 1e-8
    weights   = weights / row_sums
    
    pub_pc_signal = weights @ train_returns
    
    print(f"\nPC-space KNN signal distribution:")
    print(f"  Mean:   {pub_pc_signal.mean():+.5f}")
    print(f"  Std:    {pub_pc_signal.std():.5f}")
    print(f"  Min:    {pub_pc_signal.min():+.5f}")
    print(f"  Max:    {pub_pc_signal.max():+.5f}")

    # Validate on training LOO
    print("\n=== LOO validation of PC signal on training data ===\n")
    
    loo_pc_signals = []
    for i in range(len(train_orig_ids)):
        row = train_sig_z[i]
        # Correlate with all OTHER training sessions
        others_z = np.delete(train_sig_z, i, axis=0)
        others_r = np.delete(train_returns, i)
        
        sims    = others_z @ row / len(sig_components)
        pos_sim = np.maximum(sims, 0)
        w       = pos_sim ** 2
        
        if w.sum() > 0:
            loo_pc_signals.append(np.average(others_r, weights=w))
        else:
            loo_pc_signals.append(0.0)
    
    loo_pc_signals = np.array(loo_pc_signals)
    r, p = scipy_stats.pearsonr(loo_pc_signals, train_returns)
    print(f"PC LOO signal correlation: r={r:+.4f}  p={p:.4f}")
    
    # CV Sharpe
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_scaled):
        # Build PC signal for val using train only
        tr_sig_z = train_sig_z[train_idx]
        va_sig_z = train_sig_z[val_idx]
        tr_ret   = train_returns[train_idx]
        
        cross    = va_sig_z @ tr_sig_z.T / len(sig_components)
        pos_c    = np.maximum(cross, 0) ** 2
        w        = pos_c / (pos_c.sum(axis=1, keepdims=True) + 1e-8)
        pc_sig   = w @ tr_ret
        
        p = make_positions(pc_sig, sensitivity=0.2)
        fold_sharpes.append(compute_sharpe(p, train_returns[val_idx]))
    
    print(f"PC signal CV Sharpe: {np.mean(fold_sharpes):.4f}  "
          f"std={np.std(fold_sharpes):.4f}")
    print(f"OHLC CV Sharpe:      2.8356")

=== Building full session correlation structure ===

Full session matrix: (21000, 49)
  Train sessions:   1000
  Public sessions:  10000
  Private sessions: 10000

=== PCA on price paths ===

Variance explained by components:
    1 components: 0.027
    2 components: 0.053
    3 components: 0.077
    5 components: 0.123
   10 components: 0.233
   20 components: 0.445
   49 components: 1.000

=== PCA component correlations with training returns ===

   component         r         p     var_exp
---------------------------------------------
  PC  0:        +0.0541    0.0871      0.0267  
  PC  2:        -0.0586    0.0639      0.0241  
  PC  7:        +0.0609    0.0541      0.0219  
  PC 18:        -0.0580    0.0668      0.0209  
  PC 33:        +0.0595    0.0600      0.0199  
  PC 35:        -0.0527    0.0961      0.0198  
  PC 37:        -0.0806    0.0108      0.0197  *

Significant components (p<0.05): [37]

=== CV Sharpe using PCA price path signal ===

  OHLC only                     

/tmp/ipykernel_953/1924898524.py:198: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = scipy_stats.pearsonr(loo_pc_signals, train_returns)


In [ ]:
# ── What is PC37? ─────────────────────────────────────────────────────────────

print("=== PC37 analysis ===\n")

pc37_loadings = pca.components_[37]
print(f"PC37 loadings across {len(pc37_loadings)} bars:")
print(f"  Shape: {pc37_loadings.shape}")
print(f"  Mean loading: {pc37_loadings.mean():+.4f}")
print(f"  Std loading:  {pc37_loadings.std():.4f}")
print(f"  Max loading:  {pc37_loadings.max():+.4f} at bar {pc37_loadings.argmax()}")
print(f"  Min loading:  {pc37_loadings.min():+.4f} at bar {pc37_loadings.argmin()}")

# Plot loading pattern — which bars drive PC37?
print(f"\nPC37 loading by bar position:")
print(f"{'bar':>6}  {'loading':>10}  {'bar':>6}  {'loading':>10}")
print("-" * 40)
half = len(pc37_loadings) // 2
for i in range(half):
    j = i + half
    print(f"  {i:>4}  {pc37_loadings[i]:>+10.4f}  "
          f"  {j:>4}  {pc37_loadings[j]:>+10.4f}")

# ── PC37 projection as a direct feature ───────────────────────────────────────

print("\n=== PC37 as standalone feature ===\n")

# Training projection
train_pc37 = train_pca[:, 37]  # already computed

r, p = scipy_stats.pearsonr(train_pc37, y_train.loc[train_orig_ids])
print(f"PC37 correlation with return: r={r:+.4f}  p={p:.4f}")

# Quintile analysis
pc37_series = pd.Series(train_pc37, index=train_orig_ids)
df_q = pd.DataFrame({
    'pc37':   pc37_series,
    'return': y_train.loc[train_orig_ids],
})
df_q['quintile'] = pd.qcut(df_q['pc37'], 5, labels=False)

print(f"\nQuintile Sharpe by PC37:")
print(f"{'Q':>4}  {'mean_ret':>10}  {'sharpe':>8}  {'n':>6}")
print("-" * 34)
for q, grp in df_q.groupby('quintile'):
    sh = sharpe(grp['return'])
    print(f"  Q{q+1}  {grp['return'].mean():>+10.5f}  {sh:>8.4f}  {len(grp):>6}")

# ── Apply PC37 to test sessions ───────────────────────────────────────────────

print("\n=== Applying PC37 to test sessions ===\n")

pub_pc37  = pub_pca[:,  37]
priv_pc37 = priv_pca[:, 37]

pub_pc37_series  = pd.Series(pub_pc37,  index=pub_orig_ids)
priv_pc37_series = pd.Series(priv_pc37, index=priv_orig_ids)

print(f"Train  PC37: mean={train_pc37.mean():+.4f}  std={train_pc37.std():.4f}")
print(f"Public PC37: mean={pub_pc37.mean():+.4f}   std={pub_pc37.std():.4f}")
print(f"Privat PC37: mean={priv_pc37.mean():+.4f}  std={priv_pc37.std():.4f}")

# Check distributions match — if very different, signal won't transfer
ks_stat, ks_p = scipy_stats.ks_2samp(train_pc37, pub_pc37)
print(f"\nKS test train vs public PC37: stat={ks_stat:.4f}  p={ks_p:.4f}")
ks_stat, ks_p = scipy_stats.ks_2samp(train_pc37, priv_pc37)
print(f"KS test train vs private PC37: stat={ks_stat:.4f}  p={ks_p:.4f}")

# ── CV Sharpe sweep: sensitivity for PC37 only ────────────────────────────────

print("\n=== Sensitivity sweep for PC37 signal ===\n")

# PC37 is negatively correlated (r=-0.08) so we need to flip sign
pc37_signal = -train_pc37  # flip so positive = good

scaler_pc37   = StandardScaler()
X_pc37_scaled = scaler_pc37.fit_transform(pc37_signal.reshape(-1, 1))

best_sharpe, best_sens = -np.inf, 0.2

for sens in [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]:
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_pc37_scaled):
        m = Ridge(alpha=100).fit(
            X_pc37_scaled[train_idx], y[train_idx]
        )
        p = make_positions(
            m.predict(X_pc37_scaled[val_idx]),
            sensitivity=sens
        )
        fold_sharpes.append(compute_sharpe(p, y[val_idx]))
    mean_sh = np.mean(fold_sharpes)
    print(f"  sens={sens:.2f}  sharpe={mean_sh:.4f}  "
          f"std={np.std(fold_sharpes):.4f}")
    if mean_sh > best_sharpe:
        best_sharpe = mean_sh
        best_sens   = sens

print(f"\nBest: sens={best_sens}  sharpe={best_sharpe:.4f}")

# ── OHLC + PC37 combined submission ───────────────────────────────────────────

print("\n=== Final CV: OHLC + PC37 ===\n")

# Combine OHLC features with PC37
X_combined = np.hstack([
    X_scaled,
    X_pc37_scaled,
])

for label, X_cv, sens in [
    ("OHLC only",       X_scaled,   0.2),
    ("PC37 only",       X_pc37_scaled, best_sens),
    ("OHLC + PC37",     X_combined, 0.2),
]:
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_cv):
        m = Ridge(alpha=100).fit(X_cv[train_idx], y[train_idx])
        p = make_positions(m.predict(X_cv[val_idx]), sensitivity=sens)
        fold_sharpes.append(compute_sharpe(p, y[val_idx]))
    print(f"  {label:20s}  sharpe={np.mean(fold_sharpes):.4f}  "
          f"std={np.std(fold_sharpes):.4f}")

# ── Generate submission using PC37 ────────────────────────────────────────────

print("\n=== Generating PC37-based submission ===\n")

def make_pc37_submission(bars_path, pc37_values, session_ids,
                          output_path, sensitivity=0.2):
    """
    Position based purely on PC37 projection of price path.
    PC37 is derived from the global PCA — no label leakage.
    """
    # Flip sign (negatively correlated with returns)
    signal    = -pc37_values
    z         = (signal - signal.mean()) / (signal.std() + 1e-8)
    z         = np.clip(z, -2.5, 2.5)
    positions = 1.0 + sensitivity * z
    positions = np.maximum(positions, 0.05)

    sub = pd.DataFrame({
        'session':         session_ids,
        'target_position': positions,
    }).sort_values('session')

    sub.to_csv(output_path, index=False)
    print(f"Saved {len(sub)} rows → {output_path}")
    print(f"  Mean position: {positions.mean():.4f}")
    print(f"  Std position:  {positions.std():.4f}")
    return sub

make_pc37_submission(
    bars_path   = os.path.join(DATA_DIR, "bars_seen_public_test.parquet"),
    pc37_values = pub_pc37,
    session_ids = pub_orig_ids,
    output_path = "submission_public_pc37.csv",
    sensitivity = best_sens,
)

make_pc37_submission(
    bars_path    = os.path.join(DATA_DIR, "bars_seen_private_test.parquet"),
    pc37_values  = priv_pc37,
    session_ids  = priv_orig_ids,
    output_path  = "submission_private_pc37.csv",
    sensitivity  = best_sens,
)

=== PC37 analysis ===

PC37 loadings across 49 bars:
  Shape: (49,)
  Mean loading: +0.0000
  Std loading:  0.1429
  Max loading:  +0.3261 at bar 14
  Min loading:  -0.2389 at bar 48

PC37 loading by bar position:
   bar     loading     bar     loading
----------------------------------------
     0     +0.0464      24     +0.0676
     1     -0.1407      25     +0.1344
     2     -0.1672      26     -0.0745
     3     +0.0271      27     +0.0946
     4     +0.0839      28     +0.0665
     5     +0.2179      29     -0.1887
     6     -0.2027      30     +0.0821
     7     +0.1330      31     +0.0881
     8     -0.0179      32     -0.1682
     9     -0.0502      33     +0.0186
    10     -0.1643      34     +0.1075
    11     -0.1850      35     -0.1767
    12     +0.0724      36     -0.1196
    13     +0.3224      37     +0.2093
    14     +0.3261      38     +0.0941
    15     -0.0104      39     +0.0851
    16     -0.1611      40     +0.1552
    17     -0.1882      41     -0.1078
    

,session,target_position
0,11000,1.227624
1,11001,0.630647
2,11002,1.104953
3,11003,1.299468
4,11004,0.927250
...,...,...
9995,20995,1.707642
9996,20996,1.034715
9997,20997,0.847956
9998,20998,1.637704


In [ ]:
print("=== Correct PCA: fit on TRAIN only, project test ===\n")

# Fit PCA on training sessions only
pca_train_only = PCA(n_components=49)
train_pca_clean = pca_train_only.fit_transform(M_z[:len(train_fps)])

# Project test sessions using TRAINING PCA
pub_pca_clean  = pca_train_only.transform(M_z[len(train_fps):len(train_fps)+len(public_fps)])
priv_pca_clean = pca_train_only.transform(M_z[len(train_fps)+len(public_fps):])

print(f"Train PCA shape:   {train_pca_clean.shape}")
print(f"Public PCA shape:  {pub_pca_clean.shape}")
print(f"Private PCA shape: {priv_pca_clean.shape}")

# Recheck correlations with training returns
print("\n=== Component correlations (train-only PCA) ===\n")
print(f"{'component':>12}  {'r':>8}  {'p':>8}")
print("-" * 35)

sig_clean = []
all_corrs = []
for i in range(49):
    r, p = scipy_stats.pearsonr(train_pca_clean[:, i], y_array)
    all_corrs.append((abs(r), i, r, p))
    if p < 0.10:
        stars = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "~"
        print(f"  PC{i:>3d}:       {r:>+8.4f}  {p:>8.4f}  {stars}")
        if p < 0.05:
            sig_clean.append(i)

# Top 10 by |r| with train-only PCA
top10_clean = [i for _, i, _, _ in sorted(all_corrs, reverse=True)[:10]]
print(f"\nTop 10 by |r| (train-only PCA): {sorted(top10_clean)}")
print(f"Significant (p<0.05):           {sig_clean}")

# ── CV comparison: leaky vs clean PCA ────────────────────────────────────────

print("\n=== CV Sharpe: leaky PCA vs clean PCA ===\n")
print(f"{'method':35s}  {'sharpe':>8}  {'std':>8}")
print("-" * 55)

for label, components, pca_data in [
    ("Leaky PCA top10",    
     np.argsort([abs(scipy_stats.pearsonr(train_pca[:,i], y_array)[0]) 
                 for i in range(49)])[-10:].tolist(),
     train_pca),
    ("Clean PCA top10",    
     top10_clean,          
     train_pca_clean),
    ("Clean PCA p<0.05",   
     sig_clean if sig_clean else [0],
     train_pca_clean),
    ("Clean all 49",       
     list(range(49)),      
     train_pca_clean),
]:
    if not components:
        continue
    
    X_sub = StandardScaler().fit_transform(pca_data[:, components])
    
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_sub):
        m = Ridge(alpha=100).fit(X_sub[train_idx], y_array[train_idx])
        p = make_positions(
            m.predict(X_sub[val_idx]),
            sensitivity=0.2
        )
        fold_sharpes.append(compute_sharpe(p, y_array[val_idx]))
    
    mean_sh = np.mean(fold_sharpes)
    print(f"  {label:35s}  {mean_sh:.4f}  {np.std(fold_sharpes):.4f}")

# ── If clean PCA still shows signal, generate proper submission ──────────────

print("\n=== Proper submission using train-only PCA ===\n")

# Best clean components
best_clean_components = top10_clean  # or sig_clean

X_clean_pca   = StandardScaler().fit_transform(
    train_pca_clean[:, best_clean_components]
)

# CV to get honest estimate
fold_sharpes_clean = []
for train_idx, val_idx in kf.split(X_clean_pca):
    m = Ridge(alpha=100).fit(X_clean_pca[train_idx], y_array[train_idx])
    p = make_positions(
        m.predict(X_clean_pca[val_idx]),
        sensitivity=0.2
    )
    fold_sharpes_clean.append(compute_sharpe(p, y_array[val_idx]))

print(f"Clean PCA CV Sharpe: {np.mean(fold_sharpes_clean):.4f}  "
      f"std={np.std(fold_sharpes_clean):.4f}")
print(f"Leaky PCA CV Sharpe: 3.1297")
print(f"OHLC baseline:       2.8356")
print(f"\nIf clean PCA >> 2.8356: real signal, submit")
print(f"If clean PCA ≈ 2.8356:  leakage artifact, revert to original")

# Fit and generate only if clean signal beats baseline
if np.mean(fold_sharpes_clean) > 2.88:  # meaningful threshold
    print("\nClean signal is real — generating submission...")
    
    pca_scaler_final = StandardScaler().fit(
        train_pca_clean[:, best_clean_components]
    )
    model_clean = Ridge(alpha=100).fit(
        pca_scaler_final.transform(
            train_pca_clean[:, best_clean_components]
        ),
        y_array
    )

    for split, pca_proj, out_path in [
        ("Public",  pub_pca_clean,  "submission_public_pca_clean.csv"),
        ("Private", priv_pca_clean, "submission_private_pca_clean.csv"),
    ]:
        X_test = pca_scaler_final.transform(pca_proj[:, best_clean_components])
        preds  = model_clean.predict(X_test)
        pos    = make_positions(preds, sensitivity=0.2)
        
        session_ids = pub_orig_ids if split == "Public" else priv_orig_ids
        sub = pd.DataFrame({
            'session':         session_ids,
            'target_position': pos,
        }).sort_values('session')
        sub.to_csv(out_path, index=False)
        print(f"{split}: saved {len(sub)} rows → {out_path}  "
              f"mean_pos={pos.mean():.4f}")
else:
    print("\nClean signal does not beat baseline.")
    print("The 3.13 CV was a leakage artifact.")
    print("Revert to original model (public 2.40).")

=== Correct PCA: fit on TRAIN only, project test ===

Train PCA shape:   (1000, 49)
Public PCA shape:  (10000, 49)
Private PCA shape: (10000, 49)

=== Component correlations (train-only PCA) ===

   component         r         p
-----------------------------------
  PC 11:        +0.0788    0.0127  *
  PC 17:        -0.0572    0.0705  ~
  PC 23:        +0.0662    0.0362  *
  PC 29:        -0.0695    0.0279  *
  PC 36:        -0.0529    0.0946  ~

Top 10 by |r| (train-only PCA): [0, 1, 7, 11, 15, 17, 23, 29, 36, 47]
Significant (p<0.05):           [11, 23, 29]

=== CV Sharpe: leaky PCA vs clean PCA ===

method                                 sharpe       std
-------------------------------------------------------
  Leaky PCA top10                      3.1297  0.4837
  Clean PCA top10                      3.1146  0.5114
  Clean PCA p<0.05                     3.0416  0.5801
  Clean all 49                         2.8322  0.5269

=== Proper submission using train-only PCA ===

Clean PCA CV 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# ── Build confidence metric from OHLC features ────────────────────────────────

def build_confidence_positions(bars_df, scaler, model, 
                                multiplier=5.0):
    """
    Extreme position sizing based on model confidence.
    
    Confidence = |z-score of prediction|
    Position:
      - High confidence bullish:  multiplier  (e.g. 5x)
      - Low confidence:           1.0 (baseline long)
      - High confidence bearish:  1/multiplier (tiny position)
    
    Never short — drift too strong.
    """
    feats = build_features(bars_df)[FEATURES]
    X     = scaler.transform(feats[FEATURES])
    preds = model.predict(X)
    
    # Z-score predictions
    z = (preds - preds.mean()) / (preds.std() + 1e-8)
    
    # Confidence = absolute z-score (how extreme is the prediction?)
    confidence = np.abs(z)
    conf_pct   = confidence / confidence.max()  # 0 to 1
    
    # Exaggerated sizing:
    # z > 0 (bullish): scale UP by confidence
    # z < 0 (bearish): scale DOWN by confidence (but stay positive)
    positions = np.where(
        z >= 0,
        1.0 + (multiplier - 1.0) * conf_pct,   # 1.0 to multiplier
        1.0 - (1.0 - 1.0/multiplier) * conf_pct # 1/mult to 1.0
    )
    
    print(f"Position distribution (multiplier={multiplier}x):")
    print(f"  Mean:    {positions.mean():.4f}")
    print(f"  Std:     {positions.std():.4f}")
    print(f"  Min:     {positions.min():.4f}")
    print(f"  Max:     {positions.max():.4f}")
    print(f"  >2.0:    {(positions > 2.0).mean():.3f} of sessions")
    print(f"  <0.5:    {(positions < 0.5).mean():.3f} of sessions")
    
    return pd.Series(positions, index=feats.index, name='target_position')

# ── CV validation first ───────────────────────────────────────────────────────

print("=== CV Sharpe vs multiplier ===\n")
print(f"{'multiplier':>12}  {'cv_sharpe':>12}  {'cv_std':>10}")
print("-" * 38)

for mult in [1.0, 2.0, 3.0, 5.0, 10.0, 20.0, 50.0, 100.0]:
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_scaled):
        m     = Ridge(alpha=100).fit(X_scaled[train_idx], y[train_idx])
        preds = m.predict(X_scaled[val_idx])
        z     = (preds - preds.mean()) / (preds.std() + 1e-8)
        conf  = np.abs(z) / np.abs(z).max()
        
        pos = np.where(
            z >= 0,
            1.0 + (mult - 1.0) * conf,
            1.0 - (1.0 - 1.0/mult) * conf
        )
        fold_sharpes.append(compute_sharpe(pos, y[val_idx]))
    
    print(f"  {mult:>10.1f}x  "
          f"{np.mean(fold_sharpes):>12.4f}  "
          f"{np.std(fold_sharpes):>10.4f}")

# ── Generate submissions at different multipliers ─────────────────────────────

print("\n=== Generating exaggerated submissions ===\n")

# Fit final model
final_scaler_exp = StandardScaler()
X_final_exp      = final_scaler_exp.fit_transform(X_train_raw[FEATURES])
final_model_exp  = Ridge(alpha=100).fit(X_final_exp, y)

for multiplier in [3.0, 10.0, 50.0]:
    print(f"\n── Multiplier = {multiplier}x ──")
    
    pub_positions = build_confidence_positions(
        public_bars, final_scaler_exp, final_model_exp,
        multiplier=multiplier
    )

    priv_positions = build_confidence_positions(
        private_bars, final_scaler_exp, final_model_exp,
        multiplier=multiplier
    )
    
    sub = pd.DataFrame({
        'session':         pub_positions.index,
        'target_position': pub_positions.values,
    }).sort_values('session')
    
    path = f"submission_public_extreme_{int(multiplier)}x.csv"
    sub.to_csv(path, index=False)

    sub = pd.DataFrame({
        'session':         priv_positions.index,
        'target_position': priv_positions.values,
    }).sort_values('session')
    
    path = f"submission_private_extreme_{int(multiplier)}x.csv"
    sub.to_csv(path, index=False)
    print(f"  Saved → {path}")

=== CV Sharpe vs multiplier ===

  multiplier     cv_sharpe      cv_std
--------------------------------------
         1.0x        2.7838      0.5857
         2.0x        2.7710      0.4926
         3.0x        2.6487      0.4463
         5.0x        2.4338      0.4139
        10.0x        2.1531      0.4104
        20.0x        1.9585      0.4240
        50.0x        1.8214      0.4391
       100.0x        1.7721      0.4455

=== Generating exaggerated submissions ===


── Multiplier = 3.0x ──
Position distribution (multiplier=3.0x):
  Mean:    1.1316
  Std:     0.3551
  Min:     0.5301
  Max:     3.0000
  >2.0:    0.028 of sessions
  <0.5:    0.000 of sessions
Position distribution (multiplier=3.0x):
  Mean:    1.1233
  Std:     0.3340
  Min:     0.5567
  Max:     3.0000
  >2.0:    0.023 of sessions
  <0.5:    0.000 of sessions
  Saved → submission_private_extreme_3x.csv

── Multiplier = 10.0x ──
Position distribution (multiplier=10.0x):
  Mean:    1.7994
  Std:     1.4280
  Min:   

In [ ]:
# ── Headline count per session ────────────────────────────────────────────────

headline_counts_train = (
    headlines_train
    .groupby('session')['headline']
    .count()
    .rename('n_headlines')
)

print("=== Headline count distribution ===\n")
print(headline_counts_train.describe())
print(f"\nValue counts:")
print(headline_counts_train.value_counts().sort_index())

# ── Does headline count correlate with return volatility? ─────────────────────

print("\n=== Headline count vs return ===\n")

df_hl = pd.DataFrame({
    'n_headlines': headline_counts_train,
    'return':      y_train,
}).dropna()

# Correlation with return
r, p = scipy_stats.pearsonr(df_hl['n_headlines'], df_hl['return'])
print(f"n_headlines vs return:        r={r:+.4f}  p={p:.4f}")

# Correlation with |return| (volatility proxy)
r2, p2 = scipy_stats.pearsonr(df_hl['n_headlines'], df_hl['return'].abs())
print(f"n_headlines vs |return|:      r={r2:+.4f}  p={p2:.4f}")

# Correlation with return^2 (variance proxy)
r3, p3 = scipy_stats.pearsonr(df_hl['n_headlines'], df_hl['return']**2)
print(f"n_headlines vs return^2:      r={r3:+.4f}  p={p3:.4f}")

# ── Sharpe by headline count bucket ───────────────────────────────────────────

print("\n=== Sharpe by headline count ===\n")
print(f"{'n_headlines':>14}  {'n_sessions':>12}  "
      f"{'mean_ret':>10}  {'sharpe':>8}")
print("-" * 50)

for n_hl, grp in df_hl.groupby('n_headlines'):
    if len(grp) < 5:
        continue
    sh = sharpe(grp['return'])
    print(f"  {n_hl:>12}  {len(grp):>12}  "
          f"{grp['return'].mean():>+10.5f}  {sh:>8.4f}")

# ── The core test: does fewer headlines = better Sharpe? ──────────────────────

print("\n=== Low vs high headline count ===\n")

median_hl = headline_counts_train.median()
low_hl    = df_hl[df_hl['n_headlines'] <= median_hl]
high_hl   = df_hl[df_hl['n_headlines'] >  median_hl]

print(f"Median headline count: {median_hl:.0f}")
print(f"\nLow  (≤{median_hl:.0f} headlines):  "
      f"n={len(low_hl):4d}  "
      f"sharpe={sharpe(low_hl['return']):.4f}  "
      f"std={low_hl['return'].std():.5f}")
print(f"High (> {median_hl:.0f} headlines):  "
      f"n={len(high_hl):4d}  "
      f"sharpe={sharpe(high_hl['return']):.4f}  "
      f"std={high_hl['return'].std():.5f}")

t, p = scipy_stats.ttest_ind(low_hl['return'], high_hl['return'])
print(f"\nt-test:  t={t:+.4f}  p={p:.4f}")

# ── CV test: size inversely by headline count ──────────────────────────────────

print("\n=== CV Sharpe: inverse headline count sizing ===\n")
print(f"{'method':35s}  {'sharpe':>8}  {'std':>8}")
print("-" * 55)

hl_count_train = headline_counts_train.reindex(
    X_train_raw.index
).fillna(headline_counts_train.median()).values

for label, size_fn in [
    # Baseline — ignore headline count
    ("Uniform (baseline)",
     lambda hl: np.ones(len(hl))),
    
    # Inverse: fewer headlines → bigger position
    ("Inverse count (1/n)",
     lambda hl: 1.0 / hl),
    
    # Inverse sqrt: gentler scaling
    ("Inverse sqrt (1/√n)",
     lambda hl: 1.0 / np.sqrt(hl)),
    
    # Binary: low count gets 1.5x, high gets 0.5x
    ("Binary low/high",
     lambda hl: np.where(hl <= median_hl, 1.5, 0.5)),
    
    # Normalised inverse: z-score then apply to position
    ("Normalised inverse",
     lambda hl: 1.0 + 0.3 * (-(hl - hl.mean()) / (hl.std() + 1e-8))),
]:
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_scaled):
        # OHLC model prediction
        m     = Ridge(alpha=100).fit(X_scaled[train_idx], y[train_idx])
        preds = m.predict(X_scaled[val_idx])
        z     = (preds - preds.mean()) / (preds.std() + 1e-8)
        z     = np.clip(z, -2.5, 2.5)
        
        # Base position from OHLC signal
        base_pos = 1.0 + 0.2 * z
        
        # Scale by headline count sizing
        hl_val   = hl_count_train[val_idx].astype(float)
        hl_scale = size_fn(hl_val)
        
        # Normalise hl_scale to mean=1 (don't change overall exposure)
        hl_scale = hl_scale / hl_scale.mean()
        
        positions = base_pos * hl_scale
        positions = np.maximum(positions, 0.05)
        
        fold_sharpes.append(compute_sharpe(positions, y[val_idx]))
    
    print(f"  {label:35s}  "
          f"{np.mean(fold_sharpes):.4f}  "
          f"{np.std(fold_sharpes):.4f}")

# ── Also test: headline count as standalone position signal ───────────────────

print("\n=== Headline count as standalone signal ===\n")

for label, signal_fn in [
    ("Raw count",        lambda hl: hl),
    ("Negative count",   lambda hl: -hl),
    ("Inverse count",    lambda hl: 1.0/hl),
]:
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_scaled):
        hl_val  = hl_count_train[val_idx].astype(float)
        signal  = signal_fn(hl_val)
        z       = (signal - signal.mean()) / (signal.std() + 1e-8)
        z       = np.clip(z, -2.5, 2.5)
        pos     = 1.0 + 0.2 * z
        pos     = np.maximum(pos, 0.05)
        fold_sharpes.append(compute_sharpe(pos, y[val_idx]))
    
    print(f"  {label:25s}  "
          f"{np.mean(fold_sharpes):.4f}  "
          f"{np.std(fold_sharpes):.4f}")

=== Headline count distribution ===

count    1000.000000
mean        9.740000
std         2.999566
min         2.000000
25%         8.000000
50%        10.000000
75%        12.000000
max        21.000000
Name: n_headlines, dtype: float64

Value counts:
n_headlines
2       4
3       7
4      16
5      53
6      70
7      83
8     115
9     122
10    132
11    121
12    103
13     82
14     40
15     17
16     17
17      8
18      6
19      2
20      1
21      1
Name: count, dtype: int64

=== Headline count vs return ===

n_headlines vs return:        r=-0.0110  p=0.7294
n_headlines vs |return|:      r=-0.0208  p=0.5102
n_headlines vs return^2:      r=-0.0144  p=0.6489

=== Sharpe by headline count ===

   n_headlines    n_sessions    mean_ret    sharpe
--------------------------------------------------
             3             7    -0.00959   -6.6844
             4            16    +0.00745    5.1779
             5            53    -0.00031   -0.2767
             6            70    +

In [ ]:
print("=== Proper bootstrap test of Sharpe difference ===\n")

low_returns  = df_hl[df_hl['n_headlines'] <= 10]['return'].values
high_returns = df_hl[df_hl['n_headlines'] >  10]['return'].values

# Bootstrap Sharpe difference
n_bootstrap = 10000
sharpe_diffs = []

for _ in range(n_bootstrap):
    low_sample  = np.random.choice(low_returns,  len(low_returns),  replace=True)
    high_sample = np.random.choice(high_returns, len(high_returns), replace=True)
    diff = sharpe(high_sample) - sharpe(low_sample)
    sharpe_diffs.append(diff)

sharpe_diffs = np.array(sharpe_diffs)
observed_diff = sharpe(high_returns) - sharpe(low_returns)

p_bootstrap = np.mean(sharpe_diffs <= 0)  # P(high <= low)

print(f"Observed Sharpe difference: {observed_diff:+.4f}")
print(f"  Low  (≤10 hl): {sharpe(low_returns):.4f}")
print(f"  High (>10 hl): {sharpe(high_returns):.4f}")
print(f"\nBootstrap p-value (high > low): {p_bootstrap:.4f}")
print(f"Bootstrap 95% CI: [{np.percentile(sharpe_diffs,2.5):.4f}, "
      f"{np.percentile(sharpe_diffs,97.5):.4f}]")

# ── Finer granularity — Sharpe by headline count bucket ───────────────────────

print("\n=== Sharpe by headline count (bucketed) ===\n")
print(f"{'bucket':>20}  {'n':>6}  {'sharpe':>8}  {'std_ret':>10}")
print("-" * 50)

buckets = [(2,6), (7,9), (10,11), (12,13), (14,21)]
bucket_sharpes = {}

for lo, hi in buckets:
    grp = df_hl[
        (df_hl['n_headlines'] >= lo) & 
        (df_hl['n_headlines'] <= hi)
    ]
    if len(grp) < 10:
        continue
    sh  = sharpe(grp['return'])
    std = grp['return'].std()
    label = f"{lo}–{hi} headlines"
    bucket_sharpes[label] = (sh, len(grp), grp['return'].values)
    print(f"  {label:>20}  {len(grp):>6}  {sh:>8.4f}  {std:>10.5f}")

# ── CV test: SIZE UP on high headline count sessions ──────────────────────────

print("\n=== CV Sharpe: size UP on high headline sessions ===\n")
print(f"{'method':40s}  {'sharpe':>8}  {'std':>8}")
print("-" * 60)

hl_count = hl_count_train.astype(float)

for label, size_fn in [
    # Baseline
    ("Uniform baseline",
     lambda hl: np.ones(len(hl))),
    
    # Size up on high headline count
    ("Raw count (proportional)",
     lambda hl: hl / hl.mean()),
    
    # Sqrt scaling (gentler)
    ("Sqrt count",
     lambda hl: np.sqrt(hl) / np.sqrt(hl).mean()),
    
    # Binary: >10 gets 1.5x, ≤10 gets 0.5x
    ("Binary high=1.5x low=0.5x",
     lambda hl: np.where(hl > 10, 1.5, 0.5)),
    
    # Binary gentler: >10 gets 1.3x, ≤10 gets 0.7x
    ("Binary high=1.3x low=0.7x",
     lambda hl: np.where(hl > 10, 1.3, 0.7)),

    # Normalised z-score of count
    ("Z-score count sens=0.2",
     lambda hl: 1.0 + 0.2 * ((hl - hl.mean()) / (hl.std() + 1e-8))),

    ("Z-score count sens=0.3",
     lambda hl: 1.0 + 0.3 * ((hl - hl.mean()) / (hl.std() + 1e-8))),

    ("Z-score count sens=0.5",
     lambda hl: 1.0 + 0.5 * ((hl - hl.mean()) / (hl.std() + 1e-8))),
]:
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_scaled):
        m     = Ridge(alpha=100).fit(X_scaled[train_idx], y[train_idx])
        preds = m.predict(X_scaled[val_idx])
        z     = (preds - preds.mean()) / (preds.std() + 1e-8)
        z     = np.clip(z, -2.5, 2.5)

        # Base OHLC position
        base_pos = 1.0 + 0.2 * z

        # Scale by headline count
        hl_val   = hl_count[val_idx]
        hl_scale = size_fn(hl_val)
        hl_scale = np.maximum(hl_scale, 0.05)

        positions = base_pos * hl_scale
        positions = np.maximum(positions, 0.05)

        fold_sharpes.append(compute_sharpe(positions, y[val_idx]))

    print(f"  {label:40s}  "
          f"{np.mean(fold_sharpes):.4f}  "
          f"{np.std(fold_sharpes):.4f}")

# ── Also test headline count ALONE as position ────────────────────────────────

print("\n=== Headline count alone (no OHLC) ===\n")

for label, size_fn in [
    ("Raw count / mean",
     lambda hl: hl / hl.mean()),
    ("Binary >10 = 1.5x",
     lambda hl: np.where(hl > 10, 1.5, 0.5)),
    ("Z-score sens=0.2",
     lambda hl: np.maximum(
         1.0 + 0.2 * ((hl-hl.mean())/(hl.std()+1e-8)), 0.05
     )),
]:
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_scaled):
        hl_val    = hl_count[val_idx]
        positions = size_fn(hl_val)
        fold_sharpes.append(compute_sharpe(positions, y[val_idx]))

    print(f"  {label:35s}  "
          f"{np.mean(fold_sharpes):.4f}  "
          f"{np.std(fold_sharpes):.4f}")

=== Proper bootstrap test of Sharpe difference ===

Observed Sharpe difference: +0.4654
  Low  (≤10 hl): 2.5841
  High (>10 hl): 3.0495

Bootstrap p-value (high > low): 0.3265
Bootstrap 95% CI: [-1.5599, 2.5545]

=== Sharpe by headline count (bucketed) ===

              bucket       n    sharpe     std_ret
--------------------------------------------------
         2–6 headlines     150    1.2683     0.02050
         7–9 headlines     320    2.7628     0.02088
       10–11 headlines     253    3.2689     0.02049
       12–13 headlines     185    4.9465     0.01995
       14–21 headlines      92   -0.5445     0.01899

=== CV Sharpe: size UP on high headline sessions ===

method                                      sharpe       std
------------------------------------------------------------
  Uniform baseline                          2.8356  0.5296
  Raw count (proportional)                  2.6377  0.4167
  Sqrt count                                2.7804  0.4680
  Binary high=1.5x lo

In [ ]:
print("=== CV Sharpe: inverted-U headline count sizing ===\n")
print(f"{'method':45s}  {'sharpe':>8}  {'std':>8}")
print("-" * 65)

hl_count = hl_count_train.astype(float)

def inverted_u_weight(hl, peak=12, width=3, scale=0.5):
    """
    Gaussian-shaped weight centered on peak.
    Sessions near peak get maximum sizing.
    Sessions far from peak get minimum sizing.
    
    weight = 1 + scale * exp(-(hl - peak)^2 / (2 * width^2))
    """
    gaussian = np.exp(-((hl - peak) ** 2) / (2 * width ** 2))
    return 1.0 + scale * gaussian

def inverted_u_binary(hl, lo=10, hi=13, high_mult=1.5, low_mult=0.6):
    """
    Binary version: sessions in sweet spot get high_mult,
    outside get low_mult.
    """
    return np.where((hl >= lo) & (hl <= hi), high_mult, low_mult)

# Grid search over inverted-U parameters
best_sharpe, best_params = -np.inf, {}

for label, size_fn in [
    # Gaussian centered on different peaks
    ("Gaussian peak=10 width=2 scale=0.3",
     lambda hl: inverted_u_weight(hl, 10, 2, 0.3)),
    ("Gaussian peak=11 width=2 scale=0.3",
     lambda hl: inverted_u_weight(hl, 11, 2, 0.3)),
    ("Gaussian peak=12 width=2 scale=0.3",
     lambda hl: inverted_u_weight(hl, 12, 2, 0.3)),
    ("Gaussian peak=12 width=3 scale=0.3",
     lambda hl: inverted_u_weight(hl, 12, 3, 0.3)),
    ("Gaussian peak=12 width=2 scale=0.5",
     lambda hl: inverted_u_weight(hl, 12, 2, 0.5)),
    ("Gaussian peak=12 width=2 scale=1.0",
     lambda hl: inverted_u_weight(hl, 12, 2, 1.0)),
    ("Gaussian peak=11 width=3 scale=0.5",
     lambda hl: inverted_u_weight(hl, 11, 3, 0.5)),
    ("Gaussian peak=11 width=2 scale=0.5",
     lambda hl: inverted_u_weight(hl, 11, 2, 0.5)),

    # Binary sweet spot
    ("Binary 10–13 = 1.5x else 0.7x",
     lambda hl: inverted_u_binary(hl, 10, 13, 1.5, 0.7)),
    ("Binary 10–13 = 2.0x else 0.5x",
     lambda hl: inverted_u_binary(hl, 10, 13, 2.0, 0.5)),
    ("Binary 11–13 = 1.5x else 0.7x",
     lambda hl: inverted_u_binary(hl, 11, 13, 1.5, 0.7)),
    ("Binary 10–12 = 1.5x else 0.7x",
     lambda hl: inverted_u_binary(hl, 10, 12, 1.5, 0.7)),
    ("Binary 10–13 = 1.3x else 0.8x",
     lambda hl: inverted_u_binary(hl, 10, 13, 1.3, 0.8)),

    # Exclude extremes only (penalise <7 and >13)
    ("Penalise extremes <7 and >13",
     lambda hl: np.where(hl < 7,  0.5,
                np.where(hl > 13, 0.5, 1.0))),
    ("Penalise extremes <8 and >14",
     lambda hl: np.where(hl < 8,  0.6,
                np.where(hl > 14, 0.6, 1.0))),
    ("Penalise only >13",
     lambda hl: np.where(hl > 13, 0.5, 1.0)),
    ("Penalise only <7",
     lambda hl: np.where(hl < 7,  0.5, 1.0)),
    ("Penalise only <7 and >13 (gentle)",
     lambda hl: np.where(hl < 7,  0.7,
                np.where(hl > 13, 0.7, 1.0))),
]:
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_scaled):
        m     = Ridge(alpha=100).fit(X_scaled[train_idx], y[train_idx])
        preds = m.predict(X_scaled[val_idx])
        z     = (preds - preds.mean()) / (preds.std() + 1e-8)
        z     = np.clip(z, -2.5, 2.5)

        base_pos = 1.0 + 0.2 * z

        hl_val   = hl_count[val_idx]
        hl_scale = size_fn(hl_val)
        # Normalise to mean=1 to preserve overall exposure
        hl_scale = hl_scale / hl_scale.mean()

        positions = base_pos * hl_scale
        positions = np.maximum(positions, 0.05)

        fold_sharpes.append(compute_sharpe(positions, y[val_idx]))

    mean_sh = np.mean(fold_sharpes)
    std_sh  = np.std(fold_sharpes)
    marker  = " ←" if mean_sh > 2.8356 else ""
    print(f"  {label:45s}  {mean_sh:.4f}  {std_sh:.4f}{marker}")

    if mean_sh > best_sharpe:
        best_sharpe = mean_sh
        best_params = {'label': label, 'sharpe': mean_sh, 'std': std_sh}

print(f"\nBest:      {best_params['label']}")
print(f"Sharpe:    {best_params['sharpe']:.4f}")
print(f"Baseline:  2.8356")
print(f"Delta:     {best_params['sharpe'] - 2.8356:+.4f}")

# ── If best beats baseline, generate submission ────────────────────────────────

if best_params['sharpe'] > 2.8356 + 0.02:
    print(f"\n=== Generating submission with best inverted-U sizing ===\n")

    # Redefine best size function based on winner
    # (replace with actual winner from grid)
    def best_size_fn(hl):
        return inverted_u_binary(hl, 10, 13, 1.5, 0.7)  # update if different

    pub_hl_counts = (
        public_hl.groupby('session')['headline']
        .count()
        .reindex(pub_orig_ids)
        .fillna(headline_counts_train.median())
        .values.astype(float)
    )
    priv_hl_counts = (
        private_hl.groupby('session')['headline']
        .count()
        .reindex(priv_orig_ids)
        .fillna(headline_counts_train.median())
        .values.astype(float)
    )

    for split, bars_df, hl_counts, session_ids, out_path in [
        ("Public",  public_bars,  pub_hl_counts,
         pub_orig_ids,  "submission_public_invU.csv"),
        ("Private", private_bars, priv_hl_counts,
         priv_orig_ids, "submission_private_invU.csv"),
    ]:
        feats    = build_features(bars_df)[FEATURES]
        X_test   = final_scaler.transform(feats.reindex(session_ids)[FEATURES])
        preds    = final_model.predict(X_test)
        z        = (preds - preds.mean()) / (preds.std() + 1e-8)
        z        = np.clip(z, -2.5, 2.5)

        base_pos = 1.0 + 0.2 * z
        hl_scale = best_size_fn(hl_counts)
        hl_scale = hl_scale / hl_scale.mean()

        positions = np.maximum(base_pos * hl_scale, 0.05)

        sub = pd.DataFrame({
            'session':         session_ids,
            'target_position': positions,
        }).sort_values('session')
        sub.to_csv(out_path, index=False)

        print(f"{split}: {len(sub)} rows → {out_path}")
        print(f"  Mean: {positions.mean():.4f}  "
              f"Std:  {positions.std():.4f}  "
              f"Min:  {positions.min():.4f}  "
              f"Max:  {positions.max():.4f}")
else:
    print(f"\nNo inverted-U variant beats baseline by >0.02.")
    print(f"Submit original: submission_private_FINAL.csv")

=== CV Sharpe: inverted-U headline count sizing ===

method                                           sharpe       std
-----------------------------------------------------------------
  Gaussian peak=10 width=2 scale=0.3             2.8371  0.5365 ←
  Gaussian peak=11 width=2 scale=0.3             2.8646  0.5347 ←
  Gaussian peak=12 width=2 scale=0.3             2.8894  0.5378 ←
  Gaussian peak=12 width=3 scale=0.3             2.8813  0.5436 ←
  Gaussian peak=12 width=2 scale=0.5             2.9071  0.5431 ←
  Gaussian peak=12 width=2 scale=1.0             2.9196  0.5546 ←
  Gaussian peak=11 width=3 scale=0.5             2.8909  0.5561 ←
  Gaussian peak=11 width=2 scale=0.5             2.8703  0.5413 ←
  Binary 10–13 = 1.5x else 0.7x                  2.9936  0.4935 ←
  Binary 10–13 = 2.0x else 0.5x                  2.9162  0.4835 ←
  Binary 11–13 = 1.5x else 0.7x                  2.9119  0.5771 ←
  Binary 10–12 = 1.5x else 0.7x                  2.8513  0.5211 ←
  Binary 10–13 = 1.3x e

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, RepeatedKFold
from scipy import stats as scipy_stats

# Use more folds and repeats for stable std estimate
rkf = RepeatedKFold(n_splits=10, n_repeats=5, random_state=42)

def evaluate_strategy(positions_fn, X, y, cv=rkf):
    """
    Evaluate a positioning strategy.
    Returns mean Sharpe, std Sharpe, and Sharpe of Sharpes.
    """
    fold_sharpes = []
    for train_idx, val_idx in cv.split(X):
        positions = positions_fn(train_idx, val_idx)
        fold_sharpes.append(compute_sharpe(positions, y[val_idx]))
    
    fold_sharpes = np.array(fold_sharpes)
    return {
        'mean':       np.mean(fold_sharpes),
        'std':        np.std(fold_sharpes),
        'sharpe_of_sharpe': np.mean(fold_sharpes) / (np.std(fold_sharpes) + 1e-8),
        'min':        np.min(fold_sharpes),
        'p10':        np.percentile(fold_sharpes, 10),
        'p90':        np.percentile(fold_sharpes, 90),
        'pct_positive': np.mean(np.array(fold_sharpes) > 0),
    }

def print_eval(label, result):
    print(f"  {label:40s}  "
          f"mean={result['mean']:>7.4f}  "
          f"std={result['std']:>7.4f}  "
          f"SoS={result['sharpe_of_sharpe']:>7.4f}  "
          f"p10={result['p10']:>7.4f}")

print("=== Std-minimisation strategy evaluation ===")
print(f"(RepeatedKFold: 10 splits × 5 repeats = 50 folds)\n")
print(f"{'strategy':40s}  {'mean':>7}  {'std':>7}  "
      f"{'SoS':>7}  {'p10':>7}")
print("-" * 75)

# ── Strategy 1: Baseline ──────────────────────────────────────────────────────

def strategy_baseline(train_idx, val_idx):
    m = Ridge(alpha=100).fit(X_scaled[train_idx], y[train_idx])
    return make_positions(m.predict(X_scaled[val_idx]), sensitivity=0.2)

print_eval("Baseline (sens=0.2)",
           evaluate_strategy(strategy_baseline, X_scaled, y))

# ── Strategy 2: Always long — minimum variance ────────────────────────────────

def strategy_always_long(train_idx, val_idx):
    return np.ones(len(val_idx))

print_eval("Always long (sens=0.0)",
           evaluate_strategy(strategy_always_long, X_scaled, y))

# ── Strategy 3: Reduce sensitivity — stay closer to always-long ───────────────

for sens in [0.05, 0.10, 0.15]:
    def make_low_sens(s):
        def fn(train_idx, val_idx):
            m = Ridge(alpha=100).fit(X_scaled[train_idx], y[train_idx])
            return make_positions(m.predict(X_scaled[val_idx]), sensitivity=s)
        return fn
    print_eval(f"Low sensitivity (sens={sens})",
               evaluate_strategy(make_low_sens(sens), X_scaled, y))

# ── Strategy 4: Hard clip positions to narrow band ───────────────────────────

def strategy_clipped(train_idx, val_idx, lo=0.8, hi=1.2):
    m   = Ridge(alpha=100).fit(X_scaled[train_idx], y[train_idx])
    pos = make_positions(m.predict(X_scaled[val_idx]), sensitivity=0.2)
    return np.clip(pos, lo, hi)

for lo, hi in [(0.9, 1.1), (0.8, 1.2), (0.7, 1.3)]:
    def make_clip(l, h):
        def fn(train_idx, val_idx):
            return strategy_clipped(train_idx, val_idx, l, h)
        return fn
    print_eval(f"Clipped [{lo:.1f}, {hi:.1f}]",
               evaluate_strategy(make_clip(lo, hi), X_scaled, y))

# ── Strategy 5: Shrink toward always-long by mixing ──────────────────────────

for mix in [0.1, 0.2, 0.3, 0.5, 0.7]:
    def make_mix(m_weight):
        def fn(train_idx, val_idx):
            mod = Ridge(alpha=100).fit(X_scaled[train_idx], y[train_idx])
            model_pos  = make_positions(
                mod.predict(X_scaled[val_idx]), sensitivity=0.2
            )
            always_long = np.ones(len(val_idx))
            # Blend: mix_weight * model + (1-mix_weight) * always_long
            return m_weight * model_pos + (1 - m_weight) * always_long
        return fn
    print_eval(f"Mix {mix:.1f} model + {1-mix:.1f} always-long",
               evaluate_strategy(make_mix(mix), X_scaled, y))

# ── Strategy 6: Heavy regularisation to shrink predictions ───────────────────

for alpha in [1000, 5000, 10000, 50000]:
    def make_heavy_reg(a):
        def fn(train_idx, val_idx):
            m = Ridge(alpha=a).fit(X_scaled[train_idx], y[train_idx])
            return make_positions(m.predict(X_scaled[val_idx]), sensitivity=0.2)
        return fn
    print_eval(f"Heavy Ridge alpha={alpha}",
               evaluate_strategy(make_heavy_reg(alpha), X_scaled, y))

# ── Strategy 7: Winsorise returns — reduce impact of outlier sessions ─────────

def strategy_winsorised(train_idx, val_idx, pct=10):
    """
    Fit model on winsorised returns to reduce outlier influence.
    """
    y_train_fold = y[train_idx]
    lo = np.percentile(y_train_fold, pct)
    hi = np.percentile(y_train_fold, 100-pct)
    y_winsor = np.clip(y_train_fold, lo, hi)
    
    m   = Ridge(alpha=100).fit(X_scaled[train_idx], y_winsor)
    pos = make_positions(m.predict(X_scaled[val_idx]), sensitivity=0.2)
    return pos

for pct in [5, 10, 20]:
    def make_winsor(p):
        def fn(train_idx, val_idx):
            return strategy_winsorised(train_idx, val_idx, p)
        return fn
    print_eval(f"Winsorised returns (pct={pct})",
               evaluate_strategy(make_winsor(pct), X_scaled, y))

# ── Strategy 8: Rank-based positions (non-parametric) ────────────────────────

def strategy_ranked(train_idx, val_idx, sensitivity=0.2):
    """
    Instead of raw predictions, use rank of prediction.
    Ranks are uniformly distributed — no outlier sensitivity.
    """
    m     = Ridge(alpha=100).fit(X_scaled[train_idx], y[train_idx])
    preds = m.predict(X_scaled[val_idx])
    
    # Convert to ranks, normalise to [-1, +1]
    ranks = scipy_stats.rankdata(preds)
    ranks_norm = (ranks - ranks.mean()) / (ranks.std() + 1e-8)
    
    return np.maximum(1.0 + sensitivity * ranks_norm, 0.05)

for sens in [0.1, 0.2, 0.3]:
    def make_ranked(s):
        def fn(train_idx, val_idx):
            return strategy_ranked(train_idx, val_idx, s)
        return fn
    print_eval(f"Rank-based sens={sens}",
               evaluate_strategy(make_ranked(sens), X_scaled, y))

# ── Strategy 9: Headline count inverted-U × OHLC ─────────────────────────────

def strategy_invU_hl(train_idx, val_idx, peak=12, width=2, scale=0.3):
    m     = Ridge(alpha=100).fit(X_scaled[train_idx], y[train_idx])
    preds = m.predict(X_scaled[val_idx])
    z     = (preds - preds.mean()) / (preds.std() + 1e-8)
    z     = np.clip(z, -2.5, 2.5)

    base_pos = 1.0 + 0.2 * z

    hl_val   = hl_count_train[val_idx].astype(float)
    gaussian = np.exp(-((hl_val - peak)**2) / (2 * width**2))
    hl_scale = 1.0 + scale * gaussian
    hl_scale = hl_scale / hl_scale.mean()

    return np.maximum(base_pos * hl_scale, 0.05)

for peak, width, scale in [
    (12, 2, 0.3), (11, 2, 0.3), (12, 3, 0.3), (12, 2, 0.5)
]:
    def make_invU(p, w, s):
        def fn(train_idx, val_idx):
            return strategy_invU_hl(train_idx, val_idx, p, w, s)
        return fn
    print_eval(f"InvU peak={peak} width={width} scale={scale}",
               evaluate_strategy(make_invU(peak, width, scale), X_scaled, y))

# ── Strategy 10: Combination — mix + rank + clip ─────────────────────────────

def strategy_combined_robust(train_idx, val_idx):
    """
    Stack multiple std-reduction techniques:
    1. Rank-based signal (no outlier sensitivity)
    2. Mix with always-long (shrink toward baseline)
    3. Clip to narrow band (hard limit on variance)
    """
    m     = Ridge(alpha=100).fit(X_scaled[train_idx], y[train_idx])
    preds = m.predict(X_scaled[val_idx])
    
    # Rank-based signal
    ranks      = scipy_stats.rankdata(preds)
    ranks_norm = (ranks - ranks.mean()) / (ranks.std() + 1e-8)
    model_pos  = 1.0 + 0.2 * ranks_norm
    
    # Mix with always-long
    mixed = 0.3 * model_pos + 0.7 * np.ones(len(val_idx))
    
    # Clip
    clipped = np.clip(mixed, 0.8, 1.2)
    
    return clipped

print_eval("Combined robust (rank+mix+clip)",
           evaluate_strategy(strategy_combined_robust, X_scaled, y))

# ── Find best by Sharpe-of-Sharpe (mean/std) ─────────────────────────────────

print("\n=== Summary: optimising for Sharpe-of-Sharpe ===\n")
print("Best strategy = highest mean/std ratio")
print("This maximises consistency across the private leaderboard sessions\n")

# Re-evaluate top candidates with full stats
candidates = {
    "Baseline sens=0.2":          strategy_baseline,
    "Always long":                 strategy_always_long,
    "Rank sens=0.1":               make_ranked(0.1),
    "Mix 0.3 model + 0.7 long":   make_mix(0.3),
    "Clip [0.9, 1.1]":            make_clip(0.9, 1.1),
    "Combined robust":             strategy_combined_robust,
    "Heavy Ridge alpha=10000":     make_heavy_reg(10000),
    "Winsorised pct=10":          make_winsor(10),
}

print(f"{'strategy':40s}  {'mean':>7}  {'std':>7}  "
      f"{'SoS':>7}  {'p10':>7}  {'pct_pos':>7}")
print("-" * 85)

best_sos, best_strategy_label = -np.inf, ""
results = {}

for label, fn in candidates.items():
    r = evaluate_strategy(fn, X_scaled, y)
    results[label] = r
    print(f"  {label:40s}  "
          f"{r['mean']:>7.4f}  "
          f"{r['std']:>7.4f}  "
          f"{r['sharpe_of_sharpe']:>7.4f}  "
          f"{r['p10']:>7.4f}  "
          f"{r['pct_positive']:>7.3f}")
    
    if r['sharpe_of_sharpe'] > best_sos:
        best_sos = r['sharpe_of_sharpe']
        best_strategy_label = label

print(f"\nBest by Sharpe-of-Sharpe: '{best_strategy_label}'  "
      f"SoS={best_sos:.4f}")

=== Std-minimisation strategy evaluation ===
(RepeatedKFold: 10 splits × 5 repeats = 50 folds)

strategy                                     mean      std      SoS      p10
---------------------------------------------------------------------------
  Baseline (sens=0.2)                       mean= 2.8479  std= 1.7067  SoS= 1.6687  p10= 0.3638
  Always long (sens=0.0)                    mean= 2.7963  std= 1.7405  SoS= 1.6066  p10= 0.3595
  Low sensitivity (sens=0.05)               mean= 2.8200  std= 1.7332  SoS= 1.6271  p10= 0.3056
  Low sensitivity (sens=0.1)                mean= 2.8361  std= 1.7249  SoS= 1.6442  p10= 0.2881
  Low sensitivity (sens=0.15)               mean= 2.8452  std= 1.7160  SoS= 1.6580  p10= 0.3271
  Clipped [0.9, 1.1]                        mean= 2.8505  std= 1.7285  SoS= 1.6491  p10= 0.4081
  Clipped [0.8, 1.2]                        mean= 2.8696  std= 1.7240  SoS= 1.6645  p10= 0.4195
  Clipped [0.7, 1.3]                        mean= 2.8624  std= 1.7120  SoS= 1.6

In [ ]:
print("=== Combined: Rank + InvU headline sizing ===\n")
print(f"{'strategy':50s}  {'mean':>7}  {'std':>7}  "
      f"{'SoS':>7}  {'p10':>7}  {'pct_pos':>7}")
print("-" * 95)

def strategy_rank_invU(train_idx, val_idx, 
                        sens=0.3, peak=12, width=2, scale=0.5,
                        winsor_pct=20):
    """
    Three layers of robustness:
    1. Winsorised training (reduce outlier influence on model)
    2. Rank-based signal (uniform distribution, no outliers)
    3. InvU headline scaling (boost sweet spot sessions)
    """
    # Winsorised fit
    y_fold = y[train_idx]
    lo     = np.percentile(y_fold, winsor_pct)
    hi     = np.percentile(y_fold, 100 - winsor_pct)
    y_w    = np.clip(y_fold, lo, hi)
    
    m      = Ridge(alpha=100).fit(X_scaled[train_idx], y_w)
    preds  = m.predict(X_scaled[val_idx])
    
    # Rank-based positions
    ranks      = scipy_stats.rankdata(preds)
    ranks_norm = (ranks - ranks.mean()) / (ranks.std() + 1e-8)
    base_pos   = 1.0 + sens * ranks_norm
    
    # InvU headline scaling
    hl_val   = hl_count_train[val_idx].astype(float)
    gaussian = np.exp(-((hl_val - peak) ** 2) / (2 * width ** 2))
    hl_scale = 1.0 + scale * gaussian
    hl_scale = hl_scale / hl_scale.mean()
    
    positions = base_pos * hl_scale
    return np.maximum(positions, 0.05)

# Grid search over combinations
best_sos, best_cfg = -np.inf, {}

configs = [
    (sens, peak, width, scale, wp)
    for sens  in [0.2, 0.3, 0.4]
    for peak  in [11, 12]
    for width in [2, 3]
    for scale in [0.3, 0.5]
    for wp    in [10, 20]
]

print(f"Testing {len(configs)} combinations...\n")

results = []
for sens, peak, width, scale, wp in configs:
    def make_fn(s, pk, wd, sc, w):
        def fn(ti, vi):
            return strategy_rank_invU(ti, vi, s, pk, wd, sc, w)
        return fn
    
    r = evaluate_strategy(make_fn(sens, peak, width, scale, wp), 
                          X_scaled, y, cv=rkf)
    results.append((r, sens, peak, width, scale, wp))
    
    if r['sharpe_of_sharpe'] > best_sos:
        best_sos = r['sharpe_of_sharpe']
        best_cfg = dict(sens=sens, peak=peak, width=width, 
                        scale=scale, winsor_pct=wp)

# Show top 10 by SoS
results.sort(key=lambda x: x[0]['sharpe_of_sharpe'], reverse=True)

print(f"{'sens':>6} {'peak':>6} {'wid':>5} {'scl':>5} "
      f"{'wp':>4}  {'mean':>7}  {'std':>7}  {'SoS':>7}  {'p10':>7}")
print("-" * 70)
for r, sens, peak, width, scale, wp in results[:15]:
    marker = " ←" if r['sharpe_of_sharpe'] > 1.71 else ""
    print(f"  {sens:>4}  {peak:>4}  {width:>4}  {scale:>4}  {wp:>4}  "
          f"{r['mean']:>7.4f}  {r['std']:>7.4f}  "
          f"{r['sharpe_of_sharpe']:>7.4f}  {r['p10']:>7.4f}{marker}")

print(f"\nBest config: {best_cfg}")
print(f"Best SoS:    {best_sos:.4f}")
print(f"Baseline:    1.6687")

# ── Generate final submissions with best config ────────────────────────────────

print("\n=== Generating robust submissions ===\n")

def build_robust_submission(bars_df, hl_df, session_ids, 
                             cfg, output_path):
    # OHLC features
    feats   = build_features(bars_df)[FEATURES]
    feats   = feats.reindex(session_ids).fillna(0)
    X_test  = final_scaler.transform(feats[FEATURES])
    
    # Winsorised fit on full training data
    lo     = np.percentile(y, cfg['winsor_pct'])
    hi     = np.percentile(y, 100 - cfg['winsor_pct'])
    y_w    = np.clip(y, lo, hi)
    
    model_w = Ridge(alpha=100).fit(X_final, y_w)
    preds   = model_w.predict(X_test)
    
    # Rank-based positions
    ranks      = scipy_stats.rankdata(preds)
    ranks_norm = (ranks - ranks.mean()) / (ranks.std() + 1e-8)
    base_pos   = 1.0 + cfg['sens'] * ranks_norm
    
    # InvU headline scaling
    hl_counts = (
        hl_df.groupby('session')['headline']
        .count()
        .reindex(session_ids)
        .fillna(10.0)
        .values.astype(float)
    )
    gaussian = np.exp(
        -((hl_counts - cfg['peak']) ** 2) / (2 * cfg['width'] ** 2)
    )
    hl_scale = 1.0 + cfg['scale'] * gaussian
    hl_scale = hl_scale / hl_scale.mean()
    
    positions = np.maximum(base_pos * hl_scale, 0.05)
    
    sub = pd.DataFrame({
        'session':         session_ids,
        'target_position': positions,
    }).sort_values('session')
    sub.to_csv(output_path, index=False)
    
    print(f"Saved {len(sub)} rows → {output_path}")
    print(f"  Mean: {positions.mean():.4f}  "
          f"Std:  {positions.std():.4f}  "
          f"Min:  {positions.min():.4f}  "
          f"Max:  {positions.max():.4f}")
    return sub

# Load test headlines
public_hl  = pd.read_parquet(
    os.path.join(DATA_DIR, "headlines_seen_public_test.parquet"),
    engine="fastparquet"
)
private_hl = pd.read_parquet(
    os.path.join(DATA_DIR, "headlines_seen_private_test.parquet"),
    engine="fastparquet"
)

print("Public test:")
build_robust_submission(
    public_bars, public_hl, pub_orig_ids,
    best_cfg, "submission_public_robust.csv"
)

print("\nPrivate test:")
build_robust_submission(
    private_bars, private_hl, priv_orig_ids,
    best_cfg, "submission_private_robust.csv"
)

print(f"\n=== Final comparison ===")
print(f"Original model:    public 2.40063")
print(f"This model:        submit public first to validate")
print(f"Best CV SoS:       {best_sos:.4f} vs baseline 1.6687")

=== Combined: Rank + InvU headline sizing ===

strategy                                               mean      std      SoS      p10  pct_pos
-----------------------------------------------------------------------------------------------
Testing 48 combinations...

  sens   peak   wid   scl   wp     mean      std      SoS      p10
----------------------------------------------------------------------
   0.4    12     2   0.5    20   2.9683   1.6328   1.8179   1.0203 ←
   0.4    12     2   0.5    10   2.9507   1.6300   1.8103   1.0067 ←
   0.4    12     2   0.3    20   2.9520   1.6434   1.7963   0.9269 ←
   0.4    12     3   0.5    20   2.9595   1.6487   1.7950   1.0081 ←
   0.3    12     2   0.5    20   2.9748   1.6611   1.7909   0.8955 ←
   0.4    12     2   0.3    10   2.9347   1.6406   1.7888   0.9119 ←
   0.4    12     3   0.5    10   2.9423   1.6459   1.7877   0.9909 ←
   0.3    12     2   0.5    10   2.9611   1.6589   1.7850   0.8892 ←
   0.4    12     3   0.3    20   2.9446   1

In [ ]:
from sklearn.linear_model import Lasso, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, RepeatedKFold

kf  = KFold(n_splits=5, shuffle=True, random_state=42)
rkf = RepeatedKFold(n_splits=10, n_repeats=5, random_state=42)

print("=== Lasso feature selection ===\n")

# ── Find optimal alpha via LassoCV ────────────────────────────────────────────

lasso_cv = LassoCV(
    alphas=np.logspace(-6, 0, 100),
    cv=kf,
    max_iter=10000,
    random_state=42,
)
lasso_cv.fit(X_scaled, y)

print(f"Optimal Lasso alpha: {lasso_cv.alpha_:.6f}")
print(f"\nFeature coefficients at optimal alpha:")
print(f"{'feature':20s}  {'coef':>10}  {'selected':>10}")
print("-" * 45)

for feat, coef in zip(FEATURES, lasso_cv.coef_):
    selected = "YES" if abs(coef) > 1e-6 else "zeroed out"
    print(f"  {feat:20s}  {coef:>+10.6f}  {selected:>10}")

n_selected = np.sum(np.abs(lasso_cv.coef_) > 1e-6)
print(f"\nFeatures selected: {n_selected}/{len(FEATURES)}")

# ── Compare Lasso vs Ridge across alpha range ─────────────────────────────────

print("\n=== Lasso vs Ridge: CV Sharpe across alpha ===\n")
print(f"{'model':30s}  {'mean':>8}  {'std':>8}  {'SoS':>8}  {'p10':>8}")
print("-" * 65)

# Ridge baseline
r = evaluate_strategy(strategy_baseline, X_scaled, y, cv=rkf)
print(f"  {'Ridge alpha=100':30s}  "
      f"{r['mean']:>8.4f}  {r['std']:>8.4f}  "
      f"{r['sharpe_of_sharpe']:>8.4f}  {r['p10']:>8.4f}")

# Lasso across alphas
best_sos_lasso, best_alpha_lasso = -np.inf, 0

for alpha in np.logspace(-6, 0, 20):
    def make_lasso(a):
        def fn(train_idx, val_idx):
            m = Lasso(alpha=a, max_iter=10000).fit(
                X_scaled[train_idx], y[train_idx]
            )
            preds = m.predict(X_scaled[val_idx])
            return make_positions(preds, sensitivity=0.2)
        return fn
    
    r = evaluate_strategy(make_lasso(alpha), X_scaled, y, cv=rkf)
    marker = " ←" if r['sharpe_of_sharpe'] > 1.6687 else ""
    print(f"  {'Lasso alpha='+f'{alpha:.6f}':30s}  "
          f"{r['mean']:>8.4f}  {r['std']:>8.4f}  "
          f"{r['sharpe_of_sharpe']:>8.4f}  {r['p10']:>8.4f}{marker}")
    
    if r['sharpe_of_sharpe'] > best_sos_lasso:
        best_sos_lasso  = r['sharpe_of_sharpe']
        best_alpha_lasso = alpha

print(f"\nBest Lasso SoS: {best_sos_lasso:.4f} at alpha={best_alpha_lasso:.6f}")
print(f"Ridge baseline: 1.6687")

# ── Lasso with rank-based positions ──────────────────────────────────────────

print("\n=== Lasso + rank-based positions ===\n")

for alpha in [lasso_cv.alpha_, best_alpha_lasso]:
    for sens in [0.2, 0.3]:
        def make_lasso_rank(a, s):
            def fn(train_idx, val_idx):
                m = Lasso(alpha=a, max_iter=10000).fit(
                    X_scaled[train_idx], y[train_idx]
                )
                preds      = m.predict(X_scaled[val_idx])
                ranks      = scipy_stats.rankdata(preds)
                ranks_norm = (ranks - ranks.mean()) / (ranks.std() + 1e-8)
                return np.maximum(1.0 + s * ranks_norm, 0.05)
            return fn
        
        r = evaluate_strategy(
            make_lasso_rank(alpha, sens), X_scaled, y, cv=rkf
        )
        label = f"Lasso rank a={alpha:.4f} s={sens}"
        marker = " ←" if r['sharpe_of_sharpe'] > 1.6687 else ""
        print(f"  {label:40s}  "
              f"{r['mean']:>7.4f}  {r['std']:>7.4f}  "
              f"{r['sharpe_of_sharpe']:>7.4f}  {r['p10']:>7.4f}{marker}")

# ── Lasso + winsorised training ───────────────────────────────────────────────

print("\n=== Lasso + winsorised training ===\n")

for alpha in [lasso_cv.alpha_, best_alpha_lasso]:
    for wp in [10, 20]:
        def make_lasso_winsor(a, w):
            def fn(train_idx, val_idx):
                y_fold = y[train_idx]
                lo     = np.percentile(y_fold, w)
                hi     = np.percentile(y_fold, 100 - w)
                y_w    = np.clip(y_fold, lo, hi)
                
                m = Lasso(alpha=a, max_iter=10000).fit(
                    X_scaled[train_idx], y_w
                )
                preds = m.predict(X_scaled[val_idx])
                return make_positions(preds, sensitivity=0.2)
            return fn
        
        r = evaluate_strategy(
            make_lasso_winsor(alpha, wp), X_scaled, y, cv=rkf
        )
        label = f"Lasso winsor a={alpha:.4f} pct={wp}"
        marker = " ←" if r['sharpe_of_sharpe'] > 1.6687 else ""
        print(f"  {label:40s}  "
              f"{r['mean']:>7.4f}  {r['std']:>7.4f}  "
              f"{r['sharpe_of_sharpe']:>7.4f}  {r['p10']:>7.4f}{marker}")

# ── ElasticNet: best of both Ridge and Lasso ─────────────────────────────────

print("\n=== ElasticNet (Ridge + Lasso combined) ===\n")

from sklearn.linear_model import ElasticNet, ElasticNetCV

# Find optimal ElasticNet params
en_cv = ElasticNetCV(
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
    alphas=np.logspace(-6, 0, 50),
    cv=kf,
    max_iter=10000,
    random_state=42,
)
en_cv.fit(X_scaled, y)

print(f"Optimal ElasticNet alpha:    {en_cv.alpha_:.6f}")
print(f"Optimal ElasticNet l1_ratio: {en_cv.l1_ratio_:.2f}")
print(f"\nFeature coefficients:")
for feat, coef in zip(FEATURES, en_cv.coef_):
    selected = "YES" if abs(coef) > 1e-6 else "zeroed"
    print(f"  {feat:20s}  {coef:>+10.6f}  {selected}")

# Evaluate ElasticNet
for l1 in [0.1, 0.3, 0.5, 0.7, 0.9]:
    def make_enet(l):
        def fn(train_idx, val_idx):
            m = ElasticNet(
                alpha=en_cv.alpha_, l1_ratio=l,
                max_iter=10000
            ).fit(X_scaled[train_idx], y[train_idx])
            return make_positions(
                m.predict(X_scaled[val_idx]), sensitivity=0.2
            )
        return fn
    
    r = evaluate_strategy(make_enet(l1), X_scaled, y, cv=rkf)
    marker = " ←" if r['sharpe_of_sharpe'] > 1.6687 else ""
    print(f"  {'ElasticNet l1='+str(l1):30s}  "
          f"{r['mean']:>7.4f}  {r['std']:>7.4f}  "
          f"{r['sharpe_of_sharpe']:>7.4f}  {r['p10']:>7.4f}{marker}")

# ── Final submission if Lasso beats Ridge ─────────────────────────────────────

print(f"\n=== Decision ===")
print(f"Best Lasso SoS: {best_sos_lasso:.4f}")
print(f"Ridge baseline: 1.6687")

if best_sos_lasso > 1.6687 + 0.01:
    print(f"\nLasso is better — generating submission...")
    
    # Fit on all training data
    model_lasso = Lasso(alpha=best_alpha_lasso, max_iter=10000)
    model_lasso.fit(X_scaled, y)
    
    print(f"\nFinal Lasso coefficients:")
    for feat, coef in zip(FEATURES, model_lasso.coef_):
        status = "ACTIVE" if abs(coef) > 1e-6 else "zeroed"
        print(f"  {feat:20s}  {coef:>+10.6f}  {status}")
    
    for split, bars_df, session_ids, out_path in [
        ("Public",  public_bars,  pub_orig_ids,
         "submission_public_lasso.csv"),
        ("Private", private_bars, priv_orig_ids,
         "submission_private_lasso.csv"),
    ]:
        feats   = build_features(bars_df)[FEATURES]
        feats   = feats.reindex(session_ids).fillna(0)
        X_test  = final_scaler.transform(feats[FEATURES])
        preds   = model_lasso.predict(X_test)
        pos     = make_positions(preds, sensitivity=0.2)
        
        sub = pd.DataFrame({
            'session':         session_ids,
            'target_position': pos,
        }).sort_values('session')
        sub.to_csv(out_path, index=False)
        print(f"{split}: {len(sub)} rows → {out_path}  "
              f"mean={pos.mean():.4f}  std={pos.std():.4f}")
else:
    print(f"\nLasso does not meaningfully beat Ridge.")
    print(f"Stick with submission_private_FINAL.csv")

=== Lasso feature selection ===

Optimal Lasso alpha: 0.000201

Feature coefficients at optimal alpha:
feature                     coef    selected
---------------------------------------------
  cum_return             -0.001072         YES
  last3_return           +0.000983         YES
  slope                  -0.000000  zeroed out
  realized_vol           +0.000000  zeroed out
  range_mean             +0.001359         YES
  up_bar_frac            -0.000000  zeroed out
  wick_ratio             +0.000510         YES

Features selected: 4/7

=== Lasso vs Ridge: CV Sharpe across alpha ===

model                               mean       std       SoS       p10
-----------------------------------------------------------------
  Ridge alpha=100                   2.8479    1.7067    1.6687    0.3638
  Lasso alpha=0.000001              2.8251    1.7124    1.6497    0.3930
  Lasso alpha=0.000002              2.8251    1.7124    1.6498    0.3921
  Lasso alpha=0.000004              2.8253    1.

In [34]:
def build_features(bars: pd.DataFrame) -> pd.DataFrame:
    def session_feats(grp):
        grp = grp.sort_values("bar_ix")
        c   = grp["close"].values
        o   = grp["open"].values
        h   = grp["high"].values
        lo  = grp["low"].values
        n   = len(c)
        br  = np.diff(c) / c[:-1] if n > 1 else np.array([0.0])

        # ── Original ──────────────────────────────────────────────────────────
        cum_ret = c[-1] / c[0] - 1
        vol     = np.std(br)
        rng     = np.mean((h - lo) / c)

        # ── ATH / ATL ─────────────────────────────────────────────────────────
        ath = np.max(h)
        atl = np.min(lo)

        ath_atl_range  = (ath - atl) / c[0]
        close_in_range = (c[-1] - atl) / (ath - atl + 1e-8)
        close_vs_ath   = (c[-1] - ath) / ath
        close_vs_atl   = (c[-1] - atl) / atl
        open_to_ath    = (ath - c[0]) / c[0]
        open_to_atl    = (c[0] - atl) / c[0]
        excursion_asym = open_to_ath - open_to_atl
        mid            = n // 2
        ath_in_second  = float(np.argmax(h) >= mid)
        atl_in_second  = float(np.argmin(lo) >= mid)
        range_ratio    = ath_atl_range / (rng + 1e-8)
        session_mid    = (c[0] + c[-1]) / 2
        ath_vs_mid     = (ath - session_mid) / session_mid
        atl_vs_mid     = (session_mid - atl) / session_mid

        # ── Close-to-open gaps ────────────────────────────────────────────────
        if n > 1:
            gaps_pct     = (o[1:] - c[:-1]) / c[:-1]
            gap_max      = np.max(gaps_pct)
            gap_min      = np.min(gaps_pct)
            gap_mean     = np.mean(gaps_pct)
            gap_std      = np.std(gaps_pct)
            gap_abs_mean = np.mean(np.abs(gaps_pct))
            gap_asym     = np.mean(gaps_pct > 0) - 0.5
            gap_max_abs  = np.max(np.abs(gaps_pct))
            gap_last     = gaps_pct[-1]
            gap_trend    = (np.polyfit(np.arange(len(gaps_pct)),
                                        gaps_pct, 1)[0]
                            if len(gaps_pct) > 2 else 0.0)
        else:
            gap_max = gap_min = gap_mean = gap_std = 0.0
            gap_abs_mean = gap_asym = gap_max_abs = 0.0
            gap_last = gap_trend = 0.0

        # ── High-Low spread variants ──────────────────────────────────────────
        hl_spreads = (h - lo) / c
        hl_std     = np.std(hl_spreads)
        hl_max     = np.max(hl_spreads)
        hl_min     = np.min(hl_spreads)
        hl_last    = hl_spreads[-1]
        hl_first   = hl_spreads[0]
        hl_accel   = hl_last - hl_first
        hl_trend   = (np.polyfit(np.arange(n), hl_spreads, 1)[0]
                      if n > 1 else 0.0)

        return pd.Series({
            # Original
            "cum_return":      cum_ret,
            "last3_return":    c[-1] / c[max(0, n-4)] - 1,
            "slope":           np.polyfit(np.arange(n), c, 1)[0] / c.mean(),
            "realized_vol":    vol,
            "range_mean":      rng,
            "up_bar_frac":     np.mean(c > o),
            "wick_ratio":      np.mean((h - c) / (h - lo + 1e-8)),
            # ATH/ATL
            "ath_atl_range":   ath_atl_range,
            "close_in_range":  close_in_range,
            "close_vs_ath":    close_vs_ath,
            "close_vs_atl":    close_vs_atl,
            "open_to_ath":     open_to_ath,
            "open_to_atl":     open_to_atl,
            "excursion_asym":  excursion_asym,
            "ath_in_second":   ath_in_second,
            "atl_in_second":   atl_in_second,
            "range_ratio":     range_ratio,
            "ath_vs_mid":      ath_vs_mid,
            "atl_vs_mid":      atl_vs_mid,
            # Gaps
            "gap_max":         gap_max,
            "gap_min":         gap_min,
            "gap_mean":        gap_mean,
            "gap_std":         gap_std,
            "gap_abs_mean":    gap_abs_mean,
            "gap_asym":        gap_asym,
            "gap_max_abs":     gap_max_abs,
            "gap_last":        gap_last,
            "gap_trend":       gap_trend,
            # HL spread variants
            "hl_std":          hl_std,
            "hl_max":          hl_max,
            "hl_min":          hl_min,
            "hl_last":         hl_last,
            "hl_first":        hl_first,
            "hl_accel":        hl_accel,
            "hl_trend":        hl_trend,
        })

    return bars.groupby("session").apply(session_feats)

ORIGINAL_FEATURES = [
    "cum_return", "last3_return", "slope",
    "realized_vol", "range_mean", "up_bar_frac", "wick_ratio",
]
ATH_ATL_FEATURES = [
    "ath_atl_range", "close_in_range", "close_vs_ath",
    "close_vs_atl",  "open_to_ath",    "open_to_atl",
    "excursion_asym","ath_in_second",  "atl_in_second",
    "range_ratio",   "ath_vs_mid",     "atl_vs_mid",
]
GAP_FEATURES = [
    "gap_max", "gap_min", "gap_mean", "gap_std",
    "gap_abs_mean", "gap_asym", "gap_max_abs",
    "gap_last", "gap_trend",
]
HL_FEATURES = [
    "hl_std", "hl_max", "hl_min",
    "hl_last", "hl_first", "hl_accel", "hl_trend",
]

ALL_FEATURES = ORIGINAL_FEATURES + ATH_ATL_FEATURES + GAP_FEATURES + HL_FEATURES

# ── Rebuild training features ─────────────────────────────────────────────────

X_train_raw = build_features(seen_train)[ALL_FEATURES]
idx         = X_train_raw.index.intersection(y_train.index)
X_train_raw = X_train_raw.loc[idx]
y           = y_train.loc[idx].values

# ── Correlation check ─────────────────────────────────────────────────────────

print("=== Feature correlations with forward return ===\n")
print(f"{'feature':20s}  {'r':>8}  {'p':>8}  {'sig':>5}")
print("-" * 50)

for col in ALL_FEATURES:
    vals  = X_train_raw[col]
    r, p  = scipy_stats.pearsonr(vals, y_train.loc[idx])
    stars = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else ""
    group = (" ← HL"  if col in HL_FEATURES
             else " ← GAP" if col in GAP_FEATURES
             else " ← ATH" if col in ATH_ATL_FEATURES
             else "")
    print(f"  {col:20s}  {r:>+8.4f}  {p:>8.4f}  {stars:>5}{group}")

# ── Correct KFold CV ──────────────────────────────────────────────────────────

print("\n=== CV Sharpe by feature set (correct KFold) ===\n")
print(f"{'feature set':40s}  {'sharpe':>8}  {'std':>8}")
print("-" * 60)

best_sharpe, best_features = -np.inf, ORIGINAL_FEATURES

for label, feat_list in [
    ("Original 7",                    ORIGINAL_FEATURES),
    ("+ ATH/ATL",                     ORIGINAL_FEATURES + ATH_ATL_FEATURES),
    ("+ Gaps",                        ORIGINAL_FEATURES + GAP_FEATURES),
    ("+ HL variants",                 ORIGINAL_FEATURES + HL_FEATURES),
    ("+ ATH + Gaps",                  ORIGINAL_FEATURES + ATH_ATL_FEATURES + GAP_FEATURES),
    ("+ ATH + HL",                    ORIGINAL_FEATURES + ATH_ATL_FEATURES + HL_FEATURES),
    ("+ Gaps + HL",                   ORIGINAL_FEATURES + GAP_FEATURES + HL_FEATURES),
    ("All features",                  ALL_FEATURES),
]:
    X_raw = X_train_raw[feat_list].values
    fold_sharpes = []

    for train_idx, val_idx in kf.split(X_raw):
        fold_scaler = StandardScaler()
        X_tr = fold_scaler.fit_transform(X_raw[train_idx])
        X_va = fold_scaler.transform(X_raw[val_idx])

        m = Ridge(alpha=100).fit(X_tr, y[train_idx])
        p = make_positions(m.predict(X_va), sensitivity=0.2)
        fold_sharpes.append(compute_sharpe(p, y[val_idx]))

    mean_sh = np.mean(fold_sharpes)
    std_sh  = np.std(fold_sharpes)
    marker  = " ←" if mean_sh > best_sharpe else ""
    print(f"  {label:40s}  {mean_sh:.4f}  {std_sh:.4f}{marker}")

    if mean_sh > best_sharpe:
        best_sharpe   = mean_sh
        best_features = feat_list

# ── Alpha + sensitivity sweep on best feature set ─────────────────────────────

print(f"\n=== Alpha sweep on best: '{best_features}' ===\n")

X_raw_best       = X_train_raw[best_features].values
best_alpha       = 100
best_alpha_sh    = -np.inf

for alpha in [10, 50, 100, 200, 500, 1000, 5000, 10000]:
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_raw_best):
        fold_scaler = StandardScaler()
        X_tr = fold_scaler.fit_transform(X_raw_best[train_idx])
        X_va = fold_scaler.transform(X_raw_best[val_idx])
        m    = Ridge(alpha=alpha).fit(X_tr, y[train_idx])
        p    = make_positions(m.predict(X_va), sensitivity=0.2)
        fold_sharpes.append(compute_sharpe(p, y[val_idx]))

    mean_sh = np.mean(fold_sharpes)
    marker  = " ←" if mean_sh > best_alpha_sh else ""
    print(f"  alpha={alpha:>6}  {mean_sh:.4f}  "
          f"{np.std(fold_sharpes):.4f}{marker}")

    if mean_sh > best_alpha_sh:
        best_alpha_sh = mean_sh
        best_alpha    = alpha

print(f"\n=== Sensitivity sweep (alpha={best_alpha}) ===\n")

best_sens    = 0.2
best_sens_sh = -np.inf

for sens in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40]:
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_raw_best):
        fold_scaler = StandardScaler()
        X_tr = fold_scaler.fit_transform(X_raw_best[train_idx])
        X_va = fold_scaler.transform(X_raw_best[val_idx])
        m    = Ridge(alpha=best_alpha).fit(X_tr, y[train_idx])
        p    = make_positions(m.predict(X_va), sensitivity=sens)
        fold_sharpes.append(compute_sharpe(p, y[val_idx]))

    mean_sh = np.mean(fold_sharpes)
    marker  = " ←" if mean_sh > best_sens_sh else ""
    print(f"  sens={sens:.2f}  {mean_sh:.4f}  "
          f"{np.std(fold_sharpes):.4f}{marker}")

    if mean_sh > best_sens_sh:
        best_sens_sh = mean_sh
        best_sens    = sens

# ── Fit final model and generate submissions ──────────────────────────────────

print(f"\n=== Final model ===")
print(f"Features ({len(best_features)}): {best_features}")
print(f"Alpha:       {best_alpha}")
print(f"Sensitivity: {best_sens}\n")

final_scaler = StandardScaler()
X_final      = final_scaler.fit_transform(X_train_raw[best_features])
final_model  = Ridge(alpha=best_alpha).fit(X_final, y)

print("Feature coefficients:")
for feat, coef in zip(best_features, final_model.coef_):
    print(f"  {feat:20s}  {coef:>+10.6f}")

def make_submission(bars_path, output_path):
    bars  = pd.read_parquet(bars_path, engine="fastparquet")
    feats = build_features(bars)[best_features]
    X     = final_scaler.transform(feats)
    pos   = make_positions(final_model.predict(X), sensitivity=best_sens)

    sub = pd.DataFrame({
        "session":         feats.index,
        "target_position": pos,
    }).sort_values("session")

    sub.to_csv(output_path, index=False)
    print(f"Saved {len(sub)} rows → {output_path}  "
          f"mean={pos.mean():.4f}  std={pos.std():.4f}")

make_submission(
    os.path.join(DATA_DIR, "bars_seen_public_test.parquet"),
    "submission_public_hl.csv",
)
make_submission(
    os.path.join(DATA_DIR, "bars_seen_private_test.parquet"),
    "submission_private_hl.csv",
)

print("\nSubmit public first.")
print("Beat 2.40063 → submit private.")
print("Otherwise revert to submission_private_FINAL.csv.")

=== Feature correlations with forward return ===

feature                      r         p    sig
--------------------------------------------------
  cum_return             -0.0694    0.0281      *
  last3_return           +0.0441    0.1631       
  slope                  -0.0670    0.0341      *
  realized_vol           +0.0716    0.0235      *
  range_mean             +0.0787    0.0128      *
  up_bar_frac            -0.0594    0.0605       
  wick_ratio             +0.0703    0.0263      *
  ath_atl_range          +0.0181    0.5682        ← ATH
  close_in_range         -0.0548    0.0833        ← ATH
  close_vs_ath           -0.0668    0.0346      * ← ATH
  close_vs_atl           -0.0447    0.1582        ← ATH
  open_to_ath            -0.0462    0.1439        ← ATH
  open_to_atl            +0.0700    0.0269      * ← ATH
  excursion_asym         -0.0646    0.0412      * ← ATH
  ath_in_second          -0.0454    0.1514        ← ATH
  atl_in_second          +0.0148    0.6403        ← A

In [35]:
from collections import defaultdict, Counter
import itertools

# ── Extract keyword label for each headline ───────────────────────────────────

def label_headline(headline):
    """Map headline to its keyword label."""
    h_lower = headline.lower()
    for kw in keywords:  # your list of 50 keywords
        if kw in h_lower:
            return kw
    return "unknown"

# Label all training headlines
headlines_train['keyword'] = headlines_train['headline'].apply(label_headline)

print("=== Keyword frequency in training headlines ===\n")
print(headlines_train['keyword'].value_counts().head(20))

# ── Within-session keyword sequences ─────────────────────────────────────────

print("\n=== Building keyword sequences per session ===\n")

session_sequences = {}
for session_id, grp in headlines_train.groupby('session'):
    grp_sorted = grp.sort_values('bar_ix')
    session_sequences[session_id] = grp_sorted['keyword'].tolist()

# Example sequences
print("Sample sequences:")
for sid, seq in list(session_sequences.items())[:5]:
    print(f"  Session {sid}: {seq}")

# ── Transition matrix: P(next keyword | current keyword) ─────────────────────

print("\n=== Keyword transition matrix ===\n")

# Count all consecutive keyword pairs within sessions
pair_counts   = defaultdict(Counter)
keyword_counts = Counter()

for session_id, seq in session_sequences.items():
    for i in range(len(seq) - 1):
        current = seq[i]
        next_kw = seq[i + 1]
        pair_counts[current][next_kw] += 1
        keyword_counts[current] += 1

# What follows "secures" most often?
print("What keywords follow 'secures'?")
if 'secures' in pair_counts:
    total = sum(pair_counts['secures'].values())
    for kw, count in pair_counts['secures'].most_common(10):
        print(f"  {count:4d} ({count/total:.2f})  '{kw}'")

# What precedes "secures" most often?
print("\nWhat keywords precede 'secures'?")
precedes_secures = Counter()
for current, nexts in pair_counts.items():
    if 'secures' in nexts:
        precedes_secures[current] += nexts['secures']

for kw, count in precedes_secures.most_common(10):
    total = keyword_counts[kw]
    print(f"  {count:4d} ({count/total:.2f})  '{kw}'")

# ── Does transition pattern predict returns? ──────────────────────────────────

print("\n=== Transition-based return prediction ===\n")

def build_transition_features(session_sequences, pair_counts, 
                               keyword_counts, y_train):
    """
    For each session, build features based on:
    1. What keywords appear (unigram)
    2. What keyword transitions appear (bigram)
    3. Whether 'secures' follows certain keywords
    """
    # Build transition probabilities from training data
    transition_probs = {}
    for current, nexts in pair_counts.items():
        total = sum(nexts.values())
        transition_probs[current] = {
            kw: count/total for kw, count in nexts.items()
        }
    
    records = []
    for session_id, seq in session_sequences.items():
        if session_id not in y_train.index:
            continue
        
        # P(secures appears after any keyword in this session)
        p_secures_next = np.mean([
            transition_probs.get(kw, {}).get('secures', 0)
            for kw in seq[:-1]  # all but last
        ]) if len(seq) > 1 else 0.0
        
        # Does session contain a keyword that strongly predicts secures?
        max_p_secures = max([
            transition_probs.get(kw, {}).get('secures', 0)
            for kw in seq
        ], default=0.0)
        
        # Bigram presence features
        bigrams = [(seq[i], seq[i+1]) for i in range(len(seq)-1)]
        
        # P(positive keyword follows current keywords)
        positive_kws = {
            "beats analyst expectations with strong earnings growth",
            "reports record quarterly",
            "raises full-year guidance citing robust demand",
            "secures",
            "margin improvement",
            "announces breakthrough in",
        }
        negative_kws = {
            "lowers full-year guidance amid softening demand",
            "misses quarterly revenue estimates",
            "faces class action",
            "steps down unexpectedly",
            "loses key contract",
        }
        
        p_positive_next = np.mean([
            sum(transition_probs.get(kw, {}).get(pk, 0) 
                for pk in positive_kws)
            for kw in seq[:-1]
        ]) if len(seq) > 1 else 0.0
        
        p_negative_next = np.mean([
            sum(transition_probs.get(kw, {}).get(nk, 0) 
                for nk in negative_kws)
            for kw in seq[:-1]
        ]) if len(seq) > 1 else 0.0
        
        # Net transition sentiment
        transition_sentiment = p_positive_next - p_negative_next
        
        # Entropy of transitions (low = predictable sequence)
        trans_probs_all = [
            transition_probs.get(kw, {})
            for kw in seq[:-1]
        ]
        entropies = []
        for tp in trans_probs_all:
            if tp:
                probs = np.array(list(tp.values()))
                entropies.append(-np.sum(probs * np.log(probs + 1e-8)))
        transition_entropy = np.mean(entropies) if entropies else 0.0
        
        records.append({
            'session':              session_id,
            'p_secures_next':       p_secures_next,
            'max_p_secures':        max_p_secures,
            'p_positive_next':      p_positive_next,
            'p_negative_next':      p_negative_next,
            'transition_sentiment': transition_sentiment,
            'transition_entropy':   transition_entropy,
            'n_unique_keywords':    len(set(seq)),
            'n_bigrams':            len(bigrams),
        })
    
    return pd.DataFrame(records).set_index('session')

# ── LOO: compute transitions from training fold only ─────────────────────────

print("Building transition features (LOO-safe)...\n")

# For honest evaluation, build transitions within CV folds
TRANS_FEATURES = [
    'p_secures_next', 'max_p_secures', 'p_positive_next',
    'p_negative_next', 'transition_sentiment', 'transition_entropy',
    'n_unique_keywords', 'n_bigrams',
]

fold_sharpes_trans  = []
fold_sharpes_ohlc   = []
fold_sharpes_combined = []

session_list = list(session_sequences.keys())
session_list = [s for s in session_list if s in y_train.index]
session_arr  = np.array(session_list)

for fold, (train_idx, val_idx) in enumerate(kf.split(session_arr)):
    train_sids = session_arr[train_idx]
    val_sids   = session_arr[val_idx]
    
    # Build transition matrix from TRAIN sessions only
    fold_pair_counts   = defaultdict(Counter)
    fold_kw_counts     = Counter()
    
    for sid in train_sids:
        seq = session_sequences[sid]
        for i in range(len(seq) - 1):
            fold_pair_counts[seq[i]][seq[i+1]] += 1
            fold_kw_counts[seq[i]] += 1
    
    # Build train features
    train_seqs = {s: session_sequences[s] for s in train_sids}
    val_seqs   = {s: session_sequences[s] for s in val_sids}
    
    train_trans = build_transition_features(
        train_seqs, fold_pair_counts, fold_kw_counts, y_train
    )
    val_trans   = build_transition_features(
        val_seqs, fold_pair_counts, fold_kw_counts, y_train
    )
    
    y_tr = y_train.loc[train_sids].values
    y_va = y_train.loc[val_sids].values
    
    # Transition features only
    X_tr_t = StandardScaler().fit_transform(
        train_trans.reindex(train_sids)[TRANS_FEATURES].fillna(0)
    )
    X_va_t = StandardScaler().fit_transform(
        val_trans.reindex(val_sids)[TRANS_FEATURES].fillna(0)
    )
    scaler_t = StandardScaler()
    X_tr_t   = scaler_t.fit_transform(
        train_trans.reindex(train_sids)[TRANS_FEATURES].fillna(0)
    )
    X_va_t   = scaler_t.transform(
        val_trans.reindex(val_sids)[TRANS_FEATURES].fillna(0)
    )
    
    m_t = Ridge(alpha=100).fit(X_tr_t, y_tr)
    p_t = make_positions(m_t.predict(X_va_t), sensitivity=0.2)
    fold_sharpes_trans.append(compute_sharpe(p_t, y_va))
    
    # OHLC only
    X_ohlc_raw = X_train_raw[ORIGINAL_FEATURES].values
    scaler_o   = StandardScaler()
    X_tr_o     = scaler_o.fit_transform(X_ohlc_raw[train_idx])
    X_va_o     = scaler_o.transform(X_ohlc_raw[val_idx])
    
    m_o = Ridge(alpha=10000).fit(X_tr_o, y_tr)
    p_o = make_positions(m_o.predict(X_va_o), sensitivity=0.25)
    fold_sharpes_ohlc.append(compute_sharpe(p_o, y_va))
    
    # Combined
    X_tr_c = np.hstack([X_tr_o, X_tr_t])
    X_va_c = np.hstack([X_va_o, X_va_t])
    
    m_c = Ridge(alpha=10000).fit(X_tr_c, y_tr)
    p_c = make_positions(m_c.predict(X_va_c), sensitivity=0.25)
    fold_sharpes_combined.append(compute_sharpe(p_c, y_va))

print(f"{'model':30s}  {'sharpe':>8}  {'std':>8}")
print("-" * 50)
print(f"  {'OHLC only (alpha=10000)':30s}  "
      f"{np.mean(fold_sharpes_ohlc):.4f}  "
      f"{np.std(fold_sharpes_ohlc):.4f}")
print(f"  {'Transition features only':30s}  "
      f"{np.mean(fold_sharpes_trans):.4f}  "
      f"{np.std(fold_sharpes_trans):.4f}")
print(f"  {'OHLC + transitions':30s}  "
      f"{np.mean(fold_sharpes_combined):.4f}  "
      f"{np.std(fold_sharpes_combined):.4f}")

# ── Correlation check on transition features ──────────────────────────────────

print("\n=== Transition feature correlations ===\n")

# Build on full training data for correlation check
full_trans = build_transition_features(
    session_sequences, pair_counts, keyword_counts, y_train
)

for col in TRANS_FEATURES:
    vals = full_trans[col].reindex(y_train.index).fillna(0)
    r, p = scipy_stats.pearsonr(vals, y_train)
    stars = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else ""
    print(f"  {col:25s}  r={r:>+8.4f}  p={p:.4f}  {stars}")

=== Keyword frequency in training headlines ===

keyword
secures                                     1602
begins scheduled maintenance                 270
reports record quarterly                     251
explores strategic alternatives              250
share buyback program                        239
drop in new customer orders this quarter     238
revises long-term strategy with focus on     238
delays product launch                        238
completes strategic acquisition              237
misses quarterly revenue estimates           237
launches next-generation                     237
to host investor day focused on              232
faces regulatory review                      231
reports rising costs pressuring margins      230
decline in operating income                  225
files for regulatory                         225
wins industry award                          223
sees mixed results in                        221
files routine                                221
faces class 

In [ ]:
print("=== Breakthrough headline analysis ===\n")

breakthrough = headlines_train[
    headlines_train['keyword'] == 'announces breakthrough in'
]
print(f"Total breakthrough headlines: {len(breakthrough)}")
print(f"Sessions with breakthrough:   {breakthrough['session'].nunique()}")

if len(breakthrough) == 0:
    print("No breakthrough headlines in training data.")
else:
    # Return distribution for sessions containing breakthrough
    breakthrough_sessions = breakthrough['session'].unique()
    breakthrough_returns  = y_train.loc[
        [s for s in breakthrough_sessions if s in y_train.index]
    ]
    other_returns = y_train.loc[
        [s for s in y_train.index if s not in breakthrough_sessions]
    ]

    print(f"\nSharpe — breakthrough sessions: "
          f"{sharpe(breakthrough_returns):.4f}  "
          f"(n={len(breakthrough_returns)})")
    print(f"Sharpe — other sessions:        "
          f"{sharpe(other_returns):.4f}  "
          f"(n={len(other_returns)})")

    t, p = scipy_stats.ttest_ind(breakthrough_returns, other_returns)
    print(f"t-test:  t={t:+.4f}  p={p:.4f}")

=== Breakthrough headline analysis ===

Total breakthrough headlines: 214
Sessions with breakthrough:   196

Sharpe — breakthrough sessions: 4.5627  (n=196)
Sharpe — other sessions:        2.3650  (n=804)
t-test:  t=+1.4838  p=0.1382


In [39]:
print("=== Sharpe by keyword presence (all keywords) ===\n")
print(f"{'keyword':50s}  {'n_sess':>7}  {'sharpe_with':>12}  "
      f"{'sharpe_without':>15}  {'diff':>8}  {'p':>8}")
print("-" * 105)

keyword_results = []

for kw in keywords:
    kw_sessions = headlines_train[
        headlines_train['keyword'] == kw
    ]['session'].unique()
    
    valid_with    = [s for s in kw_sessions    if s in y_train.index]
    valid_without = [s for s in y_train.index  if s not in kw_sessions]
    
    if len(valid_with) < 10:
        continue
    
    ret_with    = y_train.loc[valid_with]
    ret_without = y_train.loc[valid_without]
    
    sh_with    = sharpe(ret_with)
    sh_without = sharpe(ret_without)
    diff       = sh_with - sh_without
    
    t, p = scipy_stats.ttest_ind(ret_with, ret_without)
    
    keyword_results.append({
        'keyword':      kw,
        'n_with':       len(valid_with),
        'sharpe_with':  sh_with,
        'sharpe_without': sh_without,
        'diff':         diff,
        'p':            p,
        't':            t,
    })

# Sort by absolute diff
keyword_results.sort(key=lambda x: abs(x['diff']), reverse=True)

for r in keyword_results:
    stars  = "***" if r['p']<0.001 else "**" if r['p']<0.01 else "*" if r['p']<0.05 else "~" if r['p']<0.15 else ""
    marker = " ←" if r['p'] < 0.15 and abs(r['diff']) > 1.0 else ""
    print(f"  {r['keyword']:50s}  "
          f"{r['n_with']:>7}  "
          f"{r['sharpe_with']:>12.4f}  "
          f"{r['sharpe_without']:>15.4f}  "
          f"{r['diff']:>+8.4f}  "
          f"{r['p']:>8.4f} {stars}{marker}")

# ── Bootstrap test specifically for breakthrough ───────────────────────────────

print("\n=== Bootstrap test: breakthrough vs others ===\n")

ret_with    = y_train.loc[[s for s in headlines_train[
    headlines_train['keyword'] == 'announces breakthrough in'
]['session'].unique() if s in y_train.index]]

ret_without = y_train.loc[[s for s in y_train.index 
                            if s not in ret_with.index]]

n_bootstrap = 10000
boot_diffs  = []

for _ in range(n_bootstrap):
    s_with    = np.random.choice(ret_with,    len(ret_with),    replace=True)
    s_without = np.random.choice(ret_without, len(ret_without), replace=True)
    boot_diffs.append(sharpe(s_with) - sharpe(s_without))

boot_diffs    = np.array(boot_diffs)
observed_diff = sharpe(ret_with) - sharpe(ret_without)
p_bootstrap   = np.mean(boot_diffs <= 0)

print(f"Observed Sharpe difference:  {observed_diff:+.4f}")
print(f"Bootstrap p-value:           {p_bootstrap:.4f}")
print(f"Bootstrap 95% CI:            [{np.percentile(boot_diffs, 2.5):.4f}, "
      f"{np.percentile(boot_diffs, 97.5):.4f}]")

# ── CV test: use breakthrough as binary feature ────────────────────────────────

print("\n=== CV Sharpe: OHLC + breakthrough flag ===\n")

has_breakthrough = pd.Series(0, index=y_train.index)
breakthrough_sids = headlines_train[
    headlines_train['keyword'] == 'announces breakthrough in'
]['session'].unique()
has_breakthrough.loc[
    [s for s in breakthrough_sids if s in y_train.index]
] = 1

X_ohlc_raw = X_train_raw[ORIGINAL_FEATURES].values

for label, extra in [
    ("OHLC only",                    None),
    ("OHLC + breakthrough flag",     has_breakthrough.values.reshape(-1,1)),
    ("Breakthrough flag only",       has_breakthrough.values.reshape(-1,1)),
]:
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_ohlc_raw):
        fold_scaler = StandardScaler()
        X_tr = fold_scaler.fit_transform(X_ohlc_raw[train_idx])
        X_va = fold_scaler.transform(X_ohlc_raw[val_idx])
        
        if extra is not None:
            X_tr = np.hstack([X_tr, extra[train_idx]])
            X_va = np.hstack([X_va, extra[val_idx]])
            if label == "Breakthrough flag only":
                X_tr = extra[train_idx]
                X_va = extra[val_idx]
        
        m = Ridge(alpha=10000).fit(X_tr, y[train_idx])
        p = make_positions(m.predict(X_va), sensitivity=0.25)
        fold_sharpes.append(compute_sharpe(p, y[val_idx]))
    
    print(f"  {label:35s}  "
          f"sharpe={np.mean(fold_sharpes):.4f}  "
          f"std={np.std(fold_sharpes):.4f}")

# ── Position sizing: size UP on breakthrough sessions ─────────────────────────

print("\n=== Size UP on breakthrough sessions ===\n")

for mult in [1.5, 2.0, 3.0, 5.0]:
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_ohlc_raw):
        fold_scaler = StandardScaler()
        X_tr = fold_scaler.fit_transform(X_ohlc_raw[train_idx])
        X_va = fold_scaler.transform(X_ohlc_raw[val_idx])
        
        m    = Ridge(alpha=10000).fit(X_tr, y[train_idx])
        pos  = make_positions(m.predict(X_va), sensitivity=0.25)
        
        # Scale up breakthrough sessions
        is_breakthrough = has_breakthrough.values[val_idx]
        pos = np.where(is_breakthrough, pos * mult, pos)
        
        fold_sharpes.append(compute_sharpe(pos, y[val_idx]))
    
    print(f"  breakthrough mult={mult:.1f}x  "
          f"sharpe={np.mean(fold_sharpes):.4f}  "
          f"std={np.std(fold_sharpes):.4f}")

=== Sharpe by keyword presence (all keywords) ===

keyword                                              n_sess   sharpe_with   sharpe_without      diff         p
---------------------------------------------------------------------------------------------------------
  in talks for potential merger, details undisclosed       22       -4.6113           2.9529   -7.5641    0.0207 * ←
  board meeting                                            11       -1.4281           2.7963   -4.2244    0.4454 
  lowers full-year guidance amid softening demand          23       -0.7154           2.8674   -3.5827    0.2693 
  achieves key regulatory milestone ahead of schedule       25       -0.3663           2.8270   -3.1934    0.3366 
  launches next-generation                                216        5.0102           2.2230   +2.7872    0.0637 ~ ←
  reports record quarterly                                229        0.8428           3.3396   -2.4968    0.0380 * ←
  completes planned facility upgrade  

In [40]:
print("=== Building dominant keyword sentiment feature ===\n")

# Empirical Sharpe-based sentiment from the table above
# Use the diff column — empirically validated, not hand-coded
keyword_sharpe_diff = {r['keyword']: r['diff'] 
                       for r in keyword_results}

def build_dominant_sentiment(headlines_df, keyword_sharpe_diff, y_train):
    """
    For each session:
    1. Score each headline by its keyword's Sharpe diff
    2. Take the MAX score (dominant signal)
    3. Also take mean, and fraction positive
    """
    records = []
    for session_id, grp in headlines_df.groupby('session'):
        scores = []
        for _, row in grp.iterrows():
            kw    = row['keyword']
            score = keyword_sharpe_diff.get(kw, 0.0)
            scores.append(score)
        
        if not scores:
            records.append({
                'session':          session_id,
                'kw_max':           0.0,
                'kw_min':           0.0,
                'kw_mean':          0.0,
                'kw_dominant':      0.0,
                'kw_frac_positive': 0.5,
                'kw_spread':        0.0,
            })
            continue
        
        scores = np.array(scores)
        
        # Dominant = highest absolute value score
        max_abs_idx = np.argmax(np.abs(scores))
        
        records.append({
            'session':          session_id,
            'kw_max':           np.max(scores),
            'kw_min':           np.min(scores),
            'kw_mean':          np.mean(scores),
            'kw_dominant':      scores[max_abs_idx],  # strongest signal
            'kw_frac_positive': np.mean(scores > 0),
            'kw_spread':        np.max(scores) - np.min(scores),
        })
    
    return pd.DataFrame(records).set_index('session')

KW_FEATURES = [
    'kw_max', 'kw_min', 'kw_mean', 
    'kw_dominant', 'kw_frac_positive', 'kw_spread'
]

# ── CRITICAL: must use LOO to avoid leakage ───────────────────────────────────
# The keyword_sharpe_diff was computed on ALL training data
# We must recompute it within each fold

print("=== LOO-safe CV with keyword sentiment ===\n")
print(f"{'model':35s}  {'sharpe':>8}  {'std':>8}")
print("-" * 55)

X_ohlc_raw = X_train_raw[ORIGINAL_FEATURES].values
session_arr = np.array(y_train.index.tolist())

fold_sharpes_ohlc     = []
fold_sharpes_kw       = []
fold_sharpes_combined = []

for train_idx, val_idx in kf.split(X_ohlc_raw):
    train_sids = session_arr[train_idx]
    val_sids   = session_arr[val_idx]

    # ── Recompute keyword Sharpe diff on TRAIN fold only ─────────────────────
    fold_kw_diff = {}
    train_hl = headlines_train[headlines_train['session'].isin(train_sids)]
    y_tr     = y_train.loc[train_sids]

    for kw in keywords:
        kw_sids   = train_hl[train_hl['keyword'] == kw]['session'].unique()
        valid_with = [s for s in kw_sids    if s in y_tr.index]
        valid_wo   = [s for s in y_tr.index if s not in kw_sids]

        if len(valid_with) < 5 or len(valid_wo) < 5:
            fold_kw_diff[kw] = 0.0
            continue

        sh_with = sharpe(y_tr.loc[valid_with])
        sh_wo   = sharpe(y_tr.loc[valid_wo])
        fold_kw_diff[kw] = sh_with - sh_wo

    # ── Build features for train and val ────────────────────────────────────
    train_hl_fold = headlines_train[
        headlines_train['session'].isin(train_sids)
    ]
    val_hl_fold   = headlines_train[
        headlines_train['session'].isin(val_sids)
    ]

    train_kw = build_dominant_sentiment(
        train_hl_fold, fold_kw_diff, y_train
    )[KW_FEATURES].reindex(train_sids).fillna(0).values

    val_kw   = build_dominant_sentiment(
        val_hl_fold, fold_kw_diff, y_train
    )[KW_FEATURES].reindex(val_sids).fillna(0).values

    y_tr_arr = y_train.loc[train_sids].values
    y_va_arr = y_train.loc[val_sids].values

    # OHLC only
    scaler_o = StandardScaler()
    X_tr_o   = scaler_o.fit_transform(X_ohlc_raw[train_idx])
    X_va_o   = scaler_o.transform(X_ohlc_raw[val_idx])
    m_o      = Ridge(alpha=10000).fit(X_tr_o, y_tr_arr)
    p_o      = make_positions(m_o.predict(X_va_o), sensitivity=0.25)
    fold_sharpes_ohlc.append(compute_sharpe(p_o, y_va_arr))

    # Keyword sentiment only
    scaler_k = StandardScaler()
    X_tr_k   = scaler_k.fit_transform(train_kw)
    X_va_k   = scaler_k.transform(val_kw)
    m_k      = Ridge(alpha=10000).fit(X_tr_k, y_tr_arr)
    p_k      = make_positions(m_k.predict(X_va_k), sensitivity=0.25)
    fold_sharpes_kw.append(compute_sharpe(p_k, y_va_arr))

    # Combined
    X_tr_c   = np.hstack([X_tr_o, X_tr_k])
    X_va_c   = np.hstack([X_va_o, X_va_k])
    m_c      = Ridge(alpha=10000).fit(X_tr_c, y_tr_arr)
    p_c      = make_positions(m_c.predict(X_va_c), sensitivity=0.25)
    fold_sharpes_combined.append(compute_sharpe(p_c, y_va_arr))

print(f"  {'OHLC only':35s}  "
      f"{np.mean(fold_sharpes_ohlc):.4f}  "
      f"{np.std(fold_sharpes_ohlc):.4f}")
print(f"  {'Keyword sentiment only':35s}  "
      f"{np.mean(fold_sharpes_kw):.4f}  "
      f"{np.std(fold_sharpes_kw):.4f}")
print(f"  {'OHLC + keyword sentiment':35s}  "
      f"{np.mean(fold_sharpes_combined):.4f}  "
      f"{np.std(fold_sharpes_combined):.4f}")

# ── Also test: size up on positive cluster sessions ───────────────────────────

print("\n=== Size up on positive keyword cluster sessions ===\n")

# Positive cluster: keywords with diff > 1.5
positive_cluster = {r['keyword'] for r in keyword_results 
                    if r['diff'] > 1.5}
negative_cluster = {r['keyword'] for r in keyword_results 
                    if r['diff'] < -1.5}

print(f"Positive cluster ({len(positive_cluster)} keywords):")
for kw in sorted(positive_cluster):
    print(f"  + {kw}")
print(f"\nNegative cluster ({len(negative_cluster)} keywords):")
for kw in sorted(negative_cluster):
    print(f"  - {kw}")

# Flag sessions
def flag_session_cluster(headlines_df, positive_cluster, negative_cluster):
    records = []
    for session_id, grp in headlines_df.groupby('session'):
        kws = set(grp['keyword'].tolist())
        n_pos = len(kws & positive_cluster)
        n_neg = len(kws & negative_cluster)
        records.append({
            'session':   session_id,
            'n_pos_kw':  n_pos,
            'n_neg_kw':  n_neg,
            'cluster':   'positive' if n_pos > n_neg
                         else 'negative' if n_neg > n_pos
                         else 'neutral',
        })
    return pd.DataFrame(records).set_index('session')

cluster_flags = flag_session_cluster(
    headlines_train, positive_cluster, negative_cluster
)

print(f"\nSession cluster distribution:")
print(cluster_flags['cluster'].value_counts())

# Sharpe by cluster
for cluster, grp in cluster_flags.groupby('cluster'):
    valid = [s for s in grp.index if s in y_train.index]
    if valid:
        print(f"  {cluster:10s}:  "
              f"sharpe={sharpe(y_train.loc[valid]):.4f}  "
              f"n={len(valid)}")

# CV: size by cluster
print("\n=== CV: size up positive cluster, down negative ===\n")

for hi_mult, lo_mult in [(10, 0.1),(2, 0.5), (1.5, 0.5), (2.0, 0.5), (1.3, 0.7), (2.0, 1.0)]:
    fold_sharpes = []
    for train_idx, val_idx in kf.split(X_ohlc_raw):
        train_sids = session_arr[train_idx]
        val_sids   = session_arr[val_idx]

        # Recompute clusters on train fold only
        fold_kw_results = []
        train_hl = headlines_train[
            headlines_train['session'].isin(train_sids)
        ]
        y_tr = y_train.loc[train_sids]

        for kw in keywords:
            kw_sids    = train_hl[train_hl['keyword'] == kw]['session'].unique()
            valid_with = [s for s in kw_sids    if s in y_tr.index]
            valid_wo   = [s for s in y_tr.index if s not in kw_sids]
            if len(valid_with) < 5 or len(valid_wo) < 5:
                continue
            diff = sharpe(y_tr.loc[valid_with]) - sharpe(y_tr.loc[valid_wo])
            fold_kw_results.append({'keyword': kw, 'diff': diff})

        fold_pos_cluster = {r['keyword'] for r in fold_kw_results
                            if r['diff'] > 1.5}
        fold_neg_cluster = {r['keyword'] for r in fold_kw_results
                            if r['diff'] < -1.5}

        # Flag val sessions
        val_hl    = headlines_train[
            headlines_train['session'].isin(val_sids)
        ]
        val_flags = flag_session_cluster(
            val_hl, fold_pos_cluster, fold_neg_cluster
        ).reindex(val_sids).fillna('neutral')

        # OHLC base positions
        scaler_o = StandardScaler()
        X_tr_o   = scaler_o.fit_transform(X_ohlc_raw[train_idx])
        X_va_o   = scaler_o.transform(X_ohlc_raw[val_idx])
        m_o      = Ridge(alpha=10000).fit(X_tr_o, y_train.loc[train_sids].values)
        base_pos = make_positions(m_o.predict(X_va_o), sensitivity=0.25)

        # Scale by cluster
        scale = np.where(
            val_flags['cluster'].values == 'positive', hi_mult,
            np.where(
                val_flags['cluster'].values == 'negative', lo_mult,
                1.0
            )
        )
        positions = np.maximum(base_pos * scale, 0.05)
        fold_sharpes.append(
            compute_sharpe(positions, y_train.loc[val_sids].values)
        )

    print(f"  pos={hi_mult:.1f}x neg={lo_mult:.1f}x  "
          f"sharpe={np.mean(fold_sharpes):.4f}  "
          f"std={np.std(fold_sharpes):.4f}")

=== Building dominant keyword sentiment feature ===

=== LOO-safe CV with keyword sentiment ===

model                                  sharpe       std
-------------------------------------------------------
  OHLC only                            2.8660  0.5438
  Keyword sentiment only               2.7148  0.4793
  OHLC + keyword sentiment             2.7790  0.4738

=== Size up on positive keyword cluster sessions ===

Positive cluster (9 keywords):
  + announces breakthrough in
  + announces major organizational restructuring
  + confirms participation
  + expands operations into
  + launches next-generation
  + margin improvement
  + opens new office
  + revises long-term strategy with focus on
  + to present at

Negative cluster (12 keywords):
  - achieves key regulatory milestone ahead of schedule
  - addresses investor concerns in open letter
  - announces restructuring plan
  - board meeting
  - completes planned facility upgrade
  - explores strategic alternatives
  - files f

In [ ]:
print("=== Separate models: breakthrough vs non-breakthrough ===\n")

# ── Identify breakthrough sessions ───────────────────────────────────────────

breakthrough_sids = set(
    headlines_train[
        headlines_train['keyword'] == 'announces breakthrough in'
    ]['session'].unique()
)

all_sids     = np.array(y_train.index.tolist())
is_bt        = np.array([s in breakthrough_sids for s in all_sids])

bt_idx       = np.where(is_bt)[0]
non_bt_idx   = np.where(~is_bt)[0]

print(f"Breakthrough sessions:     {is_bt.sum():4d} ({is_bt.mean():.1%})")
print(f"Non-breakthrough sessions: {(~is_bt).sum():4d} ({(~is_bt).mean():.1%})")

X_raw = X_train_raw[ORIGINAL_FEATURES].values

# ── In-sample sanity check ────────────────────────────────────────────────────

print(f"\nIn-sample Sharpe (always long):")
print(f"  Breakthrough:      {sharpe(y_train.iloc[bt_idx]):.4f}")
print(f"  Non-breakthrough:  {sharpe(y_train.iloc[non_bt_idx]):.4f}")
print(f"  All sessions:      {sharpe(y_train):.4f}")

# ── Proper CV: split within each population ───────────────────────────────────

print("\n=== CV Sharpe: separate models per population ===\n")
print(f"{'model':45s}  {'sharpe':>8}  {'std':>8}")
print("-" * 62)

# Strategy 1: Single model for all (baseline)
fold_sharpes_single = []
for train_idx, val_idx in kf.split(X_raw):
    fs = StandardScaler()
    X_tr = fs.fit_transform(X_raw[train_idx])
    X_va = fs.transform(X_raw[val_idx])
    m    = Ridge(alpha=10000).fit(X_tr, y[train_idx])
    p    = make_positions(m.predict(X_va), sensitivity=0.25)
    fold_sharpes_single.append(compute_sharpe(p, y[val_idx]))

print(f"  {'Single model (baseline)':45s}  "
      f"{np.mean(fold_sharpes_single):.4f}  "
      f"{np.std(fold_sharpes_single):.4f}")

# Strategy 2: Separate models, combined portfolio
# Key insight: CV must respect the population split
# For each val session, use the model trained on same-population train sessions

fold_sharpes_separate  = []
fold_sharpes_bt_only   = []
fold_sharpes_nbt_only  = []

for train_idx, val_idx in kf.split(X_raw):
    y_tr_all = y[train_idx]
    y_va_all = y[val_idx]
    is_bt_tr = is_bt[train_idx]
    is_bt_va = is_bt[val_idx]

    # ── Breakthrough model ────────────────────────────────────────────────────
    bt_tr_mask  = is_bt_tr
    bt_va_mask  = is_bt_va

    if bt_tr_mask.sum() >= 10 and bt_va_mask.sum() >= 1:
        fs_bt   = StandardScaler()
        X_tr_bt = fs_bt.fit_transform(X_raw[train_idx][bt_tr_mask])
        X_va_bt = fs_bt.transform(X_raw[val_idx][bt_va_mask])
        m_bt    = Ridge(alpha=10000).fit(X_tr_bt, y_tr_all[bt_tr_mask])
        p_bt    = make_positions(m_bt.predict(X_va_bt), sensitivity=0.25)
    else:
        p_bt    = np.ones(bt_va_mask.sum())

    # ── Non-breakthrough model ────────────────────────────────────────────────
    nbt_tr_mask = ~is_bt_tr
    nbt_va_mask = ~is_bt_va

    if nbt_tr_mask.sum() >= 10 and nbt_va_mask.sum() >= 1:
        fs_nbt   = StandardScaler()
        X_tr_nbt = fs_nbt.fit_transform(X_raw[train_idx][nbt_tr_mask])
        X_va_nbt = fs_nbt.transform(X_raw[val_idx][nbt_va_mask])
        m_nbt    = Ridge(alpha=10000).fit(X_tr_nbt, y_tr_all[nbt_tr_mask])
        p_nbt    = make_positions(m_nbt.predict(X_va_nbt), sensitivity=0.25)
    else:
        p_nbt    = np.ones(nbt_va_mask.sum())

    # ── Combine into full val positions ──────────────────────────────────────
    positions_all          = np.ones(len(val_idx))
    positions_all[bt_va_mask]  = p_bt
    positions_all[~bt_va_mask] = p_nbt

    fold_sharpes_separate.append(
        compute_sharpe(positions_all, y_va_all)
    )

    # Track sub-populations separately
    if bt_va_mask.sum() > 0:
        fold_sharpes_bt_only.append(
            compute_sharpe(p_bt, y_va_all[bt_va_mask])
        )
    if nbt_va_mask.sum() > 0:
        fold_sharpes_nbt_only.append(
            compute_sharpe(p_nbt, y_va_all[nbt_va_mask])
        )

print(f"  {'Separate models (combined)':45s}  "
      f"{np.mean(fold_sharpes_separate):.4f}  "
      f"{np.std(fold_sharpes_separate):.4f}")
print(f"  {'  → Breakthrough sub-portfolio':45s}  "
      f"{np.mean(fold_sharpes_bt_only):.4f}  "
      f"{np.std(fold_sharpes_bt_only):.4f}")
print(f"  {'  → Non-breakthrough sub-portfolio':45s}  "
      f"{np.mean(fold_sharpes_nbt_only):.4f}  "
      f"{np.std(fold_sharpes_nbt_only):.4f}")

# Strategy 3: Separate alpha and sensitivity per population
print("\n=== Tune alpha + sensitivity separately per population ===\n")
print(f"{'bt_alpha':>10}  {'bt_sens':>8}  {'nbt_alpha':>10}  "
      f"{'nbt_sens':>9}  {'sharpe':>8}  {'std':>8}")
print("-" * 62)

best_sharpe, best_cfg = -np.inf, {}

for bt_alpha, nbt_alpha in [(1000,10000),(10000,10000),(10000,1000),(1000,1000)]:
    for bt_sens, nbt_sens in [(0.2,0.25),(0.25,0.25),(0.3,0.25),(0.25,0.2)]:
        fold_sharpes = []
        for train_idx, val_idx in kf.split(X_raw):
            y_tr_all    = y[train_idx]
            y_va_all    = y[val_idx]
            is_bt_tr    = is_bt[train_idx]
            is_bt_va    = is_bt[val_idx]
            positions_all = np.ones(len(val_idx))

            # Breakthrough
            bt_tr = is_bt_tr
            bt_va = is_bt_va
            if bt_tr.sum() >= 10 and bt_va.sum() >= 1:
                fs      = StandardScaler()
                X_tr_bt = fs.fit_transform(X_raw[train_idx][bt_tr])
                X_va_bt = fs.transform(X_raw[val_idx][bt_va])
                m       = Ridge(alpha=bt_alpha).fit(X_tr_bt, y_tr_all[bt_tr])
                positions_all[bt_va] = make_positions(
                    m.predict(X_va_bt), sensitivity=bt_sens
                ) * 2 + 1

            # Non-breakthrough
            nbt_tr = ~is_bt_tr
            nbt_va = ~is_bt_va
            if nbt_tr.sum() >= 10 and nbt_va.sum() >= 1:
                fs       = StandardScaler()
                X_tr_nbt = fs.fit_transform(X_raw[train_idx][nbt_tr])
                X_va_nbt = fs.transform(X_raw[val_idx][nbt_va])
                m        = Ridge(alpha=nbt_alpha).fit(X_tr_nbt, y_tr_all[nbt_tr])
                positions_all[nbt_va] = make_positions(
                    m.predict(X_va_nbt), sensitivity=nbt_sens
                )

            fold_sharpes.append(compute_sharpe(positions_all, y_va_all))

        mean_sh = np.mean(fold_sharpes)
        marker  = " ←" if mean_sh > best_sharpe else ""
        print(f"  {bt_alpha:>8}  {bt_sens:>8}  {nbt_alpha:>10}  "
              f"{nbt_sens:>9}  {mean_sh:>8.4f}  "
              f"{np.std(fold_sharpes):>8.4f}{marker}")

        if mean_sh > best_sharpe:
            best_sharpe = mean_sh
            best_cfg    = dict(
                bt_alpha=bt_alpha,   bt_sens=bt_sens,
                nbt_alpha=nbt_alpha, nbt_sens=nbt_sens
            )

print(f"\nBest config: {best_cfg}")
print(f"Best CV Sharpe: {best_sharpe:.4f}")
print(f"Baseline:       2.8660")

# ── Generate submission if improvement found ──────────────────────────────────

if best_sharpe >0.02:
    print(f"\n=== Generating separate-model submissions ===\n")

    # Fit on full training data
    bt_mask  = is_bt
    nbt_mask = ~is_bt

    # Breakthrough model
    fs_bt_final  = StandardScaler()
    X_bt_final   = fs_bt_final.fit_transform(X_raw[bt_mask])
    m_bt_final   = Ridge(alpha=best_cfg['bt_alpha']).fit(
        X_bt_final, y[bt_mask]
    )

    # Non-breakthrough model
    fs_nbt_final = StandardScaler()
    X_nbt_final  = fs_nbt_final.fit_transform(X_raw[nbt_mask])
    m_nbt_final  = Ridge(alpha=best_cfg['nbt_alpha']).fit(
        X_nbt_final, y[nbt_mask]
    )

    def make_split_submission(bars_path, hl_path, output_path):
        bars = pd.read_parquet(bars_path, engine="fastparquet")
        hl   = pd.read_parquet(hl_path,  engine="fastparquet")

        feats   = build_features(bars)[ORIGINAL_FEATURES]
        sessions = feats.index.tolist()

        # Identify breakthrough sessions in test
        hl['keyword']   = hl['headline'].apply(label_headline)
        test_bt_sids    = set(
            hl[hl['keyword'] == 'announces breakthrough in']['session'].unique()
        )
        is_bt_test      = np.array([s in test_bt_sids for s in sessions])

        print(f"  Test breakthrough sessions:     {is_bt_test.sum()} "
              f"({is_bt_test.mean():.1%})")
        print(f"  Test non-breakthrough sessions: {(~is_bt_test).sum()}")

        positions = np.ones(len(sessions))

        # Breakthrough positions
        if is_bt_test.sum() > 0:
            X_bt  = fs_bt_final.transform(
                feats.values[is_bt_test]
            )
            positions[is_bt_test] = make_positions(
                m_bt_final.predict(X_bt),
                sensitivity=best_cfg['bt_sens']
            ) * 2 + 1

        # Non-breakthrough positions
        if (~is_bt_test).sum() > 0:
            X_nbt = fs_nbt_final.transform(
                feats.values[~is_bt_test]
            )
            positions[~is_bt_test] = make_positions(
                m_nbt_final.predict(X_nbt),
                sensitivity=best_cfg['nbt_sens']
            )

        sub = pd.DataFrame({
            'session':         sessions,
            'target_position': positions,
        }).sort_values('session')
        sub.to_csv(output_path, index=False)
        print(f"  Saved {len(sub)} rows → {output_path}  "
              f"mean={positions.mean():.4f}  std={positions.std():.4f}")

    make_split_submission(
        os.path.join(DATA_DIR, "bars_seen_public_test.parquet"),
        os.path.join(DATA_DIR, "headlines_seen_public_test.parquet"),
        "submission_public_split.csv",
    )
    make_split_submission(
        os.path.join(DATA_DIR, "bars_seen_private_test.parquet"),
        os.path.join(DATA_DIR, "headlines_seen_private_test.parquet"),
        "submission_private_split.csv",
    )
else:
    print(f"\nSeparate models do not beat baseline by >0.02.")
    print(f"Submit submission_private_hl.csv")

=== Separate models: breakthrough vs non-breakthrough ===

Breakthrough sessions:      196 (19.6%)
Non-breakthrough sessions:  804 (80.4%)

In-sample Sharpe (always long):
  Breakthrough:      4.5627
  Non-breakthrough:  2.3650
  All sessions:      2.7661

=== CV Sharpe: separate models per population ===

model                                            sharpe       std
--------------------------------------------------------------
  Single model (baseline)                        2.8660  0.5438
  Separate models (combined)                     2.8144  0.5651
    → Breakthrough sub-portfolio                 4.3693  1.5235
    → Non-breakthrough sub-portfolio             2.4958  0.5212

=== Tune alpha + sensitivity separately per population ===

  bt_alpha   bt_sens   nbt_alpha   nbt_sens    sharpe       std
--------------------------------------------------------------
      1000       0.2       10000       0.25    2.8732    0.5555 ←
      1000      0.25       10000       0.25    2.8514

In [41]:
def sharpe(returns):
    return returns.mean() / returns.std() * 16

from collections import defaultdict, Counter
import itertools
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats

# ── Merge all headlines across all sessions into one global feed ──────────────

print("=== Global headline feed analysis ===\n")

# Assign a global time index: sort by session then bar_ix
global_feed = (
    headlines_train
    .copy()
    .sort_values(['session', 'bar_ix'])
    .reset_index(drop=True)
)
global_feed['global_ix'] = global_feed.index
global_feed['keyword']   = global_feed['headline'].apply(label_headline)

print(f"Total headlines in global feed: {len(global_feed)}")
print(f"Unique keywords:                {global_feed['keyword'].nunique()}")
print(f"\nFirst 20 headlines in global feed:")
print(global_feed[['session','bar_ix','keyword']].head(20).to_string())

# ── Global transition matrix (across session boundaries) ─────────────────────

print("\n=== Global transition matrix (ignoring session boundaries) ===\n")

global_keywords = global_feed['keyword'].tolist()

# Count all consecutive pairs in the global feed
global_pairs = defaultdict(Counter)
for i in range(len(global_keywords) - 1):
    current = global_keywords[i]
    next_kw = global_keywords[i + 1]
    global_pairs[current][next_kw] += 1

# Compare: within-session vs cross-session transitions
within_pairs = defaultdict(Counter)
cross_pairs  = defaultdict(Counter)

for i in range(len(global_feed) - 1):
    current_row = global_feed.iloc[i]
    next_row    = global_feed.iloc[i + 1]
    current_kw  = current_row['keyword']
    next_kw     = next_row['keyword']
    
    if current_row['session'] == next_row['session']:
        within_pairs[current_kw][next_kw] += 1
    else:
        cross_pairs[current_kw][next_kw] += 1

print(f"Within-session transitions: {sum(sum(v.values()) for v in within_pairs.values())}")
print(f"Cross-session transitions:  {sum(sum(v.values()) for v in cross_pairs.values())}")

# ── Key question: do cross-session transitions differ from within-session? ─────

print("\n=== Cross-session vs within-session transition entropy ===\n")

def transition_entropy(pair_dict):
    """Measure how uniform the transition distribution is."""
    entropies = []
    for current, nexts in pair_dict.items():
        total  = sum(nexts.values())
        probs  = np.array([c/total for c in nexts.values()])
        h      = -np.sum(probs * np.log(probs + 1e-8))
        entropies.append(h)
    return np.mean(entropies)

print(f"Within-session transition entropy: {transition_entropy(within_pairs):.4f}")
print(f"Cross-session transition entropy:  {transition_entropy(cross_pairs):.4f}")
print(f"(Higher entropy = more uniform = more random)")

# ── Co-occurrence analysis across sessions ────────────────────────────────────

print("\n=== Global keyword co-occurrence analysis ===\n")

# For each session, get its keyword set
session_keyword_sets = {}
for session_id, grp in headlines_train.groupby('session'):
    session_keyword_sets[session_id] = list(grp['keyword'].values)

# Build co-occurrence matrix: how often do keyword A and B 
# appear in sessions with above-average returns?
print("Building keyword co-occurrence vs return matrix...")

# For each pair of keywords, find sessions containing BOTH
# and check if their returns differ from sessions with neither
keyword_list = list(keywords)
pair_results = []

for kw1, kw2 in itertools.combinations(keyword_list, 2):
    # Sessions with both
    both = [s for s, kws in session_keyword_sets.items()
            if kw1 in kws and kw2 in kws
            and s in y_train.index]
    # Sessions with neither
    neither = [s for s, kws in session_keyword_sets.items()
               if kw1 not in kws and kw2 not in kws
               and s in y_train.index]
    
    if len(both) < 10 or len(neither) < 10:
        continue
    
    ret_both    = y_train.loc[both]
    ret_neither = y_train.loc[neither]
    
    sh_both    = sharpe(ret_both)
    sh_neither = sharpe(ret_neither)
    diff       = sh_both - sh_neither
    
    t, p = scipy_stats.ttest_ind(ret_both, ret_neither)
    
    pair_results.append({
        'kw1':       kw1,
        'kw2':       kw2,
        'n_both':    len(both),
        'n_neither': len(neither),
        'sh_both':   sh_both,
        'sh_neither':sh_neither,
        'diff':      diff,
        'p':         p,
    })

pair_results.sort(key=lambda x: abs(x['diff']), reverse=True)

print(f"Keyword pairs tested: {len(pair_results)}")
print(f"\nTop 20 pairs by Sharpe difference:")
print(f"{'kw1':35s}  {'kw2':35s}  "
      f"{'n':>5}  {'diff':>8}  {'p':>8}")
print("-" * 100)

sig_pairs = []
for r in pair_results[:20]:
    stars  = "***" if r['p']<0.001 else "**" if r['p']<0.01 \
             else "*" if r['p']<0.05 else "~" if r['p']<0.15 else ""
    print(f"  {r['kw1']:35s}  {r['kw2']:35s}  "
          f"{r['n_both']:>5}  {r['diff']:>+8.4f}  "
          f"{r['p']:>8.4f} {stars}")
    if r['p'] < 0.05:
        sig_pairs.append(r)

print(f"\nStatistically significant pairs (p<0.05): {len(sig_pairs)}")

# ── Expected number of significant pairs by chance ────────────────────────────

n_pairs_tested = len(pair_results)
expected_false_positives = n_pairs_tested * 0.05
print(f"Pairs tested:                    {n_pairs_tested}")
print(f"Expected false positives at p<0.05: {expected_false_positives:.1f}")
print(f"Actual significant pairs:           {len(sig_pairs)}")

if len(sig_pairs) <= expected_false_positives * 1.5:
    print("\n→ Significant pairs consistent with random chance.")
    print("  No real co-occurrence signal.")
else:
    print("\n→ More significant pairs than expected by chance.")
    print("  Possible real co-occurrence signal — investigate further.")

# ── Global sequential patterns: does breakthrough cluster in time? ─────────────

print("\n=== Global temporal clustering of breakthrough ===\n")

# Sort sessions by session ID (proxy for time order)
# Does breakthrough cluster in certain session ranges?

bt_sessions = sorted([
    s for s in headlines_train[
        headlines_train['keyword'] == 'announces breakthrough in'
    ]['session'].unique()
    if s in y_train.index
])

all_sessions = sorted(y_train.index.tolist())
n_sessions   = len(all_sessions)

# Position of each breakthrough session in the global order
bt_positions = [all_sessions.index(s) / n_sessions 
                for s in bt_sessions if s in all_sessions]

print(f"Breakthrough session positions (0=first, 1=last):")
print(f"  Mean position: {np.mean(bt_positions):.3f} "
      f"(0.5 = uniformly distributed)")
print(f"  Std position:  {np.std(bt_positions):.3f}")

# KS test: are breakthrough sessions uniformly distributed?
ks_stat, ks_p = scipy_stats.kstest(bt_positions, 'uniform')
print(f"  KS test vs uniform: stat={ks_stat:.4f}  p={ks_p:.4f}")
print(f"  {'Clustered in time' if ks_p < 0.05 else 'Uniformly distributed'}")

# ── Do consecutive sessions share return direction? ───────────────────────────

print("\n=== Consecutive session return autocorrelation ===\n")

# Sort sessions by ID and check if returns are autocorrelated
sorted_sessions = sorted(y_train.index.tolist())
sorted_returns  = y_train.loc[sorted_sessions].values

for lag in [1, 2, 3, 5, 10]:
    r, p = scipy_stats.pearsonr(
        sorted_returns[:-lag], 
        sorted_returns[lag:]
    )
    print(f"  Lag {lag:2d}: r={r:+.4f}  p={p:.4f}")

# ── Does the keyword preceding a session's first headline predict return? ───────

print("\n=== Cross-session: does previous session's last keyword predict next return? ===\n")

# For each session (except first), get the last keyword of the previous session
# and check if it predicts the current session's return
prev_last_kw_signal = []

for i in range(1, len(sorted_sessions)):
    prev_sid = sorted_sessions[i - 1]
    curr_sid = sorted_sessions[i]
    
    if curr_sid not in y_train.index:
        continue
    
    prev_hl = headlines_train[headlines_train['session'] == prev_sid]
    if len(prev_hl) == 0:
        continue
    
    prev_last_kw = (prev_hl.sort_values('bar_ix')
                           .iloc[-1]['keyword'])
    
    # Use the keyword's empirical Sharpe diff as signal
    kw_signal = keyword_sharpe_diff.get(prev_last_kw, 0.0)
    
    prev_last_kw_signal.append({
        'session':    curr_sid,
        'prev_kw':    prev_last_kw,
        'kw_signal':  kw_signal,
        'return':     y_train[curr_sid],
    })

df_cross = pd.DataFrame(prev_last_kw_signal)

r, p = scipy_stats.pearsonr(df_cross['kw_signal'], df_cross['return'])
print(f"Previous session last keyword → current return:")
print(f"  r={r:+.4f}  p={p:.4f}")
print(f"  {'Signal exists' if p < 0.05 else 'No signal'}")

# Also check: does previous session's return predict current session's return?
prev_returns = y_train.loc[sorted_sessions[:-1]].values
curr_returns = y_train.loc[sorted_sessions[1:]].values
r2, p2 = scipy_stats.pearsonr(prev_returns, curr_returns)
print(f"\nPrevious session return → current session return:")
print(f"  r={r2:+.4f}  p={p2:.4f}")
print(f"  {'Autocorrelation exists' if p2 < 0.05 else 'Sessions are independent'}")

=== Global headline feed analysis ===

Total headlines in global feed: 9740
Unique keywords:                50

First 20 headlines in global feed:
    session  bar_ix                           keyword
0         0       6                  opens new office
1         0      12                           secures
2         0      14                         names new
3         0      20                           secures
4         0      22                           secures
5         0      26       decline in operating income
6         0      26                           secures
7         0      33           expands operations into
8         0      44             sees mixed results in
9         0      47                           secures
10        0      48                faces class action
11        1       3       decline in operating income
12        1       5          reports record quarterly
13        1       6                loses key contract
14        1       6  increase in customer a

In [43]:
print("=== LOO-safe CV: keyword pair co-occurrence signal ===\n")

session_arr  = np.array(y_train.index.tolist())
X_ohlc_raw   = X_train_raw[ORIGINAL_FEATURES].values

fold_sharpes_ohlc     = []
fold_sharpes_pairs    = []
fold_sharpes_combined = []

for train_idx, val_idx in kf.split(X_ohlc_raw):
    train_sids = session_arr[train_idx]
    val_sids   = session_arr[val_idx]

    y_tr = y_train.loc[train_sids].values
    y_va = y_train.loc[val_sids].values

    # ── Recompute keyword pairs on TRAIN fold only ────────────────────────────
    train_kw_sets = {
        s: set(headlines_train[
            headlines_train['session'] == s
        ]['keyword'].tolist())
        for s in train_sids
    }

    fold_pair_signals = {}
    for kw1, kw2 in itertools.combinations(keyword_list, 2):
        both    = [s for s, kws in train_kw_sets.items()
                   if kw1 in kws and kw2 in kws]
        neither = [s for s, kws in train_kw_sets.items()
                   if kw1 not in kws and kw2 not in kws]

        if len(both) < 8 or len(neither) < 8:
            continue

        diff = (sharpe(y_train.loc[both]) -
                sharpe(y_train.loc[neither]))
        fold_pair_signals[(kw1, kw2)] = diff

    # ── Build pair signal for TRAIN sessions ──────────────────────────────────
    def session_pair_signal(sids, kw_sets, pair_signals):
        signals = []
        for s in sids:
            kws    = kw_sets.get(s, set())
            scores = [
                diff for (kw1, kw2), diff in pair_signals.items()
                if kw1 in kws and kw2 in kws
            ]
            signals.append(np.mean(scores) if scores else 0.0)
        return np.array(signals)

    val_kw_sets = {
        s: set(headlines_train[
            headlines_train['session'] == s
        ]['keyword'].tolist())
        for s in val_sids
    }

    pair_tr = session_pair_signal(
        train_sids, train_kw_sets, fold_pair_signals
    ).reshape(-1, 1)
    pair_va = session_pair_signal(
        val_sids, val_kw_sets, fold_pair_signals
    ).reshape(-1, 1)

    # ── OHLC only ─────────────────────────────────────────────────────────────
    scaler_o = StandardScaler()
    X_tr_o   = scaler_o.fit_transform(X_ohlc_raw[train_idx])
    X_va_o   = scaler_o.transform(X_ohlc_raw[val_idx])
    m_o      = Ridge(alpha=10000).fit(X_tr_o, y_tr)
    p_o      = make_positions(m_o.predict(X_va_o), sensitivity=0.25)
    fold_sharpes_ohlc.append(compute_sharpe(p_o, y_va))

    # ── Pair signal only ──────────────────────────────────────────────────────
    scaler_p  = StandardScaler()
    pair_tr_s = scaler_p.fit_transform(pair_tr)
    pair_va_s = scaler_p.transform(pair_va)

    m_p  = Ridge(alpha=10000).fit(pair_tr_s, y_tr)
    p_p  = make_positions(m_p.predict(pair_va_s), sensitivity=0.25)
    fold_sharpes_pairs.append(compute_sharpe(p_p, y_va))

    # ── Combined OHLC + pairs ─────────────────────────────────────────────────
    X_tr_comb = np.hstack([X_tr_o, pair_tr_s])
    X_va_comb = np.hstack([X_va_o, pair_va_s])

    m_c = Ridge(alpha=10000).fit(X_tr_comb, y_tr)
    p_c = make_positions(m_c.predict(X_va_comb), sensitivity=0.25)
    fold_sharpes_combined.append(compute_sharpe(p_c, y_va))

print(f"{'model':35s}  {'sharpe':>8}  {'std':>8}")
print("-" * 55)
print(f"  {'OHLC only':35s}  "
      f"{np.mean(fold_sharpes_ohlc):.4f}  "
      f"{np.std(fold_sharpes_ohlc):.4f}")
print(f"  {'Keyword pairs only':35s}  "
      f"{np.mean(fold_sharpes_pairs):.4f}  "
      f"{np.std(fold_sharpes_pairs):.4f}")
print(f"  {'OHLC + keyword pairs':35s}  "
      f"{np.mean(fold_sharpes_combined):.4f}  "
      f"{np.std(fold_sharpes_combined):.4f}")
print(f"\nBaseline: 2.8660")

=== LOO-safe CV: keyword pair co-occurrence signal ===

model                                  sharpe       std
-------------------------------------------------------
  OHLC only                            2.8660  0.5438
  Keyword pairs only                   2.7968  0.5178
  OHLC + keyword pairs                 2.8702  0.5097

Baseline: 2.8660


In [48]:
print("=== Tuning pair signal contribution ===\n")
print(f"{'alpha':>8}  {'pair_sens':>10}  {'ohlc_sens':>10}  "
      f"{'sharpe':>8}  {'std':>8}  {'SoS':>8}")
print("-" * 62)

best_sos, best_cfg = -np.inf, {}

for alpha in [100]:
    for pair_sens in [0.05, 0.10, 0.15, 0.25]:
        for ohlc_sens in [0.20, 0.25, 0.30]:
            fold_sharpes = []

            for train_idx, val_idx in kf.split(X_ohlc_raw):
                train_sids = session_arr[train_idx]
                val_sids   = session_arr[val_idx]

                y_tr = y_train.loc[train_sids].values
                y_va = y_train.loc[val_sids].values

                # Recompute pairs on train fold
                train_kw_sets = {
                    s: set(headlines_train[
                        headlines_train['session'] == s
                    ]['keyword'].tolist())
                    for s in train_sids
                }

                fold_pair_signals = {}
                for kw1, kw2 in itertools.combinations(keyword_list, 2):
                    both    = [s for s, kws in train_kw_sets.items()
                               if kw1 in kws and kw2 in kws]
                    neither = [s for s, kws in train_kw_sets.items()
                               if kw1 not in kws and kw2 not in kws]
                    if len(both) < 8 or len(neither) < 8:
                        continue
                    diff = (sharpe(y_train.loc[both]) -
                            sharpe(y_train.loc[neither]))
                    fold_pair_signals[(kw1, kw2)] = diff

                val_kw_sets = {
                    s: set(headlines_train[
                        headlines_train['session'] == s
                    ]['keyword'].tolist())
                    for s in val_sids
                }

                pair_tr = session_pair_signal(
                    train_sids, train_kw_sets, fold_pair_signals
                ).reshape(-1, 1)
                pair_va = session_pair_signal(
                    val_sids, val_kw_sets, fold_pair_signals
                ).reshape(-1, 1)

                # OHLC
                scaler_o  = StandardScaler()
                X_tr_o    = scaler_o.fit_transform(X_ohlc_raw[train_idx])
                X_va_o    = scaler_o.transform(X_ohlc_raw[val_idx])

                # Pairs
                scaler_p  = StandardScaler()
                pair_tr_s = scaler_p.fit_transform(pair_tr)
                pair_va_s = scaler_p.transform(pair_va)

                # Combined model
                X_tr_c = np.hstack([X_tr_o, pair_tr_s])
                X_va_c = np.hstack([X_va_o, pair_va_s])

                m = Ridge(alpha=alpha).fit(X_tr_c, y_tr)

                # Separate sensitivity for OHLC and pair components
                ohlc_pred = Ridge(alpha=alpha).fit(
                    X_tr_o, y_tr
                ).predict(X_va_o)
                pair_pred = Ridge(alpha=alpha).fit(
                    pair_tr_s, y_tr
                ).predict(pair_va_s)

                # Blend signals with separate sensitivities
                z_ohlc = (ohlc_pred - ohlc_pred.mean()) / (ohlc_pred.std() + 1e-8)
                z_pair = (pair_pred - pair_pred.mean()) / (pair_pred.std() + 1e-8)
                z_ohlc = np.clip(z_ohlc, -2.5, 2.5)
                z_pair = np.clip(z_pair, -2.5, 2.5)

                positions = np.maximum(
                    1.0 + ohlc_sens * z_ohlc + pair_sens * z_pair,
                    0.05
                )
                fold_sharpes.append(compute_sharpe(positions, y_va))

            mean_sh = np.mean(fold_sharpes)
            std_sh  = np.std(fold_sharpes)
            sos     = mean_sh / (std_sh + 1e-8)
            marker  = " ←" if sos > best_sos else ""

            if sos > best_sos:
                best_sos = sos
                best_cfg = dict(
                    alpha=alpha,
                    pair_sens=pair_sens,
                    ohlc_sens=ohlc_sens,
                    mean=mean_sh,
                    std=std_sh,
                )

            # Only print improvements
            if mean_sh > 2.8660 or std_sh < 0.5097:
                print(f"  {alpha:>8}  {pair_sens:>10}  {ohlc_sens:>10}  "
                      f"{mean_sh:>8.4f}  {std_sh:>8.4f}  {sos:>8.4f}{marker}")

print(f"\nBest by SoS: {best_cfg}")
print(f"Baseline:    mean=2.8660  std=0.5438  SoS={2.8660/0.5438:.4f}")

# ── Generate submission if SoS improved ───────────────────────────────────────

baseline_sos = 2.8660 / 0.5438

if best_sos > baseline_sos + 0.05:
    print(f"\n=== Generating pair signal submission ===\n")

    # Recompute pair signals on full training data
    all_kw_sets = {
        s: set(headlines_train[
            headlines_train['session'] == s
        ]['keyword'].tolist())
        for s in y_train.index
    }

    full_pair_signals = {}
    for kw1, kw2 in itertools.combinations(keyword_list, 2):
        both    = [s for s, kws in all_kw_sets.items()
                   if kw1 in kws and kw2 in kws]
        neither = [s for s, kws in all_kw_sets.items()
                   if kw1 not in kws and kw2 not in kws]
        if len(both) < 8 or len(neither) < 8:
            continue
        diff = (sharpe(y_train.loc[both]) -
                sharpe(y_train.loc[neither]))
        full_pair_signals[(kw1, kw2)] = diff

    # Fit final models
    pair_train = session_pair_signal(
        list(y_train.index), all_kw_sets, full_pair_signals
    ).reshape(-1, 1)

    scaler_o_final = StandardScaler()
    X_ohlc_final   = scaler_o_final.fit_transform(X_ohlc_raw)
    m_ohlc_final   = Ridge(alpha=best_cfg['alpha']).fit(X_ohlc_final, y)

    scaler_p_final = StandardScaler()
    pair_final_s   = scaler_p_final.fit_transform(pair_train)
    m_pair_final   = Ridge(alpha=best_cfg['alpha']).fit(pair_final_s, y)

    def make_pair_submission(bars_path, hl_path, output_path):
        bars  = pd.read_parquet(bars_path, engine="fastparquet")
        hl    = pd.read_parquet(hl_path,   engine="fastparquet")
        hl['keyword'] = hl['headline'].apply(label_headline)

        feats    = build_features(bars)[ORIGINAL_FEATURES]
        sessions = feats.index.tolist()

        test_kw_sets = {
            s: set(hl[hl['session'] == s]['keyword'].tolist())
            for s in sessions
        }

        # OHLC signal
        X_test_o   = scaler_o_final.transform(feats.values)
        ohlc_pred  = m_ohlc_final.predict(X_test_o)

        # Pair signal
        pair_test  = session_pair_signal(
            sessions, test_kw_sets, full_pair_signals
        ).reshape(-1, 1)
        pair_test_s = scaler_p_final.transform(pair_test)
        pair_pred   = m_pair_final.predict(pair_test_s)

        # Blend
        z_ohlc = (ohlc_pred - ohlc_pred.mean()) / (ohlc_pred.std() + 1e-8)
        z_pair = (pair_pred - pair_pred.mean()) / (pair_pred.std() + 1e-8)
        z_ohlc = np.clip(z_ohlc, -2.5, 2.5)
        z_pair = np.clip(z_pair, -2.5, 2.5)

        positions = np.maximum(
            1.0 + best_cfg['ohlc_sens'] * z_ohlc
                + best_cfg['pair_sens'] * z_pair,
            0.05
        )

        sub = pd.DataFrame({
            'session':         sessions,
            'target_position': positions,
        }).sort_values('session')
        sub.to_csv(output_path, index=False)
        print(f"Saved {len(sub)} rows → {output_path}  "
              f"mean={positions.mean():.4f}  std={positions.std():.4f}")

    make_pair_submission(
        os.path.join(DATA_DIR, "bars_seen_public_test.parquet"),
        os.path.join(DATA_DIR, "headlines_seen_public_test.parquet"),
        "submission_public_pairs.csv",
    )
    make_pair_submission(
        os.path.join(DATA_DIR, "bars_seen_private_test.parquet"),
        os.path.join(DATA_DIR, "headlines_seen_private_test.parquet"),
        "submission_private_pairs.csv",
    )
else:
    print(f"\nSoS improvement insufficient.")
    print(f"Submit submission_private_hl.csv")

=== Tuning pair signal contribution ===

   alpha   pair_sens   ohlc_sens    sharpe       std       SoS
--------------------------------------------------------------
       100        0.05         0.3    2.8388    0.5019    5.6559 ←
       100         0.1        0.25    2.8560    0.5037    5.6704 ←
       100         0.1         0.3    2.8481    0.4968    5.7325 ←
       100        0.15         0.2    2.8626    0.5042    5.6777
       100        0.15        0.25    2.8601    0.4977    5.7466 ←
       100        0.15         0.3    2.8526    0.4926    5.7909 ←
       100        0.25         0.2    2.8547    0.4919    5.8037 ←
       100        0.25        0.25    2.8539    0.4880    5.8487 ←
       100        0.25         0.3    2.8465    0.4862    5.8548 ←

Best by SoS: {'alpha': 100, 'pair_sens': 0.25, 'ohlc_sens': 0.3, 'mean': np.float64(2.84652571373677), 'std': np.float64(0.486186693108231)}
Baseline:    mean=2.8660  std=0.5438  SoS=5.2703

=== Generating pair signal submission ==

In [51]:
# Step 1: Identify the 16-session clusters via return correlation
n_clusters = 16
bars_df = pd.read_parquet("../data/bars_seen_train.parquet")
unseen_bars_df = pd.read_parquet("../data/bars_unseen_train.parquet")
sentiment_df = pd.read_csv("../sentiment_overview.csv")
headlines_df_seen = pd.read_parquet("../data/headlines_seen_train.parquet")
headlines_df_unseen = pd.read_parquet("../data/headlines_unseen_train.parquet")


returns = bars_df.groupby('session')['close'].last() / \
          bars_df.groupby('session')['close'].first() - 1

# Compute pairwise correlation of bar-level returns (align on bar_ix)
pivot = bars_df.pivot(index='bar_ix', columns='session', values='close').pct_change()
corr = pivot.corr()

# Cluster into groups of 16
from sklearn.cluster import AgglomerativeClustering
labels = AgglomerativeClustering(n_clusters=n_clusters, metric='precomputed', 
                                  linkage='average').fit_predict(1 - corr)

In [ ]:
# Step 2: Extract company mentions from headlines (simple NER or regex on capitalized phrases)
import re

def extract_companies(headline):
    # Fictional companies tend to be Title Case multi-word phrases
    return " ".join(headline.split()[:2])

headlines_df_seen['companies'] = headlines_df_seen['headline'].apply(extract_companies)

In [64]:
# Step 3: For each test session, aggregate sentiment from ALL sessions in the same cluster
# at the seen bars — this gives you "what is the cluster-wide news saying right now?"

sentiment_df = pd.read_csv("../anna/headline_stats_seen_train.csv")
sentiment_df

,session,headline,bar_ix,decided_label,prob_positive,prob_negative,prob_neutral,linear_score,polarity_score,cluster
0,0,Relvos Biosciences,6,neutral,0.387032,0.013454,0.599514,0.373578,0.932811,0
1,0,Orevex Renewables,12,positive,0.948091,0.012295,0.039614,0.935795,0.974395,1
2,0,Relvos Biosciences,14,neutral,0.061307,0.040272,0.898421,0.021034,0.207074,0
3,0,Calvis Sciences,20,positive,0.945060,0.010088,0.044852,0.934971,0.978876,0
4,0,Yorvov Pharmaceuticals,22,positive,0.947761,0.012083,0.040155,0.935678,0.974823,0
...,...,...,...,...,...,...,...,...,...,...
9735,999,Yorval Partners,33,neutral,0.064574,0.016120,0.919306,0.048454,0.600466,3
9736,999,Yorval Partners,33,positive,0.611134,0.013840,0.375025,0.597294,0.955710,3
9737,999,Krevan Investments,41,negative,0.008476,0.949368,0.042156,-0.940891,-0.982301,3
9738,999,Wrelal Financial,45,positive,0.944212,0.012828,0.042960,0.931384,0.973193,3


In [ ]:
"""
Zurich Datathon 2026 - Full Submission Pipeline
================================================
Combines OHLC technical features with cluster-aware headline sentiment.

Usage:
    python solution.py --data_dir ./data --output submission.csv

Dependencies:
    pip install pandas numpy scikit-learn transformers torch lightgbm tqdm
"""

import argparse
import re
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────────────────────
# 1.  CLUSTER DEFINITIONS
# ──────────────────────────────────────────────────────────────

pharma_cluster = [
    "Calvis Sciences", "Calvos Genomics", "Frelis Genomics", "Grevel Diagnostics",
    "Grevon Biotech", "Jorvix Diagnostics", "Krevum Pharmaceuticals", "Myrnon Therapeutics",
    "Nerval Biopharma", "Relvos Biosciences", "Wyrnik Sciences", "Xelvol Biotech",
    "Yorvov Pharmaceuticals", "Zelvix Therapeutics", "Zelvon Biosciences", "Zrovum Biopharma",
]
energy_cluster = [
    "Jorvis Fuels", "Kelvik Power", "Kelvos Resources", "Nerven Grid",
    "Nolvol Resources", "Orevex Renewables", "Orevov Solutions", "Plevep Power",
    "Plevik Energy", "Relvon Fuels", "Strynal Industries", "Ulvon Renewables",
    "Urvel Grid", "Wyrnor Solutions", "Zelval Energy", "Zrovex Industries",
]
commerce_cluster = [
    "Arnik Commerce", "Arnos Marketplace", "Crevol Retail", "Frelex Outlets",
    "Halvav Brands", "Holtar Stores", "Ixenis Outlets", "Jorval Trading",
    "Joval Brands", "Nolvav Commerce", "Orevar Marketplace", "Pleven Trading",
    "Talvyn Goods", "Varvov Retail", "Xovep Stores", "Xovol Goods",
]
money_cluster = [
    "Creven Securities", "Ervan Capital", "Halven Investments", "Halvix Holdings",
    "Holtum Asset", "Ilval Financial", "Jorvyl Securities", "Jovik Asset",
    "Krevan Investments", "Plevum Partners", "Plevyl Advisors", "Talvix Holdings",
    "Wrelal Financial", "Yorval Partners", "Zelvel Capital", "Zrovov Advisors",
]
hardware_cluster = [
    "Brevep Systems", "Brevon Microchips", "Crevex Labs", "Dralol Computing",
    "Frelol Software", "Halvax Networks", "Ixenix Technologies", "Jovor Networks",
    "Myrnep Technologies", "Nolvis Devices", "Prynis Systems", "Relvan Software",
    "Talvep Computing", "Ulvyn Microchips", "Volval Devices", "Yorven Labs",
]

CLUSTERS = {
    "pharma": pharma_cluster,
    "energy": energy_cluster,
    "commerce": commerce_cluster,
    "money": money_cluster,
    "hardware": hardware_cluster,
}

# company -> (cluster_name, cluster_index)
COMPANY_TO_CLUSTER: dict[str, tuple[str, int]] = {}
CLUSTER_NAMES = list(CLUSTERS.keys())
for _idx, (_name, _members) in enumerate(CLUSTERS.items()):
    for _company in _members:
        COMPANY_TO_CLUSTER[_company] = (_name, _idx)

ALL_COMPANIES = list(COMPANY_TO_CLUSTER.keys())


# ──────────────────────────────────────────────────────────────
# 2.  COMPANY MENTION EXTRACTION
# ──────────────────────────────────────────────────────────────

def extract_mentions(headline: str) -> list[str]:
    """Return all known company names found in the headline."""
    return [c for c in ALL_COMPANIES if c in headline]


def tag_headlines(df: pd.DataFrame) -> pd.DataFrame:
    """Add 'mentioned_companies' and 'mentioned_cluster_indices' columns."""
    df = df.copy()
    df["mentioned_companies"] = df["headline"].apply(lambda x: [x])
    df["mentioned_cluster_indices"] = df["mentioned_companies"].apply(
        lambda cs: list({COMPANY_TO_CLUSTER[c][1] for c in cs})
    )
    return df


# ──────────────────────────────────────────────────────────────
# 3.  SESSION → COMPANY IDENTITY
# ──────────────────────────────────────────────────────────────

def identify_sessions(headlines_df: pd.DataFrame) -> pd.DataFrame:
    """
    For every session, vote on which company it represents by counting
    per-company mentions. The dominant company wins.

    Returns a DataFrame indexed by session with columns:
        company, cluster_name, cluster_idx
    """
    records = []
    for session_id, grp in tqdm(
        headlines_df.groupby("session"), desc="Identifying sessions"
    ):
        counts: dict[str, int] = defaultdict(int)
        for mentions in grp["mentioned_companies"]:
            for c in mentions:
                counts[c] += 1

        if counts:
            top_company = max(counts, key=counts.get)
            cluster_name, cluster_idx = COMPANY_TO_CLUSTER[top_company]
        else:
            top_company, cluster_name, cluster_idx = None, None, -1

        records.append(
            dict(
                session=session_id,
                company=top_company,
                cluster_name=cluster_name,
                cluster_idx=cluster_idx,
            )
        )

    return pd.DataFrame(records).set_index("session")


# ──────────────────────────────────────────────────────────────
# 4.  SENTIMENT SCORING
# ──────────────────────────────────────────────────────────────

_LABEL_SIGN = {"positive": 1.0, "negative": -1.0, "neutral": 0.0}


# ──────────────────────────────────────────────────────────────
# 5.  FEATURE ENGINEERING
# ──────────────────────────────────────────────────────────────

def ohlc_features(bars: pd.DataFrame) -> dict:
    """Technical features from the SEEN portion of a session's bars."""
    c = bars["close"].values
    h = bars["high"].values
    lo = bars["low"].values
    o = bars["open"].values
    n     = len(c)
    br    = np.diff(c) / c[:-1] if n > 1 else np.array([0.0])

    ret = np.diff(c) / c[:-1]

    half_close = c[-1]   # close at halfway point (what we trade at)

    feats = {
        # momentum
        "momentum_full": c[-1] / c[0] - 1,
        # volatility
        "volatility": ret.std() if len(ret) > 1 else 0.0,
        "range_ratio": (h.max() - lo.min()) / c[0],
        # microstructure
        "close_vs_high": (c[-1] - h[-1]) / (h[-1] - lo[-1] + 1e-9),
        "close_vs_open_last": (c[-1] - o[-1]) / (o[-1] + 1e-9),
        # trend
        "linear_trend": np.polyfit(np.arange(len(c)), c / c[0], 1)[0],
        "half_close": half_close,
        "n_bars_seen": len(c),
        "cum_return":   c[-1] / c[0] - 1,
        "last3_return": c[-1] / c[max(0, n - 4)] - 1,
        "slope":        np.polyfit(np.arange(n), c, 1)[0] / c.mean(),
        "realized_vol": np.std(br),
        "range_mean":   np.mean((h - lo) / c),
        "up_bar_frac":  np.mean(c > o),
        "wick_ratio":   np.mean((h - c) / (h - lo + 1e-8)),
    }
    """
            

    """
    return feats


def headline_features(
    session_id: int,
    headlines_df: pd.DataFrame,
    session_identity: pd.DataFrame,
    train_identity: pd.DataFrame,
    train_headlines_df: pd.DataFrame,
    seen_bar_count: int,
) -> dict:
    """
    Sentiment features for a session, using cluster-aware weighting.
    """
    if session_id not in session_identity.index:
        return {k: 0.0 for k in [
            "direct_sentiment", "peer_sentiment_train",
            "cluster_sentiment_weighted", "distractor_ratio",
            "cluster_idx",
        ]}

    own_company = session_identity.loc[session_id, "company"]
    own_cluster_idx = session_identity.loc[session_id, "cluster_idx"]

    # ── Seen headlines for this session ──────────────────────
    sess_hl = headlines_df[headlines_df["session"] == session_id].copy()
    sess_hl = sess_hl[sess_hl["bar_ix"] < seen_bar_count]

    def _weight_row(row):
        cs = row["mentioned_companies"]
        if own_company and own_company in cs:
            return row["sentiment"] * 3.0          # own company → strong
        elif any(COMPANY_TO_CLUSTER.get(c, (None, -1))[1] == own_cluster_idx for c in cs):
            return row["sentiment"] * 1.0          # cluster peer → normal
        else:
            return row["sentiment"] * 0.1          # distractor → near-zero

    direct_sentiment = 0.0
    distractor_ratio = 0.0
    if len(sess_hl):
        sess_hl["w_sentiment"] = sess_hl.apply(_weight_row, axis=1)
        direct_sentiment = sess_hl["w_sentiment"].mean()

        n_distractor = sess_hl["mentioned_cluster_indices"].apply(
            lambda idxs: own_cluster_idx not in idxs and len(idxs) > 0
        ).sum()
        distractor_ratio = n_distractor / len(sess_hl)

    # ── Peer sessions in the same cluster (training data only) ──
    if own_cluster_idx >= 0:
        peer_sessions = train_identity[
            (train_identity["cluster_idx"] == own_cluster_idx)
        ].index.tolist()
    else:
        peer_sessions = []

    peer_hl = train_headlines_df[train_headlines_df["session"].isin(peer_sessions)]
    peer_sentiment_train = peer_hl["sentiment"].mean() if len(peer_hl) else 0.0

    # Weighted cluster sentiment (recency-weighted within seen bars)
    all_cluster_hl = pd.concat([sess_hl, peer_hl]) if len(peer_hl) else sess_hl
    if len(all_cluster_hl):
        max_bar = all_cluster_hl["bar_ix"].max()
        weights = (all_cluster_hl["bar_ix"] + 1) / (max_bar + 1)
        cluster_sentiment_weighted = (
            (all_cluster_hl["sentiment"] * weights).sum() / weights.sum()
        )
    else:
        cluster_sentiment_weighted = 0.0

    return {
        "direct_sentiment":           direct_sentiment,
        "peer_sentiment_train":       peer_sentiment_train,
        "cluster_sentiment_weighted": cluster_sentiment_weighted,
        "distractor_ratio":           distractor_ratio,
        "cluster_idx":                float(own_cluster_idx),
        "avg_entropy":                sess_hl["sentiment_entropy"].mean() if len(sess_hl) else 0.0,
        "avg_confidence":             sess_hl["sentiment_confidence"].mean() if len(sess_hl) else 0.0,
    }


def build_feature_matrix(
    sessions: list[int],
    bars_df: pd.DataFrame,
    headlines_df: pd.DataFrame,
    session_identity: pd.DataFrame,
    train_identity,          # NEW
    train_headlines_df,      # NEW
    seen_bar_count: int | None = None,  # None = use all bars (training)
) -> pd.DataFrame:
    rows = []
    for sid in tqdm(sessions, desc="Building features"):
        bars = bars_df[bars_df["session"] == sid].sort_values("bar_ix")
        if seen_bar_count is not None:
            bars = bars[bars["bar_ix"] < seen_bar_count]

        if len(bars) == 0:
            continue

        feats = {"session": sid}
        feats.update(ohlc_features(bars))
        feats.update(
            headline_features(sid, headlines_df, session_identity, train_identity, train_headlines_df,
                              seen_bar_count or bars["bar_ix"].max() + 1)
        )
        rows.append(feats)

    return pd.DataFrame(rows).set_index("session")


# ──────────────────────────────────────────────────────────────
# 6.  TARGET: DIRECTION & MAGNITUDE
# ──────────────────────────────────────────────────────────────

def compute_targets(
    sessions: list[int],
    seen_bars: pd.DataFrame,
    unseen_bars: pd.DataFrame,
) -> pd.Series:
    """
    For training: target = return from halfway-close to end-close.
    We predict the sign and scale our position accordingly.
    """
    records = {}
    for sid in sessions:
        seen = seen_bars[seen_bars["session"] == sid].sort_values("bar_ix")
        unseen = unseen_bars[unseen_bars["session"] == sid].sort_values("bar_ix")
        if len(seen) == 0 or len(unseen) == 0:
            continue
        half_close = seen["close"].iloc[-1]
        end_close = unseen["close"].iloc[-1]
        records[sid] = end_close / half_close - 1  # raw return

    return pd.Series(records, name="target_return")


# ──────────────────────────────────────────────────────────────
# 7.  MODEL: LIGHTGBM WITH CROSS-VALIDATION
# ──────────────────────────────────────────────────────────────

def sharpe_objective(y_pred, dataset):
    y_true = dataset.get_label()
    pnl = y_pred * y_true
    mu = np.mean(pnl)
    sigma = np.std(pnl) + 1e-4  # larger floor

    n = len(y_true)
    grad = -(y_true / (n * sigma) - mu * (pnl - mu) * y_true / (n * sigma ** 3))

    # Normalize gradient to unit std — prevents exploding updates
    grad_std = np.std(grad)
    if grad_std > 1e-8:
        grad = grad / grad_std

    hess = np.ones_like(grad)
    return grad, hess


def sharpe_eval(y_pred, dataset):
    y_true = dataset.get_label()
    pnl = y_pred * y_true
    sharpe = np.mean(pnl) / (np.std(pnl) + 1e-9) * 16
    return "neg_sharpe", -sharpe, False  # minimize neg_sharpe = maximize sharpe

def train_model(X: pd.DataFrame, y: pd.Series, session_identity):
    """Train a LightGBM regressor, return fitted model."""
    import lightgbm as lgb
    from sklearn.model_selection import StratifiedKFold

    feature_cols = [c for c in X.columns if c != "half_close"]
    cluster_labels = session_identity.reindex(X.index)["cluster_idx"].fillna(-1).astype(int)
    print("Cluster label distribution:")        # ← here
    print(cluster_labels.value_counts())

    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(X))
    models = []

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X, cluster_labels)):  
        X_tr, X_val = X.iloc[tr_idx][feature_cols], X.iloc[val_idx][feature_cols]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        dtrain = lgb.Dataset(X_tr, label=y_tr)
        dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

        params = {
            "objective":          sharpe_objective,
            "metric":             "None",
            "learning_rate":      0.005,     # slower learning
            "num_leaves":         8,         # much smaller trees
            "feature_fraction":   0.5,
            "bagging_fraction":   0.5,
            "bagging_freq":       1,
            "min_child_samples":  100,       # needs 100 samples to make a leaf
            "lambda_l1":          1.0,
            "lambda_l2":          10.0,
            "min_gain_to_split":  0.1,
            "verbose":            -1,
        }

        model = lgb.train(
            params,
            dtrain,
            num_boost_round=2000,
            valid_sets=[dval],
            feval=sharpe_eval,              # <-- custom eval metric here
            callbacks=[lgb.early_stopping(200, verbose=True),
                       lgb.log_evaluation(period=100)],
        )
        oof_preds[val_idx] = model.predict(X_val[feature_cols])
        models.append((model, feature_cols))
        print(f"  Fold {fold+1} | best iter: {model.best_iteration}")

    # Evaluate OOF Sharpe
    oof_sharpe = _sharpe(y.values, oof_preds)
    print(f"\nOOF Sharpe (raw return * position): {oof_sharpe:.4f}")

    return models


def _sharpe(returns: np.ndarray, positions: np.ndarray) -> float:
    pnl = positions * returns
    return np.mean(pnl) / (np.std(pnl) + 1e-9) * 16


def predict(models, X: pd.DataFrame) -> np.ndarray:
    """Average predictions across all CV folds."""
    preds = []
    for model, feature_cols in models:
        preds.append(model.predict(X[feature_cols]))
    return np.mean(preds, axis=0)


# ──────────────────────────────────────────────────────────────
# 8.  POSITION SIZING
# ──────────────────────────────────────────────────────────────
def kelly_size_positions(
    predicted_returns: np.ndarray,
    half_closes: np.ndarray,
    vol: np.ndarray | None = None,
    max_leverage: float = 3.0,
    kelly_fraction: float = 0.25,   # fractional Kelly (IMPORTANT)
) -> np.ndarray:
    """
    Kelly-style position sizing.

    f* ≈ μ / σ²
    where:
        μ = predicted return
        σ = volatility estimate

    We:
      - use realized_vol as σ proxy
      - apply fractional Kelly (reduce risk)
      - clip leverage to avoid explosions
    """

    preds = predicted_returns.copy()

    # If no vol provided, fallback to global estimate
    if vol is None:
        vol = np.std(preds) * np.ones_like(preds)

    # Avoid division issues
    vol = np.maximum(vol, 1e-6)

    # Kelly fraction: μ / σ²
    kelly = preds / (vol ** 2)

    # Fractional Kelly (CRUCIAL for noisy signals)
    kelly *= kelly_fraction

    # Clip extreme leverage
    kelly = np.clip(kelly, -max_leverage, max_leverage)

    return kelly

def size_positions(preds, scale=100.0):
    preds = np.nan_to_num(preds, nan=0.0)  # replace NaN with 0 = no position
    clipped = np.clip(preds, np.percentile(preds, 5), np.percentile(preds, 95))
    abs_max = np.abs(clipped).max()
    return clipped / abs_max * scale if abs_max > 0 else np.zeros_like(clipped)

def use_precomputed_sentiment(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Primary signal
    df["sentiment"] = df["polarity_score"]

    # Optional extras (VERY useful)
    df["sentiment_confidence"] = (
        df[["prob_positive", "prob_negative", "prob_neutral"]].max(axis=1)
    )

    df["sentiment_entropy"] = -(
        df[["prob_positive", "prob_negative", "prob_neutral"]] *
        np.log(df[["prob_positive", "prob_negative", "prob_neutral"]] + 1e-9)
    ).sum(axis=1)

    return df

# ──────────────────────────────────────────────────────────────
# 9.  MAIN PIPELINE
# ──────────────────────────────────────────────────────────────

def run(data_dir: str, output_path: str, anna_dir:str):
    def cache_path(cache_dir: Path, name: str) -> Path:
        return cache_dir / f"{name}.parquet"


    def load_or_compute(cache_file: Path, compute_fn, force: bool = False):
        if cache_file.exists() and not force:
            print(f"Loading cached: {cache_file.name}")
            return pd.read_parquet(cache_file)
        print(f"Computing: {cache_file.name}")
        df = compute_fn()
        df.to_parquet(cache_file)
        return df
    
    data_dir = Path(data_dir)
    anna_dir = Path(anna_dir)
    cache_dir = data_dir / "cache"
    cache_dir.mkdir(exist_ok=True)

    # ── Load data ────────────────────────────────────────────
    print("Loading data…")
    seen_train_bars      = pd.read_parquet(data_dir / "bars_seen_train.parquet")
    unseen_train_bars    = pd.read_parquet(data_dir / "bars_unseen_train.parquet")
    seen_pub_bars        = pd.read_parquet(data_dir / "bars_seen_public_test.parquet")
    seen_priv_bars       = pd.read_parquet(data_dir / "bars_seen_private_test.parquet")

    # seen_train_hl        = pd.read_parquet(data_dir / "headlines_seen_train.parquet")
    # unseen_train_hl      = pd.read_parquet(data_dir / "headlines_unseen_train.parquet")
    # seen_pub_hl          = pd.read_parquet(data_dir / "headlines_seen_public_test.parquet")
    # seen_priv_hl         = pd.read_parquet(data_dir / "headlines_seen_private_test.parquet")

    seen_train_hl        = pd.read_csv(anna_dir / "headline_stats_seen_train.csv")
    unseen_train_hl      = pd.read_csv(anna_dir / "headline_stats_unseen_train.csv")
    seen_pub_hl          = pd.read_csv(anna_dir / "headline_stats_seen_pubtest.csv")
    seen_priv_hl         = pd.read_csv(anna_dir / "headline_stats_seen_privtest.csv")


    # Combine all train headlines (seen + unseen) for session identity detection
    all_train_hl         = pd.concat([seen_train_hl, unseen_train_hl], ignore_index=True)

    # ── Determine seen_bar_count from training data ───────────
    # The seen portion is the first half → infer from the split
    max_seen_bar  = seen_train_bars["bar_ix"].max()
    max_total_bar = (
        pd.concat([seen_train_bars, unseen_train_bars])["bar_ix"].max()
    )
    # seen bars go 0..N-1, unseen go N..2N-1
    seen_bar_count = max_seen_bar + 1
    print(f"Detected seen_bar_count = {seen_bar_count}, total = {max_total_bar + 1}")

    # ── Tag mentions ─────────────────────────────────────────
    print("Tagging company mentions…")
    all_train_hl  = tag_headlines(all_train_hl)
    seen_train_hl  = tag_headlines(seen_train_hl)
    unseen_train_hl  = tag_headlines(unseen_train_hl)
    seen_pub_hl   = tag_headlines(seen_pub_hl)
    seen_priv_hl  = tag_headlines(seen_priv_hl)

    # ── Score sentiment ──────────────────────────────────────
    print("Using precomputed sentiment…")
    all_train_hl = use_precomputed_sentiment(all_train_hl)
    seen_train_hl  = use_precomputed_sentiment(seen_train_hl)
    unseen_train_hl  = use_precomputed_sentiment(unseen_train_hl)
    seen_pub_hl  = use_precomputed_sentiment(seen_pub_hl)
    seen_priv_hl = use_precomputed_sentiment(seen_priv_hl)

    # ── Identify session → company ───────────────────────────
    print("Identifying sessions…")
    train_identity = identify_sessions(all_train_hl)
    pub_identity   = identify_sessions(seen_pub_hl)
    priv_identity  = identify_sessions(seen_priv_hl)

    print("\nCluster distribution (train):")
    print(train_identity["cluster_name"].value_counts())

    # ── Build features ────────────────────────────────────────
    train_sessions = seen_train_bars["session"].unique().tolist()
    pub_sessions   = seen_pub_bars["session"].unique().tolist()
    priv_sessions  = seen_priv_bars["session"].unique().tolist()

    print("Building training features…")
    X_train = load_or_compute(
        cache_path(cache_dir, "X_train"),
        lambda: build_feature_matrix(
            train_sessions, seen_train_bars, seen_train_hl,
            train_identity,
            train_identity,      # peers come from same pool, self excluded inside fn
            all_train_hl,
            seen_bar_count
        ),
    )
    print("Building public features…")
    X_pub = load_or_compute(
        cache_path(cache_dir, "X_pub"),
        lambda: build_feature_matrix(
            pub_sessions, seen_pub_bars, seen_pub_hl,
            pub_identity,
            train_identity,      # <-- training peers
            all_train_hl,        # <-- training headlines
            seen_bar_count,
        ),
    )
    print("Building private features…")
    X_priv = load_or_compute(
        cache_path(cache_dir, "X_priv"),
        lambda: build_feature_matrix(
            priv_sessions, seen_priv_bars, seen_priv_hl,
            priv_identity,
            train_identity,      # <-- training peers
            all_train_hl,        # <-- training headlines
            seen_bar_count,
        ),
    )

    # ── Compute training targets ──────────────────────────────
    y_train = compute_targets(train_sessions, seen_train_bars, unseen_train_bars)
    y_train = y_train.reindex(X_train.index).dropna()
    X_train = X_train.loc[y_train.index]

    # Simulate one gradient step manually
    y_true_sample = y_train.values[:100]
    y_pred_sample = np.zeros(100)  # LightGBM starts from zero
    dataset_mock = type('obj', (object,), {'get_label': lambda self: y_true_sample})()
    grad, hess = sharpe_objective(y_pred_sample, dataset_mock)
    print(f"Grad std: {grad.std():.6f}, mean: {grad.mean():.6f}")
    # If both are ~0.0, the gradient is degenerate

    # ── Train ─────────────────────────────────────────────────
    print("\nTraining model…")
    models = train_model(X_train, y_train, train_identity)

    # ── Predict & size ────────────────────────────────────────
    def make_submission(X: pd.DataFrame, bars_df: pd.DataFrame, sessions: list[int]):
        preds = predict(models, X)

        avg_entropy = X["avg_entropy"].fillna(X["avg_entropy"].median())
        confidence_scale = 1.0 / (1.0 + avg_entropy.values)
        confidence_scale = confidence_scale / confidence_scale.mean()

        positions = size_positions(preds) * confidence_scale

        result = pd.DataFrame({
            "session": X.index,
            "target_position": positions,
        }).set_index("session")

        # Reindex to ALL sessions, fill missing with 0
        result = result.reindex(sessions, fill_value=0.0).reset_index()

        missing = len(sessions) - len(X)
        if missing > 0:
            print(f"WARNING: {missing} sessions missing from feature matrix, filled with 0")

        return result

    sub_pub  = make_submission(X_pub,  seen_pub_bars,  pub_sessions)
    sub_priv = make_submission(X_priv, seen_priv_bars, priv_sessions)

    submission = pd.concat([sub_pub, sub_priv], ignore_index=True)
    submission.to_csv(output_path, index=False)
    print(f"\n✓ Submission saved to {output_path} ({len(submission)} rows)")
    print(submission.head(10).to_string())

    return submission


# ──────────────────────────────────────────────────────────────
# 10. ENTRYPOINT
# ──────────────────────────────────────────────────────────────

run("../data", "submi.csv", "../anna")

Loading data…
Detected seen_bar_count = 50, total = 100
Tagging company mentions…
Using precomputed sentiment…
Identifying sessions…


Identifying sessions: 100%|██████████| 9999/9999 [00:00<00:00, 19287.92it/s]



Cluster distribution (train):
cluster_name
pharma      211
hardware    205
money       198
commerce    197
energy      189
Name: count, dtype: int64
Building training features…
Loading cached: X_train.parquet
Building public features…
Loading cached: X_pub.parquet
Building private features…
Loading cached: X_priv.parquet
Grad std: 1.000000, mean: -0.179449

Training model…
Cluster label distribution:
cluster_idx
0    211
4    205
3    198
2    197
1    189
Name: count, dtype: int64
Training until validation scores don't improve for 200 rounds
[100]	valid_0's neg_sharpe: -1.94079
[200]	valid_0's neg_sharpe: -1.83957
[300]	valid_0's neg_sharpe: -1.88396
Early stopping, best iteration is:
[186]	valid_0's neg_sharpe: -2.08109
  Fold 1 | best iter: 186
Training until validation scores don't improve for 200 rounds
[100]	valid_0's neg_sharpe: -2.02246
[200]	valid_0's neg_sharpe: -2.38862
Early stopping, best iteration is:
[4]	valid_0's neg_sharpe: -2.66802
  Fold 2 | best iter: 4
Training un